# Phase 2 — Object-Oriented Programming and the Data Model

**Outcome:** build classes that behave like built-ins, design with inheritance and
composition, and understand the descriptor and metaclass machinery behind Pydantic,
SQLAlchemy and dataclasses.

| Unit | Topic |
| --- | --- |
| 2.1 | Classes and instances |
| 2.2 | Inheritance, MRO and composition |
| 2.3 | Abstractions: ABCs, `collections.abc` and Protocols |
| 2.4 | The data model: special (dunder) methods |
| 2.5 | Attribute access, descriptors and metaprogramming |
| 2.6 | Dataclasses, enums and Pythonic design patterns |
| Project | Inventory domain model |

**How to use this notebook**

- Select the **Python 3.15 (course)** kernel, then run the *Shared setup* cell first.
- Each unit starts with its own setup cell, so you can jump straight to any unit.
- *Predict the output* questions hide the answer in a collapsible block: decide first,
  then run the proof cell.
- Labs: replace the `raise NotImplementedError` in each stub, then run the check cell.
  Until then the checks print FAIL; that is expected.
- Features newer than Python 3.12 are labelled (*New in 3.13*, *Changed in 3.15*, …).

## What you need from Phase 1

Phase 2 builds directly on these Phase 1 ideas. If one feels shaky, re-read the unit
before going on.

| Phase 1 idea | Unit | Why it matters for classes |
| --- | --- | --- |
| Everything is an object; names are references | 1.2 | Classes, methods and instances are all objects you can inspect and pass around |
| Mutable vs immutable, identity vs equality | 1.2.4–1.2.5 | The shared mutable class-attribute trap; `__eq__` vs `is` |
| `dict` internals and key-sharing dictionaries | 1.6.1 | Every instance stores its attributes in a `__dict__` that shares keys with its siblings |
| `__copy__` / `__deepcopy__` | 1.7.4 | Classes customize built-in behaviour through special methods |
| Class patterns and `__match_args__` | 1.8.4 | `match` destructures your own classes |
| Functions are first-class; `*args` / `**kwargs` | 1.10 | Methods are functions stored on the class; cooperative `**kwargs` in `super()` chains |
| Scope, closures and cells | 1.11 | Zero-argument `super()` reads a hidden `__class__` cell |
| Decorators, including class decorators and `__get__` | 1.12 | `@property`, `@classmethod` and `@staticmethod` are decorators built on descriptors |
| The iterator protocol | 1.13 | `collections.abc.Iterator` and `Iterable` formalize it |
| Custom exception hierarchies, `__enter__` / `__exit__` | 1.14 | You already designed classes around inheritance and special methods |

> **Note:** Phase 1 used small classes (`Price`, `LinkedListNode`, `Timer`, `AppError`)
> before explaining them fully. This phase explains everything those classes relied on.

## Shared setup

This cell runs first in every notebook of the course. It imports common modules and
defines three helpers used everywhere:

| Helper | What it does |
| --- | --- |
| `run_checks(test_cases)` | runs `(label, thunk, expected)` cases, prints PASS/FAIL |
| `show_size(label, obj)` | prints `sys.getsizeof(obj)`, type name and `id()` |
| `find_repo_root()` | finds the folder holding `pyproject.toml` |

`run_checks` catches every exception, so a failing check never stops Run All.

> **Note:** each unit re-imports what it needs, so you can run any unit on its own
> after this cell.

In [1]:
import inspect
import os
import platform
import sys
from collections.abc import Callable
from pathlib import Path

type TestCase = tuple[str, Callable[[], object], object]


def run_checks(test_cases: list[TestCase]) -> None:
    """Run (label, thunk, expected) cases and print PASS/FAIL for each. Never raises."""
    passed_count = 0
    for label, thunk, expected in test_cases:
        try:
            actual = thunk()
        except Exception as error:  # a check cell must never stop Run All
            print(f"FAIL  {label}: raised {type(error).__name__}: {error}")
            continue
        if actual == expected:
            passed_count += 1
            print(f"PASS  {label}")
        else:
            print(f"FAIL  {label}: expected {expected!r}, got {actual!r}")
    print(f"{passed_count}/{len(test_cases)} checks passed")


def show_size(label: str, obj: object) -> None:
    """Print the shallow size in bytes, the type and the id of an object."""
    print(f"{label}: {sys.getsizeof(obj)} bytes, type={type(obj).__name__}, id={id(obj)}")


def find_repo_root(start: Path | None = None) -> Path:
    """Walk up from start (default: the working directory) to the folder holding pyproject.toml."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found above the working directory")


print(f"Python {platform.python_version()} ({sys.implementation.name}) on {platform.system()}")
print(f"Repository root: {find_repo_root().name}/")

Python 3.15.1 (cpython) on Linux
Repository root: python-course/


## Unit 2.1 — Classes and Instances

**Objectives**

- Decide when a class is the right tool, and when a function, a module or a
  dataclass is better.
- Explain what the `class` statement does at run time, down to the `type(...)` call.
- Trace object creation through `__new__` and `__init__`, and inspect instances with
  `vars()` and `__class__`.
- Predict attribute lookup (instance → class → bases) and avoid the shared mutable
  class-attribute trap.
- Choose between instance methods, `@classmethod` and `@staticmethod`, and write
  subclass-aware alternative constructors.
- Use `@property` for computed, read-only and validated attributes.
- Use Python's privacy conventions and compare them with JavaScript `#private`.

**JS bridge**

| JavaScript / TypeScript | Python | The difference that bites |
| --- | --- | --- |
| `class Cart { constructor(items) { … } }` | `class Cart:` + `def __init__(self, items):` | Construction is two steps (`__new__` allocates, `__init__` fills in) |
| `this` (set by the call site) | `self` (an explicit first parameter) | `self` is bound when you look the method up, so it is never "lost" |
| `obj.method.bind(obj)` | `obj.method` | A bound method already remembers its instance |
| Class field `items = []` (one array **per instance**) | Class attribute `items = []` (one list **shared** by all instances) | The most common Python class bug, see 2.1.4 |
| `static create() {}` | `@classmethod` / `@staticmethod` | `@classmethod` receives the class, so it builds the right subclass |
| `get total() {}` / `set total(v) {}` | `@property` / `@total.setter` | Same idea; Python has a deleter too |
| `#balance` (engine-enforced) / TS `private` (compile-time) | `_balance` (convention) / `__balance` (name mangling) | Nothing in Python is truly private |
| `Counter.prototype.reset = …` | `Counter.reset = reset` | Python classes are mutable objects too; instances see the change |
| `class` expression, `Object.create` | `type(name, bases, namespace)` | A class is built at run time by calling its metaclass |

The next cell is this unit's setup. `raised_message(thunk)` is new: like
`raised_error_name` from Phase 1, but it also returns the error message.

In [2]:
import dis
import functools
import inspect
import sys
import types
from abc import ABC, abstractmethod
from collections.abc import Callable, Mapping
from dataclasses import dataclass
from datetime import date
from decimal import ROUND_HALF_UP, Decimal
from typing import Any, ClassVar, NamedTuple, Self

PAISE = Decimal("0.01")


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


def raised_message(thunk: Callable[[], object]) -> str:
    """Call thunk and return "ErrorName: message" for the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return f"{type(error).__name__}: {error}"
    return "no error"


print(f"{sys.version_info[:2]=}")

sys.version_info[:2]=(3, 15)


### 2.1.1 Why classes

A class **bundles state with the behaviour that keeps that state valid**. Without a
class you can still keep data in a `dict` and write functions that work on it, but
nothing ties the rules to the data: any code can put the dict into an invalid state.

The next cell keeps stock levels in a plain dict. `ship_stock` refuses to go below
zero, but the last assignment ignores the rule completely.

In [3]:
def make_stock_item(sku: str, quantity: int) -> dict[str, Any]:
    """Create a stock record as a plain dict."""
    return {"sku": sku, "quantity": quantity}


def ship_stock(stock_item: dict[str, Any], units: int) -> None:
    """Ship units, refusing to go below zero."""
    if units > stock_item["quantity"]:
        raise ValueError(f"only {stock_item['quantity']} left")
    stock_item["quantity"] -= units


soap_stock = make_stock_item("SOAP", 10)
ship_stock(soap_stock, 3)
soap_stock["quantity"] = -50  # nothing stops this: the rule lives in ship_stock only
print(f"{soap_stock=}")

soap_stock={'sku': 'SOAP', 'quantity': -50}


The same idea as a class. The quantity can only change through `receive()` and
`ship()`, and the read-only `quantity` property (2.1.6) refuses direct assignment.
Notice the `AttributeError` at the end: the invalid state is now impossible from
outside.

In [4]:
class StockItem:
    """Stock for one SKU; quantity changes only through receive() and ship()."""

    def __init__(self, sku: str, quantity: int) -> None:
        self.sku = sku
        self._quantity = quantity

    @property
    def quantity(self) -> int:
        """Units on hand (read-only from outside)."""
        return self._quantity

    def receive(self, units: int) -> None:
        """Add units from a delivery."""
        self._quantity += units

    def ship(self, units: int) -> None:
        """Remove units, refusing to go below zero."""
        if units > self._quantity:
            raise ValueError(f"only {self._quantity} left")
        self._quantity -= units


soap = StockItem("SOAP", 10)
soap.ship(3)
soap.receive(5)
print(f"{soap.quantity=}")
print(raised_message(lambda: setattr(soap, "quantity", -50)))

soap.quantity=12
AttributeError: property 'quantity' of 'StockItem' object has no setter


A class is not always the answer. Python has three lighter tools, and idiomatic code
uses them first:

| You need | Best tool | Why |
| --- | --- | --- |
| A stateless calculation | a **function** | Nothing to remember between calls (`add_gst(amount)`) |
| A group of related functions and constants, one copy | a **module** | A module is already a singleton namespace (`import settings`) |
| A record of data with little behaviour | a **dataclass** or `NamedTuple` (2.6, 1.5.6) | `__init__`, `__repr__`, `__eq__` are generated for you |
| State plus rules that must always hold (invariants) | a **class** | Methods guard the state |
| Several interchangeable implementations | a **class** + ABC / Protocol (2.3) | Callers depend on the shared interface |

A common smell is a class with only `__init__` and one method: it is a function in
disguise. Both versions below compute the same thing; the function is shorter and
easier to test.

In [5]:
class GstCalculator:
    """A class that only wraps one function (a smell)."""

    def __init__(self, rate_percent: Decimal) -> None:
        self.rate_percent = rate_percent

    def calculate(self, amount: Decimal) -> Decimal:
        """Return amount plus GST, rounded to the paisa."""
        return (amount * (1 + self.rate_percent / 100)).quantize(PAISE, ROUND_HALF_UP)


def add_gst(amount: Decimal, rate_percent: Decimal = Decimal("18")) -> Decimal:
    """The same thing as one function."""
    return (amount * (1 + rate_percent / 100)).quantize(PAISE, ROUND_HALF_UP)


print(f"{GstCalculator(Decimal('18')).calculate(Decimal('499'))=}")
print(f"{add_gst(Decimal('499'))=}")

GstCalculator(Decimal('18')).calculate(Decimal('499'))=Decimal('588.82')
add_gst(Decimal('499'))=Decimal('588.82')


A **module** is an object too: `import math` gives you a `module` instance whose
attributes are the functions and constants defined in the file. That is why Python
needs no "static utility class" (`class MathUtils { static … }` in Java). The cell
builds a module object by hand to show it is just a namespace.

In [6]:
import math

print(f"{type(math).__name__=}, {math.tau=:.4f}")
settings = types.ModuleType("settings")  # what `import settings` would give you
settings.DEBUG = False
settings.PAGE_SIZE = 20
print(f"{type(settings).__name__=}, {settings.PAGE_SIZE=}")

type(math).__name__='module', math.tau=6.2832
type(settings).__name__='module', settings.PAGE_SIZE=20


For plain records, a **dataclass** (Unit 2.6) writes the boring methods for you.
Here `@dataclass` generated `__init__`, a readable `__repr__` and a value-based
`__eq__` from three annotated lines.

In [7]:
@dataclass
class ShippingAddress:
    """A plain record: the decorator writes __init__, __repr__ and __eq__."""

    city: str
    pin_code: str


home = ShippingAddress("Pune", "411001")
print(home)
print(f"{home == ShippingAddress('Pune', '411001')=}")

ShippingAddress(city='Pune', pin_code='411001')
home == ShippingAddress('Pune', '411001')=True


### 2.1.2 The `class` statement

`class` is an **executable statement**, like `def` (1.10.1). When Python reaches it:

1. It creates a fresh namespace (a dict).
2. It **runs the class body** top to bottom inside that namespace. Every assignment
   and every `def` in the body becomes an entry.
3. It calls the **metaclass**, normally `type`, as `type(name, bases, namespace)`.
   The result is the class object.
4. It binds the class name to that object in the enclosing scope.

The body is ordinary code, so `print` runs while the class is being defined, before
any instance exists. Watch the namespace grow as the body runs.

In [8]:
print("before the class statement")


class Checkout:
    """Prints while its body runs."""

    print("  class body starts")
    currency = "INR"
    print(f"  {sorted(name for name in locals() if not name.startswith('__'))=}")

    def total(self) -> int:
        """Never called here."""
        return 0

    print(f"  {sorted(name for name in locals() if not name.startswith('__'))=}")


print("after the class statement")

before the class statement
  class body starts
  sorted(name for name in locals() if not name.startswith('__'))=['currency']
  sorted(name for name in locals() if not name.startswith('__'))=['currency', 'total']
after the class statement


Step 3 can be done by hand. Calling `type` with three arguments creates a class:
a name, a tuple of base classes and a namespace dict. The result behaves exactly
like a class written with the `class` statement; JavaScript's closest equivalent is
a class expression or `Object.create(proto)`.

In [9]:
def describe_coupon(self: Any) -> str:
    """A plain function that becomes a method once it is in the namespace."""
    return f"{self.code}: {self.percent_off}% off"


Coupon = type(
    "Coupon",
    (object,),
    {"percent_off": 10, "describe": describe_coupon, "__doc__": "Built by type()."},
)
welcome = Coupon()
welcome.code = "WELCOME10"
print(f"{welcome.describe()=}")
print(f"{type(Coupon)=}, {Coupon.__name__=}, {Coupon.__bases__=}")
print(f"{type(Checkout) is type=}  (every ordinary class is an instance of type)")

welcome.describe()='WELCOME10: 10% off'
type(Coupon)=<class 'type'>, Coupon.__name__='Coupon', Coupon.__bases__=(<class 'object'>,)
type(Checkout) is type=True  (every ordinary class is an instance of type)


After creation, the namespace is stored as `ClassName.__dict__`. It is a
**`mappingproxy`**: a read-only view of the real dict. You cannot write through it,
but `setattr` (or `Checkout.currency = …`) works, because it goes through
`type.__setattr__`, which also invalidates CPython's method-lookup caches.

The keys include entries Python added for you: `__module__`, `__qualname__`,
`__doc__`, descriptors for `__dict__` and `__weakref__`, and two newer ones:
`__firstlineno__` and `__static_attributes__` (*New in 3.13*, explained in 2.1.3).

In [10]:
print(f"{type(Checkout.__dict__).__name__=}")
print(sorted(Checkout.__dict__))
try:
    Checkout.__dict__["currency"] = "USD"
except TypeError as error:
    print(f"{type(error).__name__}: {error}")
Checkout.currency = "USD"  # goes through type.__setattr__, which updates the real dict
print(f"{Checkout.__dict__['currency']=}")

type(Checkout.__dict__).__name__='mappingproxy'
['__dict__', '__doc__', '__firstlineno__', '__module__', '__static_attributes__', '__weakref__', 'currency', 'total']
TypeError: 'mappingproxy' object does not support item assignment
Checkout.__dict__['currency']='USD'


#### Under the hood: the bytecode of a `class` statement

Compiling a tiny class shows the steps above. `LOAD_BUILD_CLASS` pushes the built-in
`__build_class__`, the class body is a separate **code object** turned into a
function by `MAKE_FUNCTION`, and the `CALL` runs it: `__build_class__` executes the
body function in a new namespace and then calls the metaclass. Code objects are
shown by name only (their reprs contain memory addresses).

> **Note:** opcode names change between Python versions; the shape stays the same.

In [11]:
class_source = "class Demo:\n    rate = 18\n\n    def total(self):\n        return 0\n"
module_code = compile(class_source, "<demo>", "exec")
for instruction in dis.get_instructions(module_code):
    if isinstance(instruction.argval, types.CodeType):
        shown_argument = f"<code object {instruction.argval.co_name}>"
    else:
        shown_argument = instruction.argrepr
    print(f"{instruction.opname:<20} {shown_argument}")

RESUME               
LOAD_BUILD_CLASS     
PUSH_NULL            
LOAD_CONST           <code object Demo>
MAKE_FUNCTION        
LOAD_CONST           'Demo'
CALL                 
STORE_NAME           Demo
LOAD_COMMON_CONSTANT None
RETURN_VALUE         


### 2.1.3 Object creation

Calling a class, `StockItem("SOAP", 10)`, runs `type.__call__`, which does two
separate steps:

1. **`__new__(cls, *args)`** allocates and returns a new, empty instance. It is a
   static method that receives the class. The default, `object.__new__`, is almost
   always enough.
2. **`__init__(self, *args)`** initializes that instance: it sets attributes. It
   must return `None`.

JavaScript's `constructor` does both at once. The traced class below prints each
step; notice the instance has an empty `__dict__` between the two.

In [12]:
class TracedOrder:
    """Prints each creation step."""

    def __new__(cls, order_id: str) -> Self:
        print(f"__new__(cls={cls.__name__}, order_id={order_id!r}) allocates")
        instance = super().__new__(cls)  # object.__new__ takes only the class
        print(f"  allocated, vars={vars(instance)}")
        return instance

    def __init__(self, order_id: str) -> None:
        print(f"__init__(self, order_id={order_id!r}) fills in attributes")
        self.order_id = order_id


first_order = TracedOrder("ORD-1")
print(f"{vars(first_order)=}")

__new__(cls=TracedOrder, order_id='ORD-1') allocates
  allocated, vars={}
__init__(self, order_id='ORD-1') fills in attributes
vars(first_order)={'order_id': 'ORD-1'}


The same two steps, done by hand. This is what `TracedOrder("ORD-2")` does for you.

In [13]:
manual_order = TracedOrder.__new__(TracedOrder, "ORD-2")
TracedOrder.__init__(manual_order, "ORD-2")
print(f"{vars(manual_order)=}")

__new__(cls=TracedOrder, order_id='ORD-2') allocates
  allocated, vars={}
__init__(self, order_id='ORD-2') fills in attributes
vars(manual_order)={'order_id': 'ORD-2'}


When do you write `__new__`? Mainly when subclassing an **immutable** type such as
`int`, `str` or `tuple`: by the time `__init__` runs the value is already fixed, so
validation or conversion has to happen in `__new__`.

In [14]:
class Paise(int):
    """An int that refuses negative amounts; int is immutable, so this needs __new__."""

    def __new__(cls, amount: int) -> Self:
        if amount < 0:
            raise ValueError("amount cannot be negative")
        return super().__new__(cls, amount)


print(f"{Paise(2500) + 100=}, {isinstance(Paise(1), int)=}")
print(raised_message(lambda: Paise(-1)))

Paise(2500) + 100=2600, isinstance(Paise(1), int)=True
ValueError: amount cannot be negative


If `__new__` returns something that is **not** an instance of the class,
`type.__call__` skips `__init__` entirely. That is legal but surprising, so keep it
for special cases (caching, singletons, factories).

In [15]:
class Surprising:
    """__new__ returns an int, so __init__ never runs."""

    def __new__(cls) -> Any:
        return 42

    def __init__(self) -> None:
        print("never printed")


print(f"{Surprising()=}")

Surprising()=42


**Explicit `self`.** A method is an ordinary function whose first parameter receives
the instance. `soap.ship(2)` is shorthand for `StockItem.ship(soap, 2)`. The name
`self` is only a convention (any name works), but always use it.

In [16]:
soap = StockItem("SOAP", 10)
soap.ship(2)  # what you write
StockItem.ship(soap, 2)  # what it means
print(f"{soap.quantity=}")

soap.quantity=6


**Inspecting an instance.** `vars(instance)` returns the instance's own attribute
dict (the same object as `instance.__dict__`). `instance.__class__` is its class,
the same as `type(instance)`.

`__static_attributes__` (*New in 3.13*) is a tuple of the names assigned through
`self.<name>` anywhere in the class body, collected by the compiler. Tools use it
to know an instance's likely attributes without creating one.

In [17]:
print(f"{vars(soap)=}")
print(f"{soap.__dict__ is vars(soap)=}")
print(f"{soap.__class__=}, {type(soap) is soap.__class__=}")
if sys.version_info >= (3, 13):
    print(f"{StockItem.__static_attributes__=}")
else:
    print("__static_attributes__ needs Python 3.13+")

vars(soap)={'sku': 'SOAP', '_quantity': 6}
soap.__dict__ is vars(soap)=True
soap.__class__=<class '__main__.StockItem'>, type(soap) is soap.__class__=True
StockItem.__static_attributes__=('_quantity', 'sku')


#### Under the hood: what an instance costs

An ordinary instance is a small header plus a pointer to its attribute storage.
Since 3.11 CPython stores the attributes of most instances inline and only builds a
real `__dict__` when you ask for one (`vars()` does). Instances of the same class
share one key table (key-sharing dicts, 1.6.1). Unit 2.5 shows how `__slots__`
removes the dict altogether.

> **Note:** the byte counts depend on the Python version and platform.

In [18]:
print(f"{sys.getsizeof(soap)=} bytes (the instance itself)")
print(f"{sys.getsizeof(vars(soap))=} bytes (its materialized __dict__)")
print(f"{sys.getsizeof(StockItem)=} bytes (the class object)")

sys.getsizeof(soap)=48 bytes (the instance itself)
sys.getsizeof(vars(soap))=288 bytes (its materialized __dict__)
sys.getsizeof(StockItem)=1728 bytes (the class object)


### 2.1.4 Instance vs class attributes

Assignments in the **class body** create **class attributes**, shared by every
instance. Assignments to `self.<name>` create **instance attributes**, stored in each
instance's own `__dict__`.

Reading `instance.name` searches in this order (descriptors such as properties
change this slightly; see 2.1.6 and 2.5):

1. the instance's `__dict__`
2. the class's `__dict__`
3. each base class's `__dict__`, in MRO order (2.2.3)

```mermaid
graph LR
    instance["soap.__dict__"] -->|not found| klass["Product.__dict__"]
    klass -->|not found| base["object.__dict__"]
    base -->|not found| error["AttributeError"]
```

<details><summary>ASCII version</summary>

```text
soap.__dict__ --miss--> Product.__dict__ --miss--> object.__dict__ --miss--> AttributeError
```

</details>

In [19]:
class Product:
    """currency is a class attribute; name and price are instance attributes."""

    currency = "INR"

    def __init__(self, name: str, price: int) -> None:
        self.name = name
        self.price = price


soap = Product("Soap", 49)
tea = Product("Tea", 120)
print(f"{vars(soap)=}")
print(f"{soap.currency=} (found on the class)")
Product.currency = "USD"
print(f"{tea.currency=} (every instance sees the change)")
Product.currency = "INR"

vars(soap)={'name': 'Soap', 'price': 49}
soap.currency='INR' (found on the class)
tea.currency='USD' (every instance sees the change)


**Shadowing.** Assigning through an instance never changes the class: it creates an
instance attribute with the same name, which hides the class attribute for that one
instance. Deleting it makes the class attribute visible again.

In [20]:
tea.currency = "EUR"  # creates an instance attribute that shadows the class one
print(f"{vars(tea)=}")
print(f"{tea.currency=}, {soap.currency=}, {Product.currency=}")
del tea.currency
print(f"{tea.currency=} (the class attribute is visible again)")

vars(tea)={'name': 'Tea', 'price': 120, 'currency': 'EUR'}
tea.currency='EUR', soap.currency='INR', Product.currency='INR'
tea.currency='INR' (the class attribute is visible again)


The lookup continues into base classes. `find_attribute_owner` repeats the search
by hand (without descriptors) to show where each name is found. `__repr__` comes
all the way from `object`.

In [21]:
def find_attribute_owner(instance: object, attribute_name: str) -> str:
    """Report where attribute lookup finds attribute_name (ignoring descriptors)."""
    if attribute_name in vars(instance):
        return "instance __dict__"
    for klass in type(instance).__mro__:
        if attribute_name in vars(klass):
            return f"class {klass.__name__}"
    return "not found -> AttributeError"


class DigitalProduct(Product):
    """Adds a class attribute of its own."""

    shipping_required = False


ebook = DigitalProduct("Python e-book", 499)
for attribute_name in ["name", "shipping_required", "currency", "__repr__", "weight"]:
    print(f"{attribute_name:<18} -> {find_attribute_owner(ebook, attribute_name)}")

name               -> instance __dict__
shipping_required  -> class DigitalProduct
currency           -> class Product
__repr__           -> class object
weight             -> not found -> AttributeError


**The shared mutable class-attribute trap.** A mutable value (list, dict, set) in
the class body is created **once**, when the class body runs, and every instance
shares it. `self.line_items.append(...)` does not assign, it mutates the shared list.

> **Warning:** this is the opposite of JavaScript. In JS, a class field
> `lineItems = [];` runs once **per instance**, inside the constructor.

In [22]:
class Invoice:
    """BUG: line_items is one list shared by every invoice."""

    line_items: list[str] = []

    def add_line(self, description: str) -> None:
        """Append a line (mutates the shared class attribute)."""
        self.line_items.append(description)


march_invoice = Invoice()
april_invoice = Invoice()
march_invoice.add_line("Hosting")
print(f"{april_invoice.line_items=}")
print(f"{march_invoice.line_items is Invoice.line_items=}")

april_invoice.line_items=['Hosting']
march_invoice.line_items is Invoice.line_items=True


The fix: create mutable state in `__init__`, so each instance gets its own object.

In [23]:
class FixedInvoice:
    """Each invoice gets its own list."""

    def __init__(self) -> None:
        self.line_items: list[str] = []

    def add_line(self, description: str) -> None:
        """Append a line to this invoice only."""
        self.line_items.append(description)


march_invoice = FixedInvoice()
april_invoice = FixedInvoice()
march_invoice.add_line("Hosting")
print(f"{march_invoice.line_items=}, {april_invoice.line_items=}")

march_invoice.line_items=['Hosting'], april_invoice.line_items=[]


**Predict the output.**

```python
class Team:
    size = 0
    members = []

alpha, beta = Team(), Team()
alpha.size += 1
alpha.members += ["Asha"]
print(Team.size, beta.size, Team.members)
print("size" in vars(alpha), "members" in vars(alpha))
```

<details><summary>Answer</summary>

```text
0 0 ['Asha']
True True
```

`alpha.size += 1` reads `Team.size` (0), adds 1 and **assigns** to `alpha.size`,
creating an instance attribute; the class keeps 0. `alpha.members += ["Asha"]`
calls `list.__iadd__`, which **mutates the shared list in place**, and then also
assigns the same list to `alpha.members`. So both names end up in `vars(alpha)`,
but only the list change is visible to `beta` and the class.

</details>

Run the next cell to check your prediction.

In [24]:
class Team:
    size: int = 0
    members: list[str] = []


alpha, beta = Team(), Team()
alpha.size += 1
alpha.members += ["Asha"]
print(Team.size, beta.size, Team.members)
print("size" in vars(alpha), "members" in vars(alpha))

0 0 ['Asha']
True True


**Good uses of class attributes:** constants and counters. Constants use
`UPPER_CASE` (PEP 8) and can be annotated `ClassVar` so type checkers know they
belong to the class. A counter must be updated **through the class**
(`Ticket.issued_count += 1`); updating it through `self` would create an instance
attribute, as the predict question showed.

In [25]:
class Ticket:
    """Class constants in UPPER_CASE; a class-level counter for ticket numbers."""

    MAX_SEATS: ClassVar[int] = 4
    issued_count: ClassVar[int] = 0

    def __init__(self, seats: int) -> None:
        if not 1 <= seats <= Ticket.MAX_SEATS:
            raise ValueError(f"seats must be 1-{Ticket.MAX_SEATS}")
        Ticket.issued_count += 1  # write through the class, not self
        self.ticket_number = Ticket.issued_count
        self.seats = seats


tickets = [Ticket(2), Ticket(1), Ticket(4)]
print(f"{[ticket.ticket_number for ticket in tickets]=}, {Ticket.issued_count=}")
print(raised_message(lambda: Ticket(9)))

[ticket.ticket_number for ticket in tickets]=[1, 2, 3], Ticket.issued_count=3
ValueError: seats must be 1-4


The broken version: `self.issued_count += 1` reads the class value (0) and writes
an instance attribute every time, so the class counter never moves.

In [26]:
class BrokenTicket:
    """BUG: the counter is updated through self."""

    issued_count = 0

    def __init__(self) -> None:
        self.issued_count += 1  # reads the class value, writes an instance attribute


broken_tickets = [BrokenTicket(), BrokenTicket()]
print(f"{BrokenTicket.issued_count=}")
print(f"{[ticket.issued_count for ticket in broken_tickets]=}")

BrokenTicket.issued_count=0
[ticket.issued_count for ticket in broken_tickets]=[1, 1]


### 2.1.5 Methods

A function stored on a class is a plain function. When you read it **through an
instance**, Python returns a **bound method**: a small object that remembers the
function (`__func__`) and the instance (`__self__`). Calling the bound method calls
the function with the instance as the first argument.

`Money` below is the start of a value type that Unit 2.4 will turn into a full
numeric class.

In [27]:
class Money:
    """An amount of money in one currency (Unit 2.4 adds arithmetic)."""

    def __init__(self, amount: Decimal, currency: str = "INR") -> None:
        self.amount = amount
        self.currency = currency

    def formatted(self) -> str:
        """Return e.g. 'INR 250.00'."""
        return f"{self.currency} {self.amount:.2f}"


price = Money(Decimal("250"))
print(f"{type(Money.formatted).__name__=}  (a plain function on the class)")
bound_formatted = price.formatted
print(f"{type(bound_formatted).__name__=}")
print(f"{bound_formatted.__self__ is price=}")
print(f"{bound_formatted.__func__ is Money.formatted=}")
print(f"{bound_formatted()=}")

type(Money.formatted).__name__='function'  (a plain function on the class)
type(bound_formatted).__name__='method'
bound_formatted.__self__ is price=True
bound_formatted.__func__ is Money.formatted=True
bound_formatted()='INR 250.00'


Because the bound method carries its instance, you can store it, pass it as a
callback or put it in a dict, and it still works. No `.bind(this)` is ever needed.

Each attribute access builds a **new** bound-method object, so `is` is `False`,
but two bound methods of the same function and instance compare equal.

In [28]:
callbacks: list[Callable[[], str]] = [
    Money(Decimal("99")).formatted,
    Money(Decimal("5"), "USD").formatted,
]
print([callback() for callback in callbacks])
print(f"{price.formatted is price.formatted=}  (a new bound method on each access)")
print(f"{price.formatted == price.formatted=}")

['INR 99.00', 'USD 5.00']
price.formatted is price.formatted=False  (a new bound method on each access)
price.formatted == price.formatted=True


#### Under the hood: binding is the descriptor protocol

Functions have a `__get__` method. When attribute lookup finds a function on the
class, it calls `function.__get__(instance, owner)`, which returns the bound method.
`types.MethodType(function, instance)` builds the same object directly. Unit 2.5
covers descriptors in full; `@property`, `@classmethod` and `@staticmethod` all use
this hook.

In [29]:
function_object = Money.__dict__["formatted"]
manually_bound = function_object.__get__(price, Money)
print(f"{manually_bound()=}, {type(manually_bound) is types.MethodType=}")
print(f"{types.MethodType(function_object, price)()=}")

manually_bound()='INR 250.00', type(manually_bound) is types.MethodType=True
types.MethodType(function_object, price)()='INR 250.00'


**`@classmethod`** receives the **class** as its first argument (`cls`) instead of
an instance. Its classic use is an **alternative constructor**: another way to build
an instance, named after its input (`from_string`, `from_paise`). Because it calls
`cls(...)` rather than `Money(...)`, it builds whichever class it was called on.

**`@staticmethod`** receives neither. It is a plain function that lives in the class
namespace because it belongs to the class conceptually (here, currency-code
validation used by the constructors).

In [30]:
class Money:
    """An amount of money in one currency."""

    def __init__(self, amount: Decimal, currency: str = "INR") -> None:
        if not Money.is_valid_currency(currency):
            raise ValueError(f"invalid currency code: {currency!r}")
        self.amount = amount
        self.currency = currency

    @classmethod
    def from_string(cls, text: str) -> Self:
        """Parse 'INR 250.00' (alternative constructor)."""
        currency, amount_text = text.split()
        return cls(Decimal(amount_text), currency)

    @classmethod
    def from_paise(cls, paise: int) -> Self:
        """Build an INR amount from whole paise."""
        return cls(Decimal(paise) / 100, "INR")

    @staticmethod
    def is_valid_currency(code: str) -> bool:
        """Three upper-case letters, like 'INR' or 'USD'."""
        return len(code) == 3 and code.isalpha() and code.isupper()

    def formatted(self) -> str:
        """Return e.g. 'INR 250.00'."""
        return f"{self.currency} {self.amount:.2f}"


print(f"{Money.from_string('USD 19.99').formatted()=}")
print(f"{Money.from_paise(25_050).formatted()=}")

Money.from_string('USD 19.99').formatted()='USD 19.99'
Money.from_paise(25_050).formatted()='INR 250.50'


**Subclass-aware.** `Refund` inherits `from_paise`, and because the method uses
`cls`, `Refund.from_paise(...)` returns a `Refund`. A version that hard-codes
`Money(...)` silently returns the wrong type for every subclass.

In [31]:
class Refund(Money):
    """Money flowing back to a customer."""

    def formatted(self) -> str:
        """Prefix the amount with a minus sign."""
        return f"-{super().formatted()}"


class HardCodedMoney(Money):
    """BUG: the alternative constructor ignores cls."""

    @classmethod
    def from_paise(cls, paise: int) -> Money:
        return Money(Decimal(paise) / 100, "INR")


class HardCodedRefund(HardCodedMoney):
    """Inherits the broken constructor."""


refund = Refund.from_paise(4_950)
print(f"{type(refund).__name__=}, {refund.formatted()=}")
print(f"{type(HardCodedRefund.from_paise(4_950)).__name__=}  (wrong class)")

type(refund).__name__='Refund', refund.formatted()='-INR 49.50'
type(HardCodedRefund.from_paise(4_950)).__name__='Money'  (wrong class)


In application code the most common alternative constructor is **`from_dict`**:
building an object from a JSON payload or a database row. The same `cls(...)` rule
makes it subclass-aware. (Lab 2.1 exercise 3 asks for a stricter one, with
validation and cleaning.)

In [32]:
class ApiMoney(Money):
    """Money as it arrives in JSON payloads."""

    @classmethod
    def from_dict(cls, payload: Mapping[str, str]) -> Self:
        """Build from {"amount": "250.00", "currency": "INR"}."""
        return cls(Decimal(payload["amount"]), payload.get("currency", "INR"))


class ApiRefund(ApiMoney):
    """Inherits from_dict and gets ApiRefund instances from it."""


payload = {"amount": "99.90", "currency": "USD"}
print(f"{ApiMoney.from_dict(payload).formatted()=}")
print(f"{type(ApiRefund.from_dict({'amount': '10'})).__name__=}")

ApiMoney.from_dict(payload).formatted()='USD 99.90'
type(ApiRefund.from_dict({'amount': '10'})).__name__='ApiRefund'


The standard library uses the same pattern everywhere: `dict.fromkeys`,
`int.from_bytes`, `date.fromisoformat`, `Decimal.from_float`. Each one is a
`classmethod`, so subclasses of these types get instances of themselves.

In [33]:
print(f"{dict.fromkeys(['soap', 'tea'], 0)=}")
print(f"{int.from_bytes(bytes([1, 0]), 'big')=}")
print(f"{date.fromisoformat('2026-11-07')=}")
print(f"{Decimal.from_float(0.5)=}")
print(f"{type(dict.__dict__['fromkeys']).__name__=}")

dict.fromkeys(['soap', 'tea'], 0)={'soap': 0, 'tea': 0}
int.from_bytes(bytes([1, 0]), 'big')=256
date.fromisoformat('2026-11-07')=datetime.date(2026, 11, 7)
Decimal.from_float(0.5)=Decimal('0.5')
type(dict.__dict__['fromkeys']).__name__='classmethod_descriptor'


A static method can be called through the class or an instance; it receives no
automatic argument either way.

In [34]:
print(f"{Money.is_valid_currency('INR')=}")
print(f"{Money.from_paise(100).is_valid_currency('rupees')=}")
print(raised_message(lambda: Money(Decimal("1"), "inr")))

Money.is_valid_currency('INR')=True
Money.from_paise(100).is_valid_currency('rupees')=False
ValueError: invalid currency code: 'inr'


**Decision table**

| Kind | First argument | Use it when | Example |
| --- | --- | --- | --- |
| Instance method | `self` (the instance) | It reads or changes one object's state | `cart.add(item)` |
| `@classmethod` | `cls` (the class it was called on) | It builds instances or works with class-level data | `Money.from_paise(500)` |
| `@staticmethod` | nothing | It is a helper that belongs with the class but needs no state | `Money.is_valid_currency("INR")` |
| Module-level function | nothing | It does not belong to any one class | `add_gst(amount)` |

The cell calls each kind through an instance, a subclass instance and the subclass
itself. Notice that `cls` follows the subclass, and that an instance method called on
the class has no `self` to receive.

In [35]:
class MethodKinds:
    """One method of each kind."""

    def instance_method(self) -> str:
        return f"self is a {type(self).__name__} instance"

    @classmethod
    def class_method(cls) -> str:
        return f"cls is {cls.__name__}"

    @staticmethod
    def static_method() -> str:
        return "nothing passed"


class SubKinds(MethodKinds):
    """Inherits all three."""


for caller in [MethodKinds(), SubKinds(), SubKinds]:
    caller_label = getattr(caller, "__name__", f"{type(caller).__name__}()")
    for method_name in ["instance_method", "class_method", "static_method"]:
        result = raised_message(getattr(caller, method_name))
        if result == "no error":
            result = getattr(caller, method_name)()
        print(f"{caller_label:<14} {method_name:<16} {result}")

MethodKinds()  instance_method  self is a MethodKinds instance
MethodKinds()  class_method     cls is MethodKinds
MethodKinds()  static_method    nothing passed
SubKinds()     instance_method  self is a SubKinds instance
SubKinds()     class_method     cls is SubKinds
SubKinds()     static_method    nothing passed
SubKinds       instance_method  TypeError: MethodKinds.instance_method() missing 1 required positional argument: 'self'
SubKinds       class_method     cls is SubKinds
SubKinds       static_method    nothing passed


### 2.1.6 Properties

`@property` turns a method into an attribute that runs code when it is read.
Callers write `plan.total_fee`, not `plan.total_fee()`. Add a `.setter` to run code
on assignment and a `.deleter` to run code on `del`.

| Pattern | How | Example below |
| --- | --- | --- |
| Computed attribute | getter only, derived from other state | `total_fee` |
| Read-only attribute | getter only, backed by `_name` | `subscription_id` |
| Validation on assignment | getter + setter that checks, then stores `_name` | `months` |
| Clean-up on delete | getter + setter + deleter | `auth_token` |

First, a computed and a read-only property. Assigning to either raises
`AttributeError`, and the computed value is recalculated after `months` changes.

In [36]:
class Subscription:
    """A paid plan billed monthly."""

    def __init__(self, subscription_id: str, monthly_fee: Decimal, months: int) -> None:
        self._subscription_id = subscription_id
        self.monthly_fee = monthly_fee
        self.months = months

    @property
    def subscription_id(self) -> str:
        """Read-only: there is no setter."""
        return self._subscription_id

    @property
    def total_fee(self) -> Decimal:
        """Computed on every access."""
        return self.monthly_fee * self.months


plan = Subscription("SUB-7", Decimal("299"), 12)
print(f"{plan.total_fee=}")
plan.months = 6
print(f"{plan.total_fee=} (recomputed)")
for attribute_name in ["subscription_id", "total_fee"]:
    print(raised_message(functools.partial(setattr, plan, attribute_name, "x")))

plan.total_fee=Decimal('3588')
plan.total_fee=Decimal('1794') (recomputed)
AttributeError: property 'subscription_id' of 'Subscription' object has no setter
AttributeError: property 'total_fee' of 'Subscription' object has no setter


**Validation on assignment.** The subclass adds a `months` property whose setter
rejects bad values. The inherited `__init__` already does `self.months = months`, and
that assignment now goes through the setter too, so validation also happens at
construction. A failed assignment leaves the old value in place.

In [37]:
class ValidatedSubscription(Subscription):
    """months must be a whole number from 1 to 36, on every assignment."""

    @property
    def months(self) -> int:
        """Billing period in months."""
        return self._months

    @months.setter
    def months(self, value: int) -> None:
        if not isinstance(value, int) or not 1 <= value <= 36:
            raise ValueError(f"months must be an int from 1 to 36, got {value!r}")
        self._months = value


plan = ValidatedSubscription("SUB-8", Decimal("299"), 12)
print(raised_message(lambda: setattr(plan, "months", 48)))
print(f"{plan.months=} (unchanged after the failed assignment)")
print(raised_message(lambda: ValidatedSubscription("SUB-9", Decimal("99"), 0)))

ValueError: months must be an int from 1 to 36, got 48
plan.months=12 (unchanged after the failed assignment)
ValueError: months must be an int from 1 to 36, got 0


**Deleter.** `del session.auth_token` runs the deleter, which can clean up instead
of just removing a value. Here it records a logout and leaves the token as `None`.

In [38]:
class UserSession:
    """A login session whose token can be revoked with del."""

    def __init__(self, username: str) -> None:
        self.username = username
        self._auth_token: str | None = None
        self.events: list[str] = []

    @property
    def auth_token(self) -> str | None:
        """The current token, or None when logged out."""
        return self._auth_token

    @auth_token.setter
    def auth_token(self, token: str) -> None:
        self._auth_token = token
        self.events.append("login")

    @auth_token.deleter
    def auth_token(self) -> None:
        self._auth_token = None
        self.events.append("logout")


session = UserSession("asha")
session.auth_token = "tok_123"
del session.auth_token
print(f"{session.auth_token=}, {session.events=}")

session.auth_token=None, session.events=['login', 'logout']


`@property` is shorthand for `property(fget, fset, fdel, doc)`. The decorator form
is preferred, but the call form shows what a property object holds: up to three
functions and a docstring.

In [39]:
class SpeakerVolume:
    """Volume clamped to 0-10, written with the property() call form."""

    def __init__(self) -> None:
        self._level = 5

    def _get_level(self) -> int:
        return self._level

    def _set_level(self, value: int) -> None:
        self._level = max(0, min(10, value))

    level = property(_get_level, _set_level, doc="Volume from 0 to 10, clamped.")


speaker = SpeakerVolume()
speaker.level = 14
print(f"{speaker.level=}")
months_property = ValidatedSubscription.__dict__["months"]
print(f"{type(months_property).__name__=}, {months_property.fget.__name__=}")
print(f"{months_property.fset is not None=}, {months_property.fdel=}")
print(f"{SpeakerVolume.level.__doc__=}")

speaker.level=10
type(months_property).__name__='property', months_property.fget.__name__='months'
months_property.fset is not None=True, months_property.fdel=None
SpeakerVolume.level.__doc__='Volume from 0 to 10, clamped.'


**Uniform access principle.** Callers should not care whether a value is stored or
computed. Start with a plain attribute; if you later need validation or
normalization, turn it into a property and **no caller changes**. That is why Python
code does not write `get_email()` / `set_email()` "just in case", as Java code
often does. JavaScript getters and setters give the same freedom.

In [40]:
def send_welcome(account: Any) -> str:
    """Client code: only reads account.email."""
    return f"Welcome mail to {account.email}"


class AccountV1:
    """Version 1: a plain attribute."""

    def __init__(self, email: str) -> None:
        self.email = email


class AccountV2:
    """Version 2: the same name, now a normalizing property."""

    def __init__(self, email: str) -> None:
        self.email = email  # goes through the setter

    @property
    def email(self) -> str:
        return self._email

    @email.setter
    def email(self, value: str) -> None:
        self._email = value.strip().lower()


for account_class in [AccountV1, AccountV2]:
    print(f"{account_class.__name__}: {send_welcome(account_class('  Asha@Example.com '))!r}")

AccountV1: 'Welcome mail to   Asha@Example.com '
AccountV2: 'Welcome mail to asha@example.com'


A computed property runs on **every** read. For an expensive value that does not
change, `functools.cached_property` computes it once and stores the result in the
instance `__dict__`; later reads never call the function again.

In [41]:
class SalesSummary:
    """grand_total is computed once, on first access."""

    def __init__(self, daily_sales: list[int]) -> None:
        self.daily_sales = daily_sales

    @functools.cached_property
    def grand_total(self) -> int:
        """Sum of every day's sales."""
        print("  computing grand_total ...")
        return sum(self.daily_sales)


summary = SalesSummary([1200, 950, 1430])
print(f"{summary.grand_total=}")
print(f"{summary.grand_total=}  (no 'computing' line: cached)")
print(f"{vars(summary)=}")

  computing grand_total ...
summary.grand_total=3580
summary.grand_total=3580  (no 'computing' line: cached)
vars(summary)={'daily_sales': [1200, 950, 1430], 'grand_total': 3580}


#### Under the hood: a property beats the instance `__dict__`

A property is a **data descriptor** stored on the class (it defines `__set__`), and
data descriptors take priority over the instance `__dict__` during lookup. Even if
a value with the same name is forced into `vars(plan)`, the property still answers.
Unit 2.5 explains the full lookup order.

In [42]:
plan = Subscription("SUB-10", Decimal("100"), 3)
vars(plan)["total_fee"] = Decimal("0")  # sneak a value into the instance dict
print(f"{vars(plan)=}")
print(f"{plan.total_fee=} (the property on the class still wins)")
print(f"{inspect.isdatadescriptor(Subscription.__dict__['total_fee'])=}")

vars(plan)={'_subscription_id': 'SUB-10', 'monthly_fee': Decimal('100'), 'months': 3, 'total_fee': Decimal('0')}
plan.total_fee=Decimal('300') (the property on the class still wins)
inspect.isdatadescriptor(Subscription.__dict__['total_fee'])=True


### 2.1.7 Encapsulation by convention

Python has no `private` keyword. Instead:

- **`_name`** (one underscore) means "internal, not part of the public API". Nothing
  enforces it; it is a message to other developers. Tools respect it:
  `from module import *` skips underscore names and IDEs hide them from completion.
- **`__name`** (two leading underscores, at most one trailing) triggers **name
  mangling**: inside a class body the compiler rewrites it to `_ClassName__name`.

The motto is "we are all consenting adults": the underscore warns, it does not lock.

In [43]:
class PaymentClient:
    """Public method send(); everything with an underscore is internal."""

    def __init__(self, api_key: str) -> None:
        self._api_key = api_key  # internal: please do not touch

    def _sign(self, payload: str) -> str:
        return f"{payload}|signed-with-{self._api_key[-4:]}"

    def send(self, payload: str) -> str:
        """Sign and 'send' a payload."""
        return self._sign(payload)


client = PaymentClient("sk_test_12345678")
print(client.send("amount=500"))
print(f"{client._api_key=}  (reachable: the underscore is only a warning)")

amount=500|signed-with-5678
client._api_key='sk_test_12345678'  (reachable: the underscore is only a warning)


**Name mangling.** `self.__pin` is stored as `_Vault__pin`. Code outside the class
that writes `vault.__pin` is not mangled, so it fails; the mangled name still works
for anyone who really wants it.

In [44]:
class Vault:
    """Stores a PIN under a mangled name."""

    def __init__(self, pin: str) -> None:
        self.__pin = pin  # stored as _Vault__pin

    def check(self, attempt: str) -> bool:
        """Compare an attempt with the stored PIN."""
        return attempt == self.__pin


vault = Vault("4321")
print(f"{vars(vault)=}")
print(f"{vault.check('4321')=}")
print(raised_message(lambda: vault.__pin))
print(f"{vault._Vault__pin=}  (still reachable by the mangled name)")

vars(vault)={'_Vault__pin': '4321'}
vault.check('4321')=True
AttributeError: 'Vault' object has no attribute '__pin'
vault._Vault__pin='4321'  (still reachable by the mangled name)


Mangling exists to prevent **accidental name clashes in subclasses**, not to hide
data. Here `Cache` and its subclass both use `__entries`, but they end up as two
different attributes, so the subclass cannot break the base class by accident.

In [45]:
class Cache:
    """Base class with its own private storage."""

    def __init__(self) -> None:
        self.__entries: dict[str, str] = {}  # becomes _Cache__entries

    def cache_size(self) -> int:
        """Entries in the base-class storage."""
        return len(self.__entries)


class UserCache(Cache):
    """Reuses the name __entries without clobbering the base class."""

    def __init__(self) -> None:
        super().__init__()
        self.__entries = ["recent", "users"]  # becomes _UserCache__entries


user_cache = UserCache()
print(f"{vars(user_cache)=}")
print(f"{user_cache.cache_size()=}  (the base class still sees its own dict)")

vars(user_cache)={'_Cache__entries': {}, '_UserCache__entries': ['recent', 'users']}
user_cache.cache_size()=0  (the base class still sees its own dict)


#### Under the hood: mangling happens at compile time

The rewrite is done by the compiler, so the mangled name is already in the
method's code object (`co_names`). Strings passed to `getattr` / `hasattr` are never
mangled, and dunder names such as `__init__` (two trailing underscores) are left
alone.

In [46]:
print(f"{Vault.check.__code__.co_names=}")
mangled_name = "_Vault__pin"
print(f"{getattr(vault, mangled_name)=}")
print(f"{hasattr(vault, '__pin')=}  (strings are not mangled)")

Vault.check.__code__.co_names=('_Vault__pin',)
getattr(vault, mangled_name)='4321'
hasattr(vault, '__pin')=False  (strings are not mangled)


**Comparison with JavaScript and TypeScript**

| | Python `_x` | Python `__x` | TypeScript `private x` | JavaScript `#x` |
| --- | --- | --- | --- | --- |
| Enforced by | nobody (convention) | the compiler renames it | the TS compiler only; erased at run time | the JS engine |
| Reachable from outside | yes | yes, as `obj._Class__x` | yes, at run time | no (`SyntaxError` outside the class) |
| Visible to reflection | `vars(obj)` | `vars(obj)` (mangled) | `Object.keys(obj)` | no |
| Main purpose | mark internal API | avoid subclass clashes | design-time checking | real privacy |

> **Note:** prefer a single underscore. Use `__x` only when you expect subclasses
> from other authors that might reuse the name. Python's closest match to `#x` is a
> closure variable (1.11), which really is unreachable from outside.

### 2.1.8 JS bridge

**Classes vs prototypes.** JavaScript classes are syntax over prototypes: every
object has a hidden `[[Prototype]]` link, and methods live on `Class.prototype`.
Python has no per-object prototype link. Every instance has a `__class__`, methods
live in the class's `__dict__`, and lookup walks the class's `__mro__`.

| JavaScript | Python |
| --- | --- |
| `Object.getPrototypeOf(obj)` | `type(obj)` / `obj.__class__` |
| prototype chain | `type(obj).__mro__` |
| `Counter.prototype.reset = function () {}` | `Counter.reset = reset` |
| `Object.setPrototypeOf(obj, Other.prototype)` | `obj.__class__ = Other` (only for compatible layouts; rarely wise) |
| `this` set by the call site (`obj.m()`, `m()`, `new`, `call`, `apply`, `bind`) | `self` bound when the method is read from the instance |
| arrow functions capture the outer `this` | lambdas and closures capture `self` like any other variable |
| `static m()`: `this` is the constructor | `@classmethod`: `cls` is the class it was called on |
| `new Counter()` | `Counter()`: a class is called like a function |

**`self` vs `this`.** In JavaScript, pulling a method off its object loses `this`:

```javascript
class ClickCounter {
  count = 0;
  increment() { this.count += 1; }
}
const counter = new ClickCounter();
const increment = counter.increment;
increment();                          // TypeError: cannot read 'count' of undefined
button.addEventListener("click", counter.increment);  // same bug
const fixed = counter.increment.bind(counter);         // the usual fix
```

In Python the bound method already carries the instance, so the same code just works.

In [47]:
class ClickCounter:
    """Counts clicks."""

    def __init__(self) -> None:
        self.count = 0

    def increment(self) -> None:
        """Add one click."""
        self.count += 1


click_counter = ClickCounter()
increment = click_counter.increment  # self is already bound
for _ in range(3):
    increment()
handlers: dict[str, Callable[[], None]] = {"click": click_counter.increment}
handlers["click"]()
print(f"{click_counter.count=}")

click_counter.count=4


Like adding to `ClickCounter.prototype`, adding a function to a Python class later
makes it available to **existing** instances, because lookup reaches the class at
call time. The second line shows the Python view of the "prototype chain".

In [48]:
def reset(self: ClickCounter) -> None:
    """Set the count back to zero."""
    self.count = 0


ClickCounter.reset = reset  # like ClickCounter.prototype.reset = function () {...}
click_counter.reset()
print(f"{click_counter.count=}")
print(f"{[klass.__name__ for klass in type(click_counter).__mro__]=}")

click_counter.count=0
[klass.__name__ for klass in type(click_counter).__mro__]=['ClickCounter', 'object']


**Predict the output.**

```python
class Greeter:
    def __init__(self, name):
        self.name = name

    def greet(self):
        return f"Hi, I am {self.name}"

asha = Greeter("Asha")
say_hello = asha.greet
asha.name = "Asha K"
print(say_hello())
print(Greeter.greet(asha))
print(Greeter.greet())
```

<details><summary>Answer</summary>

```text
Hi, I am Asha K
Hi, I am Asha K
TypeError: Greeter.greet() missing 1 required positional argument: 'self'
```

`say_hello` is bound to the **object**, not to a snapshot of its attributes, so it
sees the new name. `Greeter.greet(asha)` passes the instance explicitly. Reading the
method from the **class** gives the plain function, and with no instance there is
nothing to fill `self`: Python's version of JavaScript's lost `this`, except it
fails loudly with a clear message.

</details>

Run the next cell to check your prediction.

In [49]:
class Greeter:
    """Says hello."""

    def __init__(self, name: str) -> None:
        self.name = name

    def greet(self) -> str:
        """Introduce yourself."""
        return f"Hi, I am {self.name}"


asha = Greeter("Asha")
say_hello = asha.greet
asha.name = "Asha K"
print(say_hello())
print(Greeter.greet(asha))
print(raised_message(lambda: Greeter.greet()))  # type: ignore[call-arg]

Hi, I am Asha K
Hi, I am Asha K
TypeError: Greeter.greet() missing 1 required positional argument: 'self'


### 2.1.9 The four pillars of OOP in Python

The four classic pillars all exist in Python, but they look lighter than in Java
or TypeScript, and this phase covers each one:

| Pillar | Where in this phase | Python mechanism | Java | TypeScript |
| --- | --- | --- | --- | --- |
| Encapsulation | 2.1.6–2.1.7 | properties, `_name` convention, name mangling | `private` fields + getters/setters | `private` (compile time), `#x` (run time) |
| Inheritance | 2.2 | `class B(A)`, **multiple** inheritance, C3 MRO, mixins | single `extends` + `implements` | single `extends`, mixin functions |
| Polymorphism | 2.3.4, 2.4 | duck typing, special (dunder) methods, `singledispatch` | overriding + overloading by signature | structural typing, overload signatures (types only) |
| Abstraction | 2.3 | ABCs (checked at instantiation), Protocols (checked by type checkers) | `abstract class`, `interface` | `abstract class`, `interface` (erased at run time) |

The four cells below show one pillar each on a tiny media library.

**Encapsulation**: a read-only property protects the size.

In [50]:
class MediaFile:
    """A file in a media library."""

    def __init__(self, name: str, size_bytes: int) -> None:
        self.name = name
        self._size_bytes = size_bytes

    @property
    def size_bytes(self) -> int:
        """Read-only size."""
        return self._size_bytes

    def summary(self) -> str:
        """One-line description."""
        return f"{self.name} ({self.size_bytes:,} bytes)"


print(MediaFile("notes.txt", 2_048).summary())
print(raised_error_name(lambda: setattr(MediaFile("x", 1), "size_bytes", 0)))

notes.txt (2,048 bytes)
AttributeError


**Inheritance**: `AudioFile` reuses `MediaFile` and extends `summary()` with
`super()` (Unit 2.2).

In [51]:
class AudioFile(MediaFile):
    """A media file with a duration."""

    def __init__(self, name: str, size_bytes: int, duration_seconds: int) -> None:
        super().__init__(name, size_bytes)
        self.duration_seconds = duration_seconds

    def summary(self) -> str:
        """Add the duration to the base summary."""
        minutes, seconds = divmod(self.duration_seconds, 60)
        return f"{super().summary()}, {minutes}:{seconds:02d}"


print(AudioFile("raga.mp3", 4_500_000, 245).summary())

raga.mp3 (4,500,000 bytes), 4:05


**Polymorphism**: the loop calls `summary()` without knowing the type. `VideoLink`
shares no base class with the others; having the method is enough (duck typing,
2.3.4).

In [52]:
class VideoLink:
    """Not a MediaFile at all, but it has summary()."""

    def __init__(self, url: str) -> None:
        self.url = url

    def summary(self) -> str:
        """One-line description."""
        return f"link to {self.url}"


library = [MediaFile("notes.txt", 2_048), AudioFile("raga.mp3", 4_500_000, 245)]
library.append(VideoLink("https://example.com/talk"))
for item in library:
    print(item.summary())

notes.txt (2,048 bytes)
raga.mp3 (4,500,000 bytes), 4:05
link to https://example.com/talk


**Abstraction**: an abstract base class (Unit 2.3) states what every playable item
must provide, and refuses to create incomplete objects.

In [53]:
class Playable(ABC):
    """Anything that can be played."""

    @abstractmethod
    def play(self) -> str:
        """Start playback and describe it."""


class Podcast(Playable):
    """A concrete playable item."""

    def play(self) -> str:
        return "playing podcast"


print(Podcast().play())
print(raised_message(Playable))

playing podcast
TypeError: Can't instantiate abstract class Playable without an implementation for abstract method 'play'


**How Python differs from Java/TypeScript in practice**

- **No access modifiers.** Encapsulation is a convention plus properties, not a
  compiler rule (2.1.7).
- **Multiple inheritance is allowed**, made safe by the C3 MRO and cooperative
  `super()` (2.2.2–2.2.4). Java and TypeScript allow one base class.
- **No method overloading.** A class namespace is a dict, so a second `def` with
  the same name **replaces** the first. Python uses default arguments,
  `*args`/`**kwargs` or `functools.singledispatchmethod` instead.
- **Interfaces are optional.** Duck typing works with no declaration at all;
  Protocols (2.3.3) add TypeScript-style structural checks for type checkers.

The cell shows the "no overloading" rule: only the second `add` exists.

In [54]:
class Calculator:
    """Two methods with the same name: the second replaces the first."""

    def add(self, first: int, second: int) -> int:
        return first + second

    def add(self, first: int, second: int, third: int) -> int:  # replaces the first
        return first + second + third


print(f"{Calculator().add(1, 2, 3)=}")
print(raised_message(lambda: Calculator().add(1, 2)))  # type: ignore[call-arg]

Calculator().add(1, 2, 3)=6
TypeError: Calculator.add() missing 1 required positional argument: 'third'


### Lab 2.1

Four exercises. For each, replace every `raise NotImplementedError` in the stub,
then run the check cell. Until then the checks print FAIL; that is expected. Two
structural checks ("is a classmethod" in exercise 3, "line_total is a property" in
exercise 4) already pass on the stub, because the stub has the right shape.

#### Exercise 1 — `BankAccount` with a validated balance and history

`BankAccount(owner, opening_balance=Decimal("0"))`:

- `balance` is a **read-only** property (assigning to it raises `AttributeError`).
- `deposit(amount)` and `withdraw(amount)` return the new balance. The amount must be
  a `Decimal` (otherwise `TypeError(f"amount must be a Decimal, got {type_name}")`)
  and greater than zero (otherwise `ValueError("amount must be positive")`).
- A withdrawal larger than the balance raises
  `ValueError(f"insufficient funds: balance {balance:.2f}, requested {amount:.2f}")`.
- `history` is a property returning a **tuple** of `Transaction` records, oldest
  first. The first one is `Transaction("open", opening_balance, opening_balance)`;
  then `"deposit"` / `"withdrawal"` with the balance after each.
- A failed operation changes nothing. A negative opening balance raises
  `ValueError("opening balance cannot be negative")`; a non-`Decimal` one raises
  `TypeError(f"opening_balance must be a Decimal, got {type_name}")`.

In [55]:
class Transaction(NamedTuple):
    """One entry in a BankAccount history."""

    kind: str  # "open", "deposit" or "withdrawal"
    amount: Decimal
    balance_after: Decimal


class BankAccount:
    """A bank account with a read-only, validated balance and an append-only history."""

    def __init__(self, owner: str, opening_balance: Decimal = Decimal("0")) -> None:
        raise NotImplementedError("Lab 2.1: implement BankAccount.__init__")

    @property
    def balance(self) -> Decimal:
        """Current balance (read-only)."""
        raise NotImplementedError("Lab 2.1: implement BankAccount.balance")

    @property
    def history(self) -> tuple[Transaction, ...]:
        """Every transaction so far, oldest first."""
        raise NotImplementedError("Lab 2.1: implement BankAccount.history")

    def deposit(self, amount: Decimal) -> Decimal:
        """Add amount and return the new balance."""
        raise NotImplementedError("Lab 2.1: implement BankAccount.deposit")

    def withdraw(self, amount: Decimal) -> Decimal:
        """Take amount and return the new balance."""
        raise NotImplementedError("Lab 2.1: implement BankAccount.withdraw")

Run the checks.

In [56]:
def sample_account() -> BankAccount:
    """An account opened with 100.00, then +50.00 and -30.00."""
    account = BankAccount("Asha", Decimal("100.00"))
    account.deposit(Decimal("50.00"))
    account.withdraw(Decimal("30.00"))
    return account


def balance_after_failed_withdrawal() -> tuple[Decimal, int]:
    """Balance and history length after an overdraft attempt."""
    account = sample_account()
    raised_error_name(lambda: account.withdraw(Decimal("500.00")))
    return account.balance, len(account.history)


def assign_balance() -> None:
    """Try to overwrite the read-only balance."""
    account = sample_account()
    account.balance = Decimal("1000000")  # type: ignore[misc]


def history_is_a_copy() -> bool:
    """True if the returned history is a tuple and the account still has 3 entries."""
    account = sample_account()
    history = account.history
    return type(history) is tuple and len(account.history) == 3


run_checks(
    [
        ("balance", lambda: sample_account().balance, Decimal("120.00")),
        (
            "history",
            lambda: sample_account().history,
            (
                Transaction("open", Decimal("100.00"), Decimal("100.00")),
                Transaction("deposit", Decimal("50.00"), Decimal("150.00")),
                Transaction("withdrawal", Decimal("30.00"), Decimal("120.00")),
            ),
        ),
        ("history is a tuple", history_is_a_copy, True),
        ("balance is read-only", lambda: raised_error_name(assign_balance), "AttributeError"),
        ("default opening", lambda: BankAccount("Ravi").balance, Decimal("0")),
        (
            "returns new balance",
            lambda: BankAccount("Ravi").deposit(Decimal("10.50")),
            Decimal("10.50"),
        ),
        (
            "non-positive amount",
            lambda: raised_message(lambda: BankAccount("Ravi").deposit(Decimal("0"))),
            "ValueError: amount must be positive",
        ),
        (
            "float amount",
            lambda: raised_message(lambda: BankAccount("Ravi").deposit(10.5)),  # type: ignore[arg-type]
            "TypeError: amount must be a Decimal, got float",
        ),
        (
            "overdraft",
            lambda: raised_message(lambda: sample_account().withdraw(Decimal("500"))),
            "ValueError: insufficient funds: balance 120.00, requested 500.00",
        ),
        (
            "failed withdrawal changes nothing",
            balance_after_failed_withdrawal,
            (Decimal("120.00"), 3),
        ),
        (
            "negative opening",
            lambda: raised_message(lambda: BankAccount("Ravi", Decimal("-1"))),
            "ValueError: opening balance cannot be negative",
        ),
    ]
)

FAIL  balance: raised NotImplementedError: Lab 2.1: implement BankAccount.__init__
FAIL  history: raised NotImplementedError: Lab 2.1: implement BankAccount.__init__
FAIL  history is a tuple: raised NotImplementedError: Lab 2.1: implement BankAccount.__init__
FAIL  balance is read-only: expected 'AttributeError', got 'NotImplementedError'
FAIL  default opening: raised NotImplementedError: Lab 2.1: implement BankAccount.__init__
FAIL  returns new balance: raised NotImplementedError: Lab 2.1: implement BankAccount.__init__
FAIL  non-positive amount: expected 'ValueError: amount must be positive', got 'NotImplementedError: Lab 2.1: implement BankAccount.__init__'
FAIL  float amount: expected 'TypeError: amount must be a Decimal, got float', got 'NotImplementedError: Lab 2.1: implement BankAccount.__init__'
FAIL  overdraft: expected 'ValueError: insufficient funds: balance 120.00, requested 500.00', got 'NotImplementedError: Lab 2.1: implement BankAccount.__init__'
FAIL  failed withdrawal 

#### Exercise 2 — `Temperature` with a `fahrenheit` property

`Temperature(celsius)` stores degrees Celsius:

- `celsius` is a property whose setter rejects values below absolute zero
  (−273.15) with `ValueError(f"below absolute zero: {value} °C")`. `__init__` must
  go through the setter.
- `fahrenheit` is a property computed from Celsius (`c * 9 / 5 + 32`); its
  **setter** converts back and stores Celsius, so it is validated too.
- `kelvin` is a **read-only** property (`c + 273.15`).
- A rejected assignment leaves the old value.

In [57]:
ABSOLUTE_ZERO_CELSIUS = -273.15


class Temperature:
    """A temperature stored in Celsius, readable and writable in Fahrenheit."""

    def __init__(self, celsius: float) -> None:
        raise NotImplementedError("Lab 2.1: implement Temperature")

Run the checks. Floats are rounded to two places before comparing.

In [58]:
def set_fahrenheit(value: float) -> float:
    """Set Fahrenheit on a 20 °C temperature and return the Celsius value."""
    temperature = Temperature(20)
    temperature.fahrenheit = value
    return round(temperature.celsius, 2)


def rejected_fahrenheit_keeps_value() -> float:
    """Celsius after an invalid Fahrenheit assignment."""
    temperature = Temperature(20)
    raised_error_name(lambda: setattr(temperature, "fahrenheit", -500))
    return temperature.celsius


run_checks(
    [
        ("100 °C in °F", lambda: round(Temperature(100).fahrenheit, 2), 212.0),
        ("-40 is -40", lambda: round(Temperature(-40).fahrenheit, 2), -40.0),
        ("set °F", lambda: set_fahrenheit(212), 100.0),
        ("set °F to 32", lambda: set_fahrenheit(32), 0.0),
        ("kelvin", lambda: round(Temperature(25).kelvin, 2), 298.15),
        (
            "kelvin is read-only",
            lambda: raised_error_name(lambda: setattr(Temperature(25), "kelvin", 0)),
            "AttributeError",
        ),
        (
            "below absolute zero",
            lambda: raised_message(lambda: Temperature(-300)),
            "ValueError: below absolute zero: -300 °C",
        ),
        (
            "°F setter validates",
            lambda: raised_error_name(lambda: setattr(Temperature(0), "fahrenheit", -500)),
            "ValueError",
        ),
        ("rejected value leaves state", rejected_fahrenheit_keeps_value, 20),
        (
            "fahrenheit is a property",
            lambda: type(Temperature.__dict__["fahrenheit"]).__name__,
            "property",
        ),
    ]
)

FAIL  100 °C in °F: raised NotImplementedError: Lab 2.1: implement Temperature
FAIL  -40 is -40: raised NotImplementedError: Lab 2.1: implement Temperature
FAIL  set °F: raised NotImplementedError: Lab 2.1: implement Temperature
FAIL  set °F to 32: raised NotImplementedError: Lab 2.1: implement Temperature
FAIL  kelvin: raised NotImplementedError: Lab 2.1: implement Temperature
FAIL  kelvin is read-only: expected 'AttributeError', got 'NotImplementedError'
FAIL  below absolute zero: expected 'ValueError: below absolute zero: -300 °C', got 'NotImplementedError: Lab 2.1: implement Temperature'
FAIL  °F setter validates: expected 'ValueError', got 'NotImplementedError'
FAIL  rejected value leaves state: raised NotImplementedError: Lab 2.1: implement Temperature
FAIL  fahrenheit is a property: raised KeyError: 'fahrenheit'
0/10 checks passed


#### Exercise 3 — subclass-aware `User.from_dict`

Write the `classmethod` `User.from_dict(data)`:

- `username` and `email` are required; check them in that order and raise
  `ValueError(f"missing field: {name}")` for the first one missing.
- Strip the username; strip and lower-case the email. `is_active` is optional
  (default `True`). Ignore unknown keys and do not modify `data`.
- It must build an instance of **the class it is called on**, so
  `AdminUser.from_dict(...)` returns an `AdminUser`, and so does any subclass
  written later (the check defines a new `GuestUser`).

`__init__`, `to_dict` and `AdminUser` are given.

In [59]:
class User:
    """A site user."""

    role = "member"

    def __init__(self, username: str, email: str, is_active: bool = True) -> None:
        self.username = username
        self.email = email
        self.is_active = is_active

    @classmethod
    def from_dict(cls, data: Mapping[str, Any]) -> Self:
        """Build a user from a mapping (username and email required)."""
        raise NotImplementedError("Lab 2.1: implement User.from_dict")

    def to_dict(self) -> dict[str, Any]:
        """Serialize to a plain dict, including the class-level role."""
        return {
            "username": self.username,
            "email": self.email,
            "is_active": self.is_active,
            "role": self.role,
        }


class AdminUser(User):
    """A user with the admin role."""

    role = "admin"

Run the checks.

In [60]:
class GuestUser(User):
    """A third subclass the stub has never seen."""

    role = "guest"


raw_user = {"username": "  meena ", "email": " Meena@Example.COM ", "plan": "pro"}
run_checks(
    [
        ("returns User", lambda: type(User.from_dict(raw_user)).__name__, "User"),
        ("subclass-aware", lambda: type(AdminUser.from_dict(raw_user)).__name__, "AdminUser"),
        ("unseen subclass", lambda: type(GuestUser.from_dict(raw_user)).__name__, "GuestUser"),
        (
            "fields cleaned",
            lambda: User.from_dict(raw_user).to_dict(),
            {
                "username": "meena",
                "email": "meena@example.com",
                "is_active": True,
                "role": "member",
            },
        ),
        ("admin role", lambda: AdminUser.from_dict(raw_user).to_dict()["role"], "admin"),
        (
            "is_active read",
            lambda: User.from_dict({**raw_user, "is_active": False}).is_active,
            False,
        ),
        (
            "round trip",
            lambda: AdminUser.from_dict(AdminUser.from_dict(raw_user).to_dict()).to_dict(),
            {
                "username": "meena",
                "email": "meena@example.com",
                "is_active": True,
                "role": "admin",
            },
        ),
        (
            "missing email",
            lambda: raised_message(lambda: User.from_dict({"username": "x"})),
            "ValueError: missing field: email",
        ),
        (
            "missing both",
            lambda: raised_message(lambda: User.from_dict({})),
            "ValueError: missing field: username",
        ),
        (
            "input untouched",
            lambda: (User.from_dict(raw_user), raw_user["username"])[1],
            "  meena ",
        ),
        ("is a classmethod", lambda: type(User.__dict__["from_dict"]).__name__, "classmethod"),
    ]
)

FAIL  returns User: raised NotImplementedError: Lab 2.1: implement User.from_dict
FAIL  subclass-aware: raised NotImplementedError: Lab 2.1: implement User.from_dict
FAIL  unseen subclass: raised NotImplementedError: Lab 2.1: implement User.from_dict
FAIL  fields cleaned: raised NotImplementedError: Lab 2.1: implement User.from_dict
FAIL  admin role: raised NotImplementedError: Lab 2.1: implement User.from_dict
FAIL  is_active read: raised NotImplementedError: Lab 2.1: implement User.from_dict
FAIL  round trip: raised NotImplementedError: Lab 2.1: implement User.from_dict
FAIL  missing email: expected 'ValueError: missing field: email', got 'NotImplementedError: Lab 2.1: implement User.from_dict'
FAIL  missing both: expected 'ValueError: missing field: username', got 'NotImplementedError: Lab 2.1: implement User.from_dict'
FAIL  input untouched: raised NotImplementedError: Lab 2.1: implement User.from_dict
PASS  is a classmethod
1/11 checks passed


#### Exercise 4 — `ShoppingCart` and `CartItem` with `Decimal` totals

`CartItem(sku, unit_price, quantity=1)`:

- `unit_price` must be a `Decimal`
  (`TypeError(f"unit_price must be a Decimal, got {type_name}")`) and not negative
  (`ValueError("unit price cannot be negative")`); `quantity` must be at least 1
  (`ValueError("quantity must be at least 1")`).
- `line_total` is a property: `unit_price * quantity`.

`ShoppingCart()`:

- `add(sku, unit_price, quantity=1)` returns the `CartItem`. Adding an existing SKU
  increases its quantity; a different unit price raises
  `ValueError(f"price mismatch for {sku}")`.
- `remove(sku, quantity=None)` removes that many units (all of them when `None`, or
  when the quantity reaches zero); an unknown SKU raises `KeyError(sku)`.
- Properties: `items` (a tuple of `CartItem`s in first-added order), `item_count`
  (total units) and `subtotal` (a `Decimal`, `Decimal("0")` when empty).
- `total(tax_rate_percent=Decimal("18"))` returns subtotal plus tax, rounded
  `ROUND_HALF_UP` to the paisa (`PAISE`, from the setup cell).

In [61]:
class CartItem:
    """One line of a shopping cart."""

    def __init__(self, sku: str, unit_price: Decimal, quantity: int = 1) -> None:
        raise NotImplementedError("Lab 2.1: implement CartItem.__init__")

    @property
    def line_total(self) -> Decimal:
        """unit_price × quantity."""
        raise NotImplementedError("Lab 2.1: implement CartItem.line_total")


class ShoppingCart:
    """A cart of CartItems keyed by SKU, with exact Decimal totals."""

    def __init__(self) -> None:
        raise NotImplementedError("Lab 2.1: implement ShoppingCart.__init__")

    def add(self, sku: str, unit_price: Decimal, quantity: int = 1) -> CartItem:
        """Add quantity of sku."""
        raise NotImplementedError("Lab 2.1: implement ShoppingCart.add")

    def remove(self, sku: str, quantity: int | None = None) -> None:
        """Remove quantity of sku (all of it when quantity is None)."""
        raise NotImplementedError("Lab 2.1: implement ShoppingCart.remove")

    @property
    def items(self) -> tuple[CartItem, ...]:
        """The cart lines in first-added order."""
        raise NotImplementedError("Lab 2.1: implement ShoppingCart.items")

    @property
    def item_count(self) -> int:
        """Total quantity across every line."""
        raise NotImplementedError("Lab 2.1: implement ShoppingCart.item_count")

    @property
    def subtotal(self) -> Decimal:
        """Sum of line totals before tax."""
        raise NotImplementedError("Lab 2.1: implement ShoppingCart.subtotal")

    def total(self, tax_rate_percent: Decimal = Decimal("18")) -> Decimal:
        """Subtotal plus tax, rounded half-up to the paisa."""
        raise NotImplementedError("Lab 2.1: implement ShoppingCart.total")

Run the checks.

In [62]:
def sample_cart() -> ShoppingCart:
    """Two lines: 2 × 499.00 soap (added in two calls) and 1 × 1250.50 rice."""
    cart = ShoppingCart()
    cart.add("SOAP", Decimal("499.00"))
    cart.add("RICE-5KG", Decimal("1250.50"))
    cart.add("SOAP", Decimal("499.00"))
    return cart


def lines(cart: ShoppingCart) -> list[tuple[str, int, Decimal]]:
    """(sku, quantity, line_total) for each cart line."""
    return [(item.sku, item.quantity, item.line_total) for item in cart.items]


def after_remove(sku: str, quantity: int | None) -> list[tuple[str, int, Decimal]]:
    """Lines of the sample cart after one remove call."""
    cart = sample_cart()
    cart.remove(sku, quantity)
    return lines(cart)


run_checks(
    [
        (
            "lines merge",
            lambda: lines(sample_cart()),
            [("SOAP", 2, Decimal("998.00")), ("RICE-5KG", 1, Decimal("1250.50"))],
        ),
        ("item_count", lambda: sample_cart().item_count, 3),
        ("subtotal", lambda: sample_cart().subtotal, Decimal("2248.50")),
        ("total with 18% GST", lambda: sample_cart().total(), Decimal("2653.23")),
        ("total with 5% GST", lambda: sample_cart().total(Decimal("5")), Decimal("2360.93")),
        (
            "empty cart",
            lambda: (ShoppingCart().subtotal, ShoppingCart().total()),
            (Decimal("0"), Decimal("0.00")),
        ),
        ("items is a tuple", lambda: type(sample_cart().items).__name__, "tuple"),
        (
            "price mismatch",
            lambda: raised_message(lambda: sample_cart().add("SOAP", Decimal("450.00"))),
            "ValueError: price mismatch for SOAP",
        ),
        (
            "remove some",
            lambda: after_remove("SOAP", 1),
            [("SOAP", 1, Decimal("499.00")), ("RICE-5KG", 1, Decimal("1250.50"))],
        ),
        ("remove all", lambda: after_remove("SOAP", None), [("RICE-5KG", 1, Decimal("1250.50"))]),
        (
            "remove unknown",
            lambda: raised_error_name(lambda: sample_cart().remove("TEA")),
            "KeyError",
        ),
        (
            "negative price",
            lambda: raised_message(lambda: CartItem("X", Decimal("-1"))),
            "ValueError: unit price cannot be negative",
        ),
        (
            "zero quantity",
            lambda: raised_message(lambda: CartItem("X", Decimal("1"), 0)),
            "ValueError: quantity must be at least 1",
        ),
        (
            "float price",
            lambda: raised_message(lambda: CartItem("X", 9.99)),  # type: ignore[arg-type]
            "TypeError: unit_price must be a Decimal, got float",
        ),
        (
            "line_total is a property",
            lambda: type(CartItem.__dict__["line_total"]).__name__,
            "property",
        ),
    ]
)

FAIL  lines merge: raised NotImplementedError: Lab 2.1: implement ShoppingCart.__init__
FAIL  item_count: raised NotImplementedError: Lab 2.1: implement ShoppingCart.__init__
FAIL  subtotal: raised NotImplementedError: Lab 2.1: implement ShoppingCart.__init__
FAIL  total with 18% GST: raised NotImplementedError: Lab 2.1: implement ShoppingCart.__init__
FAIL  total with 5% GST: raised NotImplementedError: Lab 2.1: implement ShoppingCart.__init__
FAIL  empty cart: raised NotImplementedError: Lab 2.1: implement ShoppingCart.__init__
FAIL  items is a tuple: raised NotImplementedError: Lab 2.1: implement ShoppingCart.__init__
FAIL  price mismatch: expected 'ValueError: price mismatch for SOAP', got 'NotImplementedError: Lab 2.1: implement ShoppingCart.__init__'
FAIL  remove some: raised NotImplementedError: Lab 2.1: implement ShoppingCart.__init__
FAIL  remove all: raised NotImplementedError: Lab 2.1: implement ShoppingCart.__init__
FAIL  remove unknown: expected 'KeyError', got 'NotImpleme

### Recap — Unit 2.1

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| Why classes | Bundle state with the rules that keep it valid; prefer functions, modules or dataclasses when there are no rules | `class` vs plain objects and functions |
| `class` statement | Runs the body in a namespace, then calls `type(name, bases, namespace)` | class expression / `Object.create` |
| `__dict__` | A class's namespace is a read-only `mappingproxy`; change it with `setattr` | `Class.prototype` |
| Creation | `__new__` allocates, `__init__` initializes; `__new__` matters for immutable bases | `constructor` (both at once) |
| `self` | Explicit first parameter; `obj.m(x)` is `Class.m(obj, x)` | implicit `this` |
| Attribute lookup | instance `__dict__` → class → bases; assignment through an instance shadows | prototype chain |
| Class-attribute trap | A mutable class attribute is shared by every instance; create it in `__init__` | class fields are per instance |
| Bound methods | `__self__` + `__func__`; usable as callbacks without binding | `.bind(this)` |
| `@classmethod` | Receives `cls`; use for subclass-aware alternative constructors | `static` + `this` |
| `@staticmethod` | Receives nothing; a helper that lives with the class | `static` |
| `@property` | Computed, read-only and validated attributes behind attribute syntax | `get` / `set` accessors |
| Privacy | `_x` by convention, `__x` mangled to `_Class__x`; nothing is truly private | `#x` (enforced), TS `private` (compile time) |
| Four pillars | All four exist, with lighter mechanisms: conventions, multiple inheritance, duck typing, ABCs/Protocols | Java/TS keywords |

**Further reading**

- [Tutorial: classes](https://docs.python.org/3/tutorial/classes.html),
  [data model: classes and instances](https://docs.python.org/3/reference/datamodel.html#classes),
  [class definitions](https://docs.python.org/3/reference/compound_stmts.html#class-definitions)
- [`type`](https://docs.python.org/3/library/functions.html#type),
  [`vars`](https://docs.python.org/3/library/functions.html#vars),
  [`property`](https://docs.python.org/3/library/functions.html#property),
  [`classmethod`](https://docs.python.org/3/library/functions.html#classmethod),
  [`staticmethod`](https://docs.python.org/3/library/functions.html#staticmethod)
- [Descriptor HowTo Guide](https://docs.python.org/3/howto/descriptor.html),
  [private name mangling](https://docs.python.org/3/reference/expressions.html#private-name-mangling),
  [`functools.cached_property`](https://docs.python.org/3/library/functools.html#functools.cached_property)
- [PEP 8 — naming conventions](https://peps.python.org/pep-0008/#naming-conventions),
  [PEP 3115 — Metaclasses in Python 3](https://peps.python.org/pep-3115/),
  [PEP 526 — `ClassVar`](https://peps.python.org/pep-0526/),
  [PEP 673 — `Self`](https://peps.python.org/pep-0673/),
  [PEP 557 — Data classes](https://peps.python.org/pep-0557/)

## Unit 2.2 — Inheritance, MRO and Composition

**Objectives**

- Override and extend methods, and call the base class with `super().__init__`.
- Explain zero-argument `super()` as "the next class in the MRO", not "the parent".
- Compute a C3 linearization (MRO) by hand and recognize an inconsistent one.
- Write mixins and order base classes correctly.
- Prefer composition when inheritance would explode into many classes or break the
  Liskov substitution principle.
- Recognize each SOLID principle in a short Python example.

**JS bridge**

| JavaScript / TypeScript | Python | The difference that bites |
| --- | --- | --- |
| `class B extends A` | `class B(A):` | Python allows **several** base classes |
| `super(...)` must run before `this` is used | `super().__init__(...)` is optional and can run anywhere | Forget it and base attributes are silently missing |
| `super.method()` (always the static parent, via `[[HomeObject]]`) | `super().method()` (the next class in `type(self).__mro__`) | Under multiple inheritance `super()` can call a **sibling** |
| Prototype chain | `__mro__` (C3 linearization) | Computed once per class; base order matters |
| Mixins via `Object.assign` or `Base => class extends Base {}` | Mixin classes listed before the base class | Order of bases decides which method wins |
| `obj instanceof A` | `isinstance(obj, A)` / `issubclass(B, A)` | Accept tuples and `A | B` unions |
| `Object.prototype` | `object` | Every class inherits from `object` |
| TS `override` keyword | `@typing.override` (3.12+) | Checked by type checkers, not at run time |

The next cell is this unit's setup. `mro_names(cls)` lists class names along
`cls.__mro__`.

In [63]:
import importlib.util
import inspect
import itertools
import json
import socketserver
import sys
from collections.abc import Callable, Iterable, Iterator
from dataclasses import dataclass
from datetime import date, datetime
from decimal import Decimal
from typing import Any, ClassVar, Protocol, override


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


def raised_message(thunk: Callable[[], object]) -> str:
    """Call thunk and return "ErrorName: message" for the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return f"{type(error).__name__}: {error}"
    return "no error"


def mro_names(cls: type) -> list[str]:
    """Class names along cls.__mro__."""
    return [klass.__name__ for klass in cls.__mro__]


print(f"{sys.version_info[:2]=}")

sys.version_info[:2]=(3, 15)


### 2.2.1 Single inheritance

A subclass inherits every attribute of its base class and can **override** (replace)
or **extend** (replace and call the original) any of them. The base class for this
section is a parcel shipment.

In [64]:
class Shipment:
    """A parcel to deliver."""

    RATE_PER_KG = Decimal("40")

    def __init__(self, tracking_id: str, weight_kg: Decimal) -> None:
        self.tracking_id = tracking_id
        self.weight_kg = weight_kg

    def shipping_cost(self) -> Decimal:
        """Price of delivery in rupees."""
        return self.weight_kg * self.RATE_PER_KG

    def label(self) -> str:
        """Text printed on the parcel."""
        return f"{self.tracking_id}: {self.weight_kg} kg, INR {self.shipping_cost():.2f}"


print(Shipment("TRK-1", Decimal("2.5")).label())

TRK-1: 2.5 kg, INR 100.00


**Overriding.** `FragileShipment` replaces `shipping_cost` and reuses the original
through `super()`. It does not touch `label`, yet the inherited `label` prints the
new cost: `self.shipping_cost()` is looked up on the **instance's** class at call
time (dynamic dispatch).

`@override` (from `typing`, 3.12+) marks the intent. At run time it only sets
`__override__ = True`; type checkers report an error if the base class has no
method with that name, which catches typos like `shiping_cost`.

In [65]:
class FragileShipment(Shipment):
    """Costs more: a fixed handling fee on top of the weight-based price."""

    HANDLING_FEE = Decimal("75")

    @override
    def shipping_cost(self) -> Decimal:
        return super().shipping_cost() + self.HANDLING_FEE


vase = FragileShipment("TRK-2", Decimal("2.5"))
print(vase.label())
print(f"{FragileShipment.shipping_cost.__override__=}")
print(f"{'label' in vars(FragileShipment)=}  (label is inherited, not copied)")

TRK-2: 2.5 kg, INR 175.00
FragileShipment.shipping_cost.__override__=True
'label' in vars(FragileShipment)=False  (label is inherited, not copied)


**Extending `__init__`.** A subclass that needs extra state defines its own
`__init__`, calls `super().__init__(...)` for the base part, then sets its own
attributes. It can extend other methods the same way.

In [66]:
class ExpressShipment(Shipment):
    """A shipment with a promised delivery date."""

    def __init__(self, tracking_id: str, weight_kg: Decimal, promised_by: date) -> None:
        super().__init__(tracking_id, weight_kg)
        self.promised_by = promised_by

    @override
    def label(self) -> str:
        return f"{super().label()} | EXPRESS by {self.promised_by:%d %b}"


express = ExpressShipment("TRK-3", Decimal("1"), date(2026, 11, 9))
print(express.label())
print(f"{vars(express)=}")

TRK-3: 1 kg, INR 40.00 | EXPRESS by 09 Nov
vars(express)={'tracking_id': 'TRK-3', 'weight_kg': Decimal('1'), 'promised_by': datetime.date(2026, 11, 9)}


Forgetting `super().__init__` is not an error in Python (unlike JavaScript, where
using `this` before `super()` throws). The object is created without the base
attributes, and the failure shows up later, far from the cause.

In [67]:
class BrokenExpressShipment(Shipment):
    """BUG: never calls super().__init__."""

    def __init__(self, tracking_id: str, weight_kg: Decimal, promised_by: date) -> None:
        self.promised_by = promised_by


broken = BrokenExpressShipment("TRK-4", Decimal("1"), date(2026, 11, 9))
print(f"{vars(broken)=}")
print(raised_message(broken.label))

vars(broken)={'promised_by': datetime.date(2026, 11, 9)}
AttributeError: 'BrokenExpressShipment' object has no attribute 'tracking_id'


**`isinstance` and `issubclass`** follow the inheritance chain. Both accept a tuple
of classes or a `X | Y` union (3.10+). Note that `bool` is a subclass of `int`, a
classic surprise in validation code.

In [68]:
print(f"{isinstance(vase, FragileShipment)=}, {isinstance(vase, Shipment)=}")
print(f"{isinstance(vase, ExpressShipment)=}")
print(f"{issubclass(ExpressShipment, Shipment)=}, {issubclass(Shipment, ExpressShipment)=}")
print(f"{isinstance(vase, (ExpressShipment, FragileShipment))=}")
print(f"{isinstance(42, int | str)=}, {issubclass(bool, int)=}, {isinstance(True, int)=}")

isinstance(vase, FragileShipment)=True, isinstance(vase, Shipment)=True
isinstance(vase, ExpressShipment)=False
issubclass(ExpressShipment, Shipment)=True, issubclass(Shipment, ExpressShipment)=False
isinstance(vase, (ExpressShipment, FragileShipment))=True
isinstance(42, int | str)=True, issubclass(bool, int)=True, isinstance(True, int)=True


**Everything inherits from `object`.** A class with no bases gets `object` as its
only base, which is where default `__repr__`, `__eq__`, `__hash__`, `__init__` and
friends come from. `__subclasses__()` lists direct subclasses (it holds weak
references, so it reflects classes that still exist).

In [69]:
print(f"{Shipment.__bases__=}")
print(f"{mro_names(FragileShipment)=}")
print(f"{sorted(klass.__name__ for klass in Shipment.__subclasses__())=}")
print(f"{len(dir(object))=} attributes on object, e.g. {sorted(dir(object))[:5]}")
print(f"{issubclass(type, object)=}, {isinstance(object, type)=}")

Shipment.__bases__=(<class 'object'>,)
mro_names(FragileShipment)=['FragileShipment', 'Shipment', 'object']
sorted(klass.__name__ for klass in Shipment.__subclasses__())=['BrokenExpressShipment', 'ExpressShipment', 'FragileShipment']
len(dir(object))=24 attributes on object, e.g. ['__class__', '__delattr__', '__dir__', '__doc__', '__eq__']
issubclass(type, object)=True, isinstance(object, type)=True


### 2.2.2 `super()` in depth

**Zero-argument `super()`** is shorthand for `super(CurrentClass, self)`. The
compiler sees `super` (or `__class__`) inside a method and gives the method a hidden
closure cell named `__class__`, holding the class being defined (closures and cells:
1.11). At call time `super()` reads that cell and the method's first argument.

In [70]:
init_function = ExpressShipment.__init__
print(f"{init_function.__code__.co_freevars=}")
print(f"{init_function.__closure__[0].cell_contents=}")
print(f"{super(ExpressShipment, express).label()=}  (the explicit two-argument form)")

init_function.__code__.co_freevars=('__class__',)
init_function.__closure__[0].cell_contents=<class '__main__.ExpressShipment'>
super(ExpressShipment, express).label()='TRK-3: 1 kg, INR 40.00'  (the explicit two-argument form)


Because the cell is created by the class statement, `super()` in a function defined
**outside** a class fails, even if you attach the function to a class later.

In [71]:
def orphan_label(self: Shipment) -> str:
    """Uses super() but was not defined inside a class body."""
    return super().label()


Shipment.orphan_label = orphan_label
print(raised_message(lambda: Shipment("TRK-5", Decimal("1")).orphan_label()))
del Shipment.orphan_label

RuntimeError: super(): __class__ cell not found


**"Next in MRO", not "parent".** `super()` means: in the MRO of `type(self)`, find
the class *after* the one where this method is defined, and look the name up from
there. With single inheritance that is the parent. With multiple inheritance it can
be a **sibling**.

Below, `Left.setup` calls `super().setup(...)`. On a plain `Left` instance that
reaches `Base`. On a `Child` instance, whose MRO is `Child, Left, Right, Base`, the
very same line reaches **`Right`**, a class `Left` knows nothing about.

In [72]:
class Base:
    def setup(self, trace: list[str]) -> None:
        trace.append("Base")


class Left(Base):
    def setup(self, trace: list[str]) -> None:
        trace.append("Left")
        super().setup(trace)


class Right(Base):
    def setup(self, trace: list[str]) -> None:
        trace.append("Right")
        super().setup(trace)


class Child(Left, Right):
    def setup(self, trace: list[str]) -> None:
        trace.append("Child")
        super().setup(trace)


for instance in [Left(), Child()]:
    setup_trace: list[str] = []
    instance.setup(setup_trace)
    print(f"{type(instance).__name__:<5} MRO {mro_names(type(instance))}  calls {setup_trace}")
print(f"{super(Left, Child()).setup.__qualname__=}")

Left  MRO ['Left', 'Base', 'object']  calls ['Left', 'Base']
Child MRO ['Child', 'Left', 'Right', 'Base', 'object']  calls ['Child', 'Left', 'Right', 'Base']
super(Left, Child()).setup.__qualname__='Right.setup'


**Cooperative `**kwargs`.** When several classes in one hierarchy each need
constructor arguments, the pattern that works with any MRO is:

1. Every `__init__` takes its own arguments as **keyword-only** and the rest as
   `**kwargs`.
2. It removes what it needs and passes the rest on with `super().__init__(**kwargs)`.
3. Every class calls `super()`, so the chain reaches `object.__init__`, which must
   receive **nothing**: a leftover keyword is a bug and raises `TypeError`.

Notice that `CatalogItem` defines no `__init__` at all, yet both base classes run.

In [73]:
class Model:
    """Root of the cooperative chain."""

    def __init__(self, **kwargs: Any) -> None:
        super().__init__(**kwargs)  # object.__init__ must receive nothing


class Named(Model):
    def __init__(self, *, name: str, **kwargs: Any) -> None:
        self.name = name
        super().__init__(**kwargs)


class Priced(Model):
    def __init__(self, *, price: Decimal, **kwargs: Any) -> None:
        self.price = price
        super().__init__(**kwargs)


class CatalogItem(Named, Priced):
    """Gets both constructors through super(), in MRO order."""


tea = CatalogItem(name="Tea", price=Decimal("120"))
print(f"{vars(tea)=}, {mro_names(CatalogItem)=}")
print(raised_message(lambda: CatalogItem(name="Tea", price=Decimal("1"), colour="red")))

vars(tea)={'name': 'Tea', 'price': Decimal('120')}, mro_names(CatalogItem)=['CatalogItem', 'Named', 'Priced', 'Model', 'object']
TypeError: object.__init__() takes exactly one argument (the instance to initialize)


### 2.2.3 Multiple inheritance and the MRO

The **method resolution order** (MRO) is the list of classes Python searches for an
attribute. Since Python 2.3 it is computed with the **C3 linearization**, which
guarantees three things: a class comes before its bases, the order of bases in the
`class` line is kept, and every class's MRO is consistent with its bases' MROs.

**The rule.** For `class C(B1, B2, …, Bn)`:

`L[C] = C + merge(L[B1], L[B2], …, L[Bn], [B1, B2, …, Bn])`

`merge` repeatedly takes the **first head** (first element of a list) that does
**not appear in the tail** (any later position) of any list, appends it to the
result and removes it from the front of every list. If no head qualifies, the
hierarchy is inconsistent and Python raises `TypeError`.

**Worked example: a diamond.**

```text
class Vehicle: ...
class LandVehicle(Vehicle): ...
class WaterVehicle(Vehicle): ...
class Amphibious(LandVehicle, WaterVehicle): ...
```

| Step | Lists still to merge | Candidate heads | Taken | Why |
| --- | --- | --- | --- | --- |
| 0 | `[LandVehicle, Vehicle, object]`, `[WaterVehicle, Vehicle, object]`, `[LandVehicle, WaterVehicle]` | — | `Amphibious` | the class itself goes first |
| 1 | (same) | `LandVehicle` | `LandVehicle` | not in any tail |
| 2 | `[Vehicle, object]`, `[WaterVehicle, Vehicle, object]`, `[WaterVehicle]` | `Vehicle` | — | `Vehicle` is in the tail of the second list: skip |
| 3 | (same) | `WaterVehicle` | `WaterVehicle` | not in any tail |
| 4 | `[Vehicle, object]`, `[Vehicle, object]` | `Vehicle` | `Vehicle` | now only ever a head |
| 5 | `[object]`, `[object]` | `object` | `object` | done |

Result: `Amphibious, LandVehicle, WaterVehicle, Vehicle, object`. The shared base
`Vehicle` appears **once**, after both of its subclasses.

In [74]:
class Vehicle:
    def describe(self) -> str:
        return "vehicle"


class LandVehicle(Vehicle):
    def describe(self) -> str:
        return "drives on land"


class WaterVehicle(Vehicle):
    def describe(self) -> str:
        return "floats on water"


class Amphibious(LandVehicle, WaterVehicle):
    """Inherits describe() from the first class in its MRO that defines it."""


print(f"{mro_names(Amphibious)=}")
print(f"{Amphibious.mro() == list(Amphibious.__mro__) == list(inspect.getmro(Amphibious))=}")
print(f"{Amphibious().describe()=}")

mro_names(Amphibious)=['Amphibious', 'LandVehicle', 'WaterVehicle', 'Vehicle', 'object']
Amphibious.mro() == list(Amphibious.__mro__) == list(inspect.getmro(Amphibious))=True
Amphibious().describe()='drives on land'


#### Under the hood: C3 step by step

`c3_linearize` implements the rule above on class **names** and prints every step
for the top-level class. The last line checks it against Python's own `__mro__`.

In [75]:
def c3_linearize(
    class_name: str, bases_of: dict[str, list[str]], verbose: bool = True
) -> list[str]:
    """C3 linearization of class_name, given each class's direct bases by name."""
    if not bases_of[class_name]:
        return [class_name]
    lists = [c3_linearize(base, bases_of, False) for base in bases_of[class_name]]
    lists.append(list(bases_of[class_name]))
    result = [class_name]
    while any(lists):
        for candidate_list in filter(None, lists):
            head = candidate_list[0]
            if not any(head in other[1:] for other in lists):
                break
        else:
            raise TypeError(f"inconsistent hierarchy, stuck at {lists}")
        if verbose:
            print(f"merge {lists} -> take {head}")
        result.append(head)
        lists = [other[1:] if other and other[0] == head else other for other in lists]
    return result


vehicle_bases = {"object": [], "Vehicle": ["object"], "LandVehicle": ["Vehicle"]}
vehicle_bases |= {"WaterVehicle": ["Vehicle"], "Amphibious": ["LandVehicle", "WaterVehicle"]}
print(f"{c3_linearize('Amphibious', vehicle_bases) == mro_names(Amphibious)=}")

merge [['LandVehicle', 'Vehicle', 'object'], ['WaterVehicle', 'Vehicle', 'object'], ['LandVehicle', 'WaterVehicle']] -> take LandVehicle
merge [['Vehicle', 'object'], ['WaterVehicle', 'Vehicle', 'object'], ['WaterVehicle']] -> take WaterVehicle
merge [['Vehicle', 'object'], ['Vehicle', 'object'], []] -> take Vehicle
merge [['object'], ['object'], []] -> take object
c3_linearize('Amphibious', vehicle_bases) == mro_names(Amphibious)=True


**The diamond problem.** In C++ a diamond can give an object two copies of the
shared base. In Python the MRO lists `Vehicle` once, so with cooperative `super()`
its `__init__` runs **once**. Calling base classes by name instead
(`Vehicle.__init__(self)` from both branches) runs it **twice**, which is how
diamonds go wrong in practice.

In [76]:
class Engine:
    def __init__(self, init_log: list[str]) -> None:
        init_log.append("Engine")


class ByNamePetrol(Engine):
    def __init__(self, init_log: list[str]) -> None:
        Engine.__init__(self, init_log)  # explicit call by name


class ByNameElectric(Engine):
    def __init__(self, init_log: list[str]) -> None:
        Engine.__init__(self, init_log)


class ByNameHybrid(ByNamePetrol, ByNameElectric):
    def __init__(self, init_log: list[str]) -> None:
        ByNamePetrol.__init__(self, init_log)
        ByNameElectric.__init__(self, init_log)


by_name_log: list[str] = []
ByNameHybrid(by_name_log)
print(f"{by_name_log=}  (Engine initialized twice)")

by_name_log=['Engine', 'Engine']  (Engine initialized twice)


The same diamond with `super()` everywhere: `Engine` runs exactly once.

In [77]:
class Petrol(Engine):
    def __init__(self, init_log: list[str]) -> None:
        init_log.append("Petrol")
        super().__init__(init_log)


class Electric(Engine):
    def __init__(self, init_log: list[str]) -> None:
        init_log.append("Electric")
        super().__init__(init_log)


class Hybrid(Petrol, Electric):
    def __init__(self, init_log: list[str]) -> None:
        init_log.append("Hybrid")
        super().__init__(init_log)


super_log: list[str] = []
Hybrid(super_log)
print(f"{super_log=}")

super_log=['Hybrid', 'Petrol', 'Electric', 'Engine']


**Inconsistent MROs.** Some base lists cannot be linearized, and Python refuses to
create the class. Two common causes:

1. Listing a base **before its own subclass** (`class Bad(Vehicle, LandVehicle)`):
   `Vehicle` must come first (base order) and after `LandVehicle` (subclass first).
2. Two parents that order the same bases **differently**.

In [78]:
def define_class(name: str, bases: tuple[type, ...]) -> str:
    """Try to create a class; report its MRO or the error."""
    try:
        return f"{mro_names(type(name, bases, {}))}"
    except TypeError as error:
        return f"{type(error).__name__}: {error}"


class FirstOrder(LandVehicle, WaterVehicle): ...


class SecondOrder(WaterVehicle, LandVehicle): ...


print(define_class("Bad", (Vehicle, LandVehicle)))
print(define_class("Conflicted", (FirstOrder, SecondOrder)))
print(define_class("Fine", (LandVehicle, Vehicle)))

TypeError: Cannot create a consistent method resolution order (MRO) for bases Vehicle, LandVehicle
TypeError: Cannot create a consistent method resolution order (MRO) for bases LandVehicle, WaterVehicle
['Fine', 'LandVehicle', 'Vehicle', 'object']


**Predict the output.**

```python
class A:
    def who(self): return "A"
class B(A):
    def who(self): return "B+" + super().who()
class C(A):
    def who(self): return "C+" + super().who()
class D(C, B):
    pass

print(D().who())
print(B().who())
```

<details><summary>Answer</summary>

```text
C+B+A
B+A
```

`D`'s MRO is `D, C, B, A, object` (bases in the order written: `C` before `B`).
`D` has no `who`, so `C.who` runs; its `super()` is the next class after `C` in
**`D`'s** MRO, which is `B`, not `A`. On a plain `B` instance the MRO is
`B, A, object`, so `B`'s `super()` reaches `A`.

</details>

Run the next cell to check your prediction.

In [79]:
class A:
    def who(self) -> str:
        return "A"


class B(A):
    def who(self) -> str:
        return "B+" + super().who()


class C(A):
    def who(self) -> str:
        return "C+" + super().who()


class D(C, B):
    pass


print(D().who())
print(B().who())

C+B+A
B+A


### 2.2.4 Mixins

A **mixin** is a small class that adds one capability and is meant to be combined
with other classes, never used alone. Good mixins:

- do one thing (a `__repr__`, JSON export, timestamps);
- keep little or no state of their own, and if they need `__init__`, take
  `**kwargs` and call `super().__init__(**kwargs)` (2.2.2);
- are named `…Mixin` so readers know they are not standalone.

Three mixins first. `ReprMixin` builds a `__repr__` from `vars(self)`.

In [80]:
class ReprMixin:
    """Adds a __repr__ built from the instance attributes."""

    def __repr__(self) -> str:
        fields = ", ".join(f"{name}={value!r}" for name, value in vars(self).items())
        return f"{type(self).__name__}({fields})"

`JsonSerializableMixin` dumps the instance attributes. This simple version only
handles values `json` already understands; Lab 2.2 exercise 2 extends it to nested
objects and datetimes.

In [81]:
class JsonSerializableMixin:
    """Adds to_json() for objects whose attributes are plain JSON values."""

    def to_json(self) -> str:
        """Serialize vars(self) with json.dumps."""
        return json.dumps(vars(self), sort_keys=True)

`TimestampMixin` needs state, so it has a **cooperative** `__init__`. Its clock is a
class attribute, so tests (and this notebook) can swap in a fake clock and get
deterministic output.

In [82]:
class TimestampMixin:
    """Adds created_at / updated_at, set from a replaceable clock."""

    clock: ClassVar[Callable[[], datetime]] = datetime.now

    def __init__(self, **kwargs: Any) -> None:
        super().__init__(**kwargs)
        self.created_at = self.clock()
        self.updated_at = self.created_at

    def touch(self) -> None:
        """Record a modification."""
        self.updated_at = self.clock()


fake_ticks = iter([datetime(2026, 11, 7, 9, 0), datetime(2026, 11, 7, 9, 45)])


def fake_now() -> datetime:
    """Return the next scripted time."""
    return next(fake_ticks)

**Combining them.** The rule for base order: **mixins on the left, the main base
class on the right**. The MRO then tries each mixin before the base, so a mixin can
add or override behaviour, and the base sits at the end of the cooperative
`__init__` chain.

In [83]:
class Entity:
    """The main base class: every stored object has an id."""

    def __init__(self, *, entity_id: int, **kwargs: Any) -> None:
        super().__init__(**kwargs)
        self.entity_id = entity_id


class Product(ReprMixin, JsonSerializableMixin, TimestampMixin, Entity):
    """A product assembled from three mixins and one base class."""

    clock = staticmethod(fake_now)

    def __init__(self, *, sku: str, price_paise: int, **kwargs: Any) -> None:
        super().__init__(**kwargs)
        self.sku = sku
        self.price_paise = price_paise


soap = Product(entity_id=7, sku="SOAP", price_paise=4_950)
soap.touch()
print(f"{mro_names(Product)=}")
print(soap)

mro_names(Product)=['Product', 'ReprMixin', 'JsonSerializableMixin', 'TimestampMixin', 'Entity', 'object']
Product(entity_id=7, created_at=datetime.datetime(2026, 11, 7, 9, 0), updated_at=datetime.datetime(2026, 11, 7, 9, 45), sku='SOAP', price_paise=4950)


The flat JSON mixin fails on the `datetime` attributes the timestamp mixin added:
`json` does not know how to encode them. A plain class with only JSON-friendly
values works fine. (Lab 2.2 exercise 2 fixes this.)

In [84]:
class Tag(JsonSerializableMixin):
    """Only JSON-friendly attributes."""

    def __init__(self, label: str, colour: str) -> None:
        self.label = label
        self.colour = colour


print(Tag("organic", "green").to_json())
print(raised_message(soap.to_json))

{"colour": "green", "label": "organic"}
TypeError: Object of type datetime is not JSON serializable


**Wrong order.** If the base class comes first and defines the same method, the
mixin never gets a chance: `AuditedEntity.__repr__` is found before
`ReprMixin.__repr__`.

In [85]:
class AuditedEntity(Entity):
    """A base class with its own __repr__."""

    def __repr__(self) -> str:
        return f"<AuditedEntity {self.entity_id}>"


class WrongOrder(AuditedEntity, ReprMixin):
    """Base first, mixin last: the mixin's __repr__ is shadowed."""


class RightOrder(ReprMixin, AuditedEntity):
    """Mixin first: the mixin's __repr__ wins."""


print(f"{mro_names(WrongOrder)=}")
print(f"{WrongOrder(entity_id=1)!r}  vs  {RightOrder(entity_id=1)!r}")

mro_names(WrongOrder)=['WrongOrder', 'AuditedEntity', 'Entity', 'ReprMixin', 'object']
<AuditedEntity 1>  vs  RightOrder(entity_id=1)


**Mixins in Django and DRF.** Web frameworks lean heavily on mixins. Django's
class-based views combine permission and behaviour mixins with a generic view, and
Django REST Framework builds its generic API views from one mixin per action:

```python
# Django: permission mixin first, generic view last
class ArticleListView(LoginRequiredMixin, ListView):
    model = Article

# Django REST Framework: one mixin per HTTP action, base view last
class ArticleList(mixins.ListModelMixin, mixins.CreateModelMixin, generics.GenericAPIView):
    queryset = Article.objects.all()
    serializer_class = ArticleSerializer

    def get(self, request, *args, **kwargs):
        return self.list(request, *args, **kwargs)   # from ListModelMixin

    def post(self, request, *args, **kwargs):
        return self.create(request, *args, **kwargs)  # from CreateModelMixin
```

Neither framework is a course dependency, so the cell below only checks whether
they are installed. The standard library uses the same pattern:
`socketserver.ThreadingTCPServer` is literally `ThreadingMixIn` + `TCPServer`.

In [86]:
for package_name in ["django", "rest_framework"]:
    if importlib.util.find_spec(package_name) is None:
        print(f"{package_name} is not installed: skipping (not needed for this course)")
    else:
        print(f"{package_name} is installed")
print(f"{mro_names(socketserver.ThreadingTCPServer)=}")

django is not installed: skipping (not needed for this course)
rest_framework is not installed: skipping (not needed for this course)
mro_names(socketserver.ThreadingTCPServer)=['ThreadingTCPServer', 'ThreadingMixIn', 'TCPServer', 'BaseServer', 'object']


### 2.2.5 Composition over inheritance

**Inheritance** models *is-a*: an `ExpressShipment` is a `Shipment` and can be used
anywhere a `Shipment` is expected. **Composition** models *has-a*: an object keeps
other objects as attributes and **delegates** work to them.

| Question | If yes |
| --- | --- |
| Is every subclass instance usable wherever the base is expected, with no surprises? | Inheritance may fit |
| Do you only want to *reuse some code*? | Composition |
| Would you need one subclass per *combination* of features? | Composition |
| Should the behaviour be swappable at run time or in tests? | Composition |

**Inheriting from a built-in exposes too much.** A "top 3" list that subclasses
`list` guards `append`, but every other mutating method (`extend`, `insert`, `+=`)
walks straight past the rule.

In [87]:
class TopThreeList(list[str]):
    """BUG: is-a list, so every list method can break the size rule."""

    def append(self, item: str) -> None:
        if len(self) >= 3:
            raise ValueError("top three is full")
        super().append(item)


podium = TopThreeList()
for runner in ["Asha", "Ravi", "Meena"]:
    podium.append(runner)
print(raised_message(lambda: podium.append("John")))
podium.extend(["John", "Fatima"])  # not guarded
podium.insert(0, "Arjun")  # not guarded
print(f"{podium=}, {len(podium)=}")

ValueError: top three is full
podium=['Arjun', 'Asha', 'Ravi', 'Meena', 'John', 'Fatima'], len(podium)=6


**Composition with explicit delegation.** `TopThree` *has* a list and exposes only
what it wants. The rule cannot be bypassed because there is no `extend` or
`insert` to call.

In [88]:
class TopThree:
    """Has a list; delegates only the safe operations to it."""

    def __init__(self) -> None:
        self._runners: list[str] = []

    def add(self, runner: str) -> None:
        """Add a finisher, refusing a fourth one."""
        if len(self._runners) >= 3:
            raise ValueError("top three is full")
        self._runners.append(runner)

    def __len__(self) -> int:
        return len(self._runners)

    def __iter__(self) -> Iterator[str]:
        return iter(self._runners)


podium = TopThree()
for runner in ["Asha", "Ravi", "Meena"]:
    podium.add(runner)
print(raised_message(lambda: podium.add("John")))
print(f"{list(podium)=}, {len(podium)=}, {hasattr(podium, 'extend')=}")

ValueError: top three is full
list(podium)=['Asha', 'Ravi', 'Meena'], len(podium)=3, hasattr(podium, 'extend')=False


**Automatic delegation with `__getattr__`.** Python calls `__getattr__` only when
normal lookup fails, so a wrapper can forward every unknown attribute to the object
it wraps (a proxy). Unit 2.5 covers attribute hooks in detail.

In [89]:
class RecordingProxy:
    """Forwards every unknown attribute to the wrapped object and records the name."""

    def __init__(self, target: object) -> None:
        self._target = target
        self.accessed: list[str] = []

    def __getattr__(self, name: str) -> Any:
        self.accessed.append(name)
        return getattr(self._target, name)


proxied_text = RecordingProxy("  Hello, Pune  ")
print(f"{proxied_text.strip().upper()=}")
print(f"{proxied_text.count('l')=}")
print(f"{proxied_text.accessed=}")

proxied_text.strip().upper()='HELLO, PUNE'
proxied_text.count('l')=2
proxied_text.accessed=['strip', 'count']


**Refactoring a `Notifier` hierarchy.** A notifier sends messages over a channel
(e-mail, SMS) and may add features (logging, retries). With inheritance every
combination needs its own class, and the feature code is copied into each one.

In [90]:
class Notifier:
    def send(self, recipient: str, message: str) -> str:
        raise NotImplementedError


class EmailNotifier(Notifier):
    def send(self, recipient: str, message: str) -> str:
        return f"email to {recipient}: {message}"


class SmsNotifier(Notifier):
    def send(self, recipient: str, message: str) -> str:
        return f"sms to {recipient}: {message}"


class LoggingEmailNotifier(EmailNotifier):
    def __init__(self, log: list[str]) -> None:
        self.log = log

    def send(self, recipient: str, message: str) -> str:
        self.log.append(f"sending to {recipient}")  # duplicated in every Logging* class
        return super().send(recipient, message)


notifier_log: list[str] = []
print(LoggingEmailNotifier(notifier_log).send("asha@example.com", "Order shipped"))
print(f"{notifier_log=}")

email to asha@example.com: Order shipped
notifier_log=['sending to asha@example.com']


The class count grows **multiplicatively**: every channel times every subset of
features. Adding one channel or one feature means writing many new classes.

In [91]:
def inheritance_class_count(channel_count: int, feature_count: int) -> int:
    """Classes needed for every channel with every subset of features."""
    return channel_count * 2**feature_count


channels = ["Email", "Sms"]
features = ["Logging", "Retrying"]
combinations = [
    "".join(chosen) + channel + "Notifier"
    for channel in channels
    for size in range(len(features) + 1)
    for chosen in itertools.combinations(features, size)
]
print(combinations)
for channel_count, feature_count in [(2, 2), (3, 2), (3, 3), (4, 4)]:
    print(
        f"{channel_count} channels, {feature_count} features -> "
        f"{inheritance_class_count(channel_count, feature_count)} classes"
    )

['EmailNotifier', 'LoggingEmailNotifier', 'RetryingEmailNotifier', 'LoggingRetryingEmailNotifier', 'SmsNotifier', 'LoggingSmsNotifier', 'RetryingSmsNotifier', 'LoggingRetryingSmsNotifier']
2 channels, 2 features -> 8 classes
3 channels, 2 features -> 12 classes
3 channels, 3 features -> 24 classes
4 channels, 4 features -> 64 classes


**The composition version.** Each channel is a small class with `send()`. Each
feature is a **wrapper** that has an inner channel and adds one behaviour (the
decorator pattern, as objects). The `Notifier` has a channel. Classes now grow
**additively** (channels + features), and combinations are built at run time.

A `FlakyChannel` fails a fixed number of times so the retry feature can be
demonstrated deterministically.

In [92]:
class EmailChannel:
    def send(self, recipient: str, message: str) -> str:
        return f"email to {recipient}: {message}"


class SmsChannel:
    def send(self, recipient: str, message: str) -> str:
        return f"sms to {recipient}: {message}"


class FlakyChannel:
    """Wraps a channel and raises ConnectionError for the first `failures` calls."""

    def __init__(self, inner: Any, failures: int) -> None:
        self.inner = inner
        self.failures_left = failures

    def send(self, recipient: str, message: str) -> str:
        if self.failures_left > 0:
            self.failures_left -= 1
            raise ConnectionError("gateway timeout")
        return self.inner.send(recipient, message)

The two feature wrappers. Each has an `inner` channel and adds one behaviour:

In [93]:
class LoggingChannel:
    """Feature: log each send attempt, then delegate."""

    def __init__(self, inner: Any, log: list[str]) -> None:
        self.inner = inner
        self.log = log

    def send(self, recipient: str, message: str) -> str:
        self.log.append(f"sending to {recipient}")
        return self.inner.send(recipient, message)


class RetryingChannel:
    """Feature: retry on ConnectionError, up to `attempts` tries in total."""

    def __init__(self, inner: Any, attempts: int) -> None:
        self.inner = inner
        self.attempts = attempts

    def send(self, recipient: str, message: str) -> str:
        for attempt in range(1, self.attempts + 1):
            try:
                return self.inner.send(recipient, message)
            except ConnectionError:
                if attempt == self.attempts:
                    raise
        raise AssertionError("unreachable")

The `Notifier` only has a channel. It knows nothing about logging or retries, so it
never changes when a feature is added.

In [94]:
class Notifier:
    """Has a channel; knows nothing about logging or retries."""

    def __init__(self, channel: Any) -> None:
        self.channel = channel

    def notify(self, recipient: str, message: str) -> str:
        return self.channel.send(recipient, message)

Any combination is now one expression. The SMS gateway fails twice; the retrying
wrapper tries three times; the logging wrapper (inside the retry) records every
attempt.

In [95]:
delivery_log: list[str] = []
reliable_sms = Notifier(
    RetryingChannel(
        LoggingChannel(FlakyChannel(SmsChannel(), failures=2), delivery_log), attempts=3
    )
)
print(reliable_sms.notify("+91-98xxxxxx01", "OTP 4821"))
print(f"{delivery_log=}")
print(Notifier(EmailChannel()).notify("asha@example.com", "Order shipped"))
print(
    raised_message(
        lambda: Notifier(RetryingChannel(FlakyChannel(EmailChannel(), 5), 2)).notify("x", "y")
    )
)

sms to +91-98xxxxxx01: OTP 4821
delivery_log=['sending to +91-98xxxxxx01', 'sending to +91-98xxxxxx01', 'sending to +91-98xxxxxx01']
email to asha@example.com: Order shipped
ConnectionError: gateway timeout


**The Square/Rectangle Liskov violation.** The **Liskov substitution principle**
(LSP) says code written for a base class must keep working, with no surprises, when
given a subclass. Mathematically a square *is a* rectangle, so `Square(Rectangle)`
looks natural. But a mutable `Rectangle` promises that setting `width` leaves
`height` alone, and a square cannot keep that promise.

In [96]:
class Rectangle:
    """A mutable rectangle: width and height change independently."""

    def __init__(self, width: float, height: float) -> None:
        self._width = width
        self._height = height

    @property
    def width(self) -> float:
        return self._width

    @width.setter
    def width(self, value: float) -> None:
        self._width = value

    @property
    def height(self) -> float:
        return self._height

    @height.setter
    def height(self, value: float) -> None:
        self._height = value

    @property
    def area(self) -> float:
        return self._width * self._height

`Square` keeps its sides equal by overriding both setters. `stretch_width` is
written against `Rectangle` and relies on its promise; it works for a rectangle and
silently breaks for a square, even though `isinstance(square, Rectangle)` is `True`.

In [97]:
class Square(Rectangle):
    """Keeps width == height, which breaks Rectangle's promise."""

    def __init__(self, side: float) -> None:
        super().__init__(side, side)

    @Rectangle.width.setter
    def width(self, value: float) -> None:
        self._width = self._height = value

    @Rectangle.height.setter
    def height(self, value: float) -> None:
        self._width = self._height = value


def stretch_width(shape: Rectangle, factor: float) -> str:
    """Client code: doubling the width should double the area."""
    expected_area = shape.area * factor
    shape.width *= factor
    return f"{type(shape).__name__}: expected {expected_area}, got {shape.area}"


print(stretch_width(Rectangle(2, 3), 2))
print(stretch_width(Square(3), 2))
print(f"{isinstance(Square(3), Rectangle)=}")

Rectangle: expected 12, got 12
Square: expected 18, got 36
isinstance(Square(3), Rectangle)=True


**One fix:** make the shapes immutable values. "Changing" the width returns a **new**
shape, so a square stretched in one direction honestly becomes a rectangle, and
`Square` no longer needs to inherit from `Rectangle` at all. Lab 2.3 builds shapes
this way behind a shared abstract base class.

In [98]:
@dataclass(frozen=True)
class FixedRectangle:
    width: float
    height: float

    def with_width(self, width: float) -> FixedRectangle:
        return FixedRectangle(width, self.height)


@dataclass(frozen=True)
class FixedSquare:
    side: float

    def with_width(self, width: float) -> FixedRectangle:
        return FixedRectangle(width, self.side)  # a stretched square is a rectangle


print(f"{FixedSquare(3).with_width(6)=}")
print(raised_error_name(lambda: setattr(FixedSquare(3), "side", 6)))

FixedSquare(3).with_width(6)=FixedRectangle(width=6, height=3)
FrozenInstanceError


### 2.2.6 SOLID in Python

SOLID is five design principles from the object-oriented world. In Python they
often take a lighter form (a function, a Protocol), but the ideas carry over.

| Letter | Principle | One-line meaning |
| --- | --- | --- |
| S | Single responsibility | A class should have one reason to change |
| O | Open/closed | Add behaviour by adding code, not by editing working code |
| L | Liskov substitution | Subclasses must honour the base class's promises |
| I | Interface segregation | Depend on small interfaces, not fat ones |
| D | Dependency inversion | Depend on abstractions; inject concrete implementations |

**S — Single responsibility.** An `Invoice` that computes totals, renders text and
saves itself changes for three unrelated reasons (tax rules, layout, storage).
Split it so each class has one.

In [99]:
class InvoiceCalculator:
    """Changes when tax rules change."""

    def total(self, line_amounts: list[Decimal], tax_rate_percent: Decimal) -> Decimal:
        return sum(line_amounts, Decimal("0")) * (1 + tax_rate_percent / 100)


class InvoiceRenderer:
    """Changes when the layout changes."""

    def render(self, invoice_number: str, total: Decimal) -> str:
        return f"Invoice {invoice_number}: INR {total:.2f}"


class InvoiceArchive:
    """Changes when storage changes."""

    def __init__(self) -> None:
        self.saved: dict[str, str] = {}

    def save(self, invoice_number: str, text: str) -> None:
        self.saved[invoice_number] = text


invoice_total = InvoiceCalculator().total([Decimal("100"), Decimal("250")], Decimal("18"))
invoice_archive = InvoiceArchive()
invoice_archive.save("INV-9", InvoiceRenderer().render("INV-9", invoice_total))
print(invoice_archive.saved)

{'INV-9': 'Invoice INV-9: INR 413.00'}


**O — Open/closed.** `checkout_total` applies whatever discount rules it is given.
A new rule is a new class; `checkout_total` never changes. (An `if rule == "…"`
chain would have to be edited for every new rule.)

In [100]:
class PercentOff:
    def __init__(self, percent: Decimal) -> None:
        self.percent = percent

    def apply(self, amount: Decimal) -> Decimal:
        return amount * (1 - self.percent / 100)


class FlatOff:
    def __init__(self, rupees: Decimal) -> None:
        self.rupees = rupees

    def apply(self, amount: Decimal) -> Decimal:
        return max(amount - self.rupees, Decimal("0"))


def checkout_total(amount: Decimal, rules: Iterable[Any]) -> Decimal:
    """Closed for modification: new rules plug in without editing this function."""
    for rule in rules:
        amount = rule.apply(amount)
    return amount


weekend_rules = [PercentOff(Decimal("10")), FlatOff(Decimal("50"))]
print(f"{checkout_total(Decimal('1000'), weekend_rules)=}")

checkout_total(Decimal('1000'), weekend_rules)=Decimal('850.0')


**L — Liskov substitution.** Besides Square/Rectangle (2.2.5), a classic violation
is a subclass that refuses an operation the base class promises. `warm_up` works for
any `Cache`, and crashes for the `ReadOnlyCache` "subtype". The fix is to not make
it a subclass (or to split the interface, as the next principle shows).

In [101]:
class Cache:
    def __init__(self) -> None:
        self._entries: dict[str, str] = {}

    def get(self, key: str) -> str | None:
        return self._entries.get(key)

    def set(self, key: str, value: str) -> None:
        self._entries[key] = value


class ReadOnlyCache(Cache):
    """BUG: strengthens the precondition of set() to 'never'."""

    def set(self, key: str, value: str) -> None:
        raise PermissionError("read-only cache")


def warm_up(cache: Cache) -> str | None:
    """Written against Cache: set, then get."""
    cache.set("home", "<html>…</html>")
    return cache.get("home")


print(f"{warm_up(Cache())=}")
print(raised_message(lambda: warm_up(ReadOnlyCache())))

warm_up(Cache())='<html>…</html>'
PermissionError: read-only cache


**I — Interface segregation.** A report viewer only reads, so it should depend on a
small "readable" interface, not on a fat storage interface with writes and deletes.
`typing.Protocol` (2.3.3) describes such interfaces; `PublicMirror` only implements
reading and still fits the viewer.

In [102]:
class Readable(Protocol):
    def read(self, key: str) -> str: ...


class Writable(Protocol):
    def write(self, key: str, value: str) -> None: ...


class ArchiveStore:
    """Implements both small interfaces."""

    def __init__(self) -> None:
        self.data = {"q3": "revenue up 12%"}

    def read(self, key: str) -> str:
        return self.data[key]

    def write(self, key: str, value: str) -> None:
        self.data[key] = value


class PublicMirror:
    """Read-only copy: implements only Readable."""

    def read(self, key: str) -> str:
        return "revenue up 12% (mirror)"


def show_report(store: Readable, key: str) -> str:
    """Needs reading only, so it asks for Readable."""
    return f"[{key}] {store.read(key)}"


print(show_report(ArchiveStore(), "q3"))
print(show_report(PublicMirror(), "q3"))

[q3] revenue up 12%
[q3] revenue up 12% (mirror)


**D — Dependency inversion.** `OrderService` does not create its own database
connection; it receives any object with `save` and `find`. Production passes a real
repository, tests pass an in-memory one. FastAPI's `Depends` (Phase 6) is built on
this idea.

In [103]:
class InMemoryOrderRepository:
    """A test double: stores orders in a dict."""

    def __init__(self) -> None:
        self.orders: dict[str, Decimal] = {}

    def save(self, order_id: str, amount: Decimal) -> None:
        self.orders[order_id] = amount

    def find(self, order_id: str) -> Decimal | None:
        return self.orders.get(order_id)


class OrderService:
    """High-level policy; the repository is injected, not constructed here."""

    def __init__(self, repository: Any) -> None:
        self.repository = repository

    def place(self, order_id: str, amount: Decimal) -> str:
        if self.repository.find(order_id) is not None:
            return f"{order_id} already placed"
        self.repository.save(order_id, amount)
        return f"{order_id} placed"


service = OrderService(InMemoryOrderRepository())
print(service.place("ORD-1", Decimal("499")), "|", service.place("ORD-1", Decimal("499")))

ORD-1 placed | ORD-1 already placed


### Lab 2.2

Three exercises. For each, replace every `raise NotImplementedError` in the stub,
then run the check cell. Until then the checks print FAIL; that is expected. Three
structural checks already pass on the stub ("mixin has no __init__" and "in the MRO"
in exercise 2, "Report has no base class" in exercise 3): keep them passing.

#### Exercise 1 — compute three MROs by hand

The cell below defines three hierarchies. For each, work out the MRO **on paper**
with `L[C] = C + merge(...)` (2.2.3), writing down every merge step, then make
`predict_mro(case_name)` return your answer as a list of class names (ending with
`"object"`). Do not call `__mro__` in your answer; the check compares it with
`__mro__` for you.

> **Note:** a failing check prints the real MRO. Finish all three on paper before
> running it.

In [104]:
# Case "diamond"
class Animal: ...


class Swimmer(Animal): ...


class Flyer(Animal): ...


class Duck(Swimmer, Flyer): ...


# Case "services"
class Base: ...


class LoggingMixin: ...


class CacheMixin: ...


class Service(LoggingMixin, CacheMixin, Base): ...


class AuditMixin(LoggingMixin): ...


class AuditedService(AuditMixin, Service): ...


# Case "classic"
class Persistent: ...


class Versioned: ...


class Searchable: ...


class Document(Persistent, Versioned): ...


class Index(Persistent, Searchable): ...


class SearchableDocument(Document, Index): ...


MRO_CASES: dict[str, type] = {
    "diamond": Duck,
    "services": AuditedService,
    "classic": SearchableDocument,
}


def predict_mro(case_name: str) -> list[str]:
    """Return the class names of the case's MRO, computed by hand."""
    raise NotImplementedError("Lab 2.2: write your three MROs in predict_mro")

Run the checks.

In [105]:
run_checks(
    [
        (case_name, lambda case_name=case_name: predict_mro(case_name), mro_names(cls))
        for case_name, cls in MRO_CASES.items()
    ]
)

FAIL  diamond: raised NotImplementedError: Lab 2.2: write your three MROs in predict_mro
FAIL  services: raised NotImplementedError: Lab 2.2: write your three MROs in predict_mro
FAIL  classic: raised NotImplementedError: Lab 2.2: write your three MROs in predict_mro
0/3 checks passed


#### Exercise 2 — a JSON mixin for nested objects and datetimes

Write `JsonMixin` with two methods and **no `__init__`**:

- `to_dict()` returns the instance's attributes (from `vars(self)`) **except** names
  starting with `_`, with each value converted recursively:

  | Value | Becomes |
  | --- | --- |
  | another `JsonMixin` object | its `to_dict()` |
  | `datetime` or `date` | `isoformat()` string |
  | `Decimal` | `str(value)` |
  | `list` or `tuple` | a list of converted items |
  | `dict` | a dict with `str` keys and converted values |
  | anything else | unchanged |

- `to_json(**dumps_options)` returns `json.dumps(self.to_dict(), **dumps_options)`.

The sample classes in the cell after the stub use the mixin.

In [106]:
class JsonMixin:
    """Adds to_dict() / to_json() to any class whose state lives in instance attributes."""

    def to_dict(self) -> dict[str, Any]:
        """Public attributes as JSON-ready values."""
        raise NotImplementedError("Lab 2.2: implement JsonMixin.to_dict")

    def to_json(self, **dumps_options: Any) -> str:
        """Serialize to_dict() with json.dumps."""
        raise NotImplementedError("Lab 2.2: implement JsonMixin.to_json")

Sample classes (given): a `Customer` with nested `Address` and `LineItem` objects,
a `datetime`, a `date`, a tuple, a dict holding another object, and a private
`_password_hash` that must not be exported.

In [107]:
class Address(JsonMixin):
    """A postal address."""

    def __init__(self, city: str, pin_code: str) -> None:
        self.city = city
        self.pin_code = pin_code


class LineItem(JsonMixin):
    """One purchased product."""

    def __init__(self, sku: str, unit_price: Decimal, quantity: int) -> None:
        self.sku = sku
        self.unit_price = unit_price
        self.quantity = quantity


class Customer(JsonMixin):
    """A customer with nested objects and a private field."""

    def __init__(
        self,
        name: str,
        joined_at: datetime,
        birthday: date,
        addresses: list[Address],
        last_order: list[LineItem],
        tags: tuple[str, ...],
        preferences: dict[str, object],
    ) -> None:
        self.name, self.joined_at, self.birthday = name, joined_at, birthday
        self.addresses, self.last_order = addresses, last_order
        self.tags, self.preferences = tags, preferences
        self._password_hash = "pbkdf2$not-for-json"


def make_sample_customer() -> Customer:
    """A fresh customer for the checks."""
    return Customer(
        name="Fatima",
        joined_at=datetime(2026, 1, 15, 9, 30),
        birthday=date(1994, 8, 2),
        addresses=[Address("Pune", "411001"), Address("Goa", "403001")],
        last_order=[LineItem("SOAP", Decimal("49.50"), 2)],
        tags=("vip", "early-adopter"),
        preferences={"newsletter": True, "home": Address("Pune", "411001")},
    )

Run the checks.

In [108]:
EXPECTED_CUSTOMER_DICT = {
    "name": "Fatima",
    "joined_at": "2026-01-15T09:30:00",
    "birthday": "1994-08-02",
    "addresses": [{"city": "Pune", "pin_code": "411001"}, {"city": "Goa", "pin_code": "403001"}],
    "last_order": [{"sku": "SOAP", "unit_price": "49.50", "quantity": 2}],
    "tags": ["vip", "early-adopter"],
    "preferences": {"newsletter": True, "home": {"city": "Pune", "pin_code": "411001"}},
}


def address_with_landmark() -> dict[str, Any]:
    """to_dict() sees attributes added after __init__."""
    address = Address("Goa", "403001")
    address.landmark = "Fort"  # type: ignore[attr-defined]
    return address.to_dict()


run_checks(
    [
        ("to_dict", lambda: make_sample_customer().to_dict(), EXPECTED_CUSTOMER_DICT),
        (
            "private field skipped",
            lambda: "_password_hash" in make_sample_customer().to_dict(),
            False,
        ),
        (
            "to_json round trip",
            lambda: json.loads(make_sample_customer().to_json()),
            EXPECTED_CUSTOMER_DICT,
        ),
        (
            "dumps options forwarded",
            lambda: Address("Pune", "411001").to_json(sort_keys=True, indent=None),
            '{"city": "Pune", "pin_code": "411001"}',
        ),
        (
            "indent forwarded",
            lambda: Address("Goa", "403001").to_json(indent=2).splitlines()[1],
            '  "city": "Goa",',
        ),
        (
            "attributes added later",
            address_with_landmark,
            {"city": "Goa", "pin_code": "403001", "landmark": "Fort"},
        ),
        ("mixin has no __init__", lambda: "__init__" in JsonMixin.__dict__, False),
        ("in the MRO", lambda: JsonMixin in Customer.__mro__, True),
    ]
)

FAIL  to_dict: raised NotImplementedError: Lab 2.2: implement JsonMixin.to_dict
FAIL  private field skipped: raised NotImplementedError: Lab 2.2: implement JsonMixin.to_dict
FAIL  to_json round trip: raised NotImplementedError: Lab 2.2: implement JsonMixin.to_json
FAIL  dumps options forwarded: raised NotImplementedError: Lab 2.2: implement JsonMixin.to_json
FAIL  indent forwarded: raised NotImplementedError: Lab 2.2: implement JsonMixin.to_json
FAIL  attributes added later: raised NotImplementedError: Lab 2.2: implement JsonMixin.to_dict
PASS  mixin has no __init__
PASS  in the MRO
2/8 checks passed


#### Exercise 3 — refactor an inheritance tree into composition

The legacy code below picks a report **format** by subclassing and adds
**deliveries** (e-mail, archive) by subclassing again. It already needs five classes
for three combinations, `EmailedTextReport` duplicates `EmailedCsvReport`, and an
"archived text report" would need yet another class.

Rewrite it with composition:

- `CsvFormatter` and `TextFormatter`, each with `format(title, rows) -> str`
  producing exactly what `CsvReport.render` / `TextReport.render` produce.
- `EmailDelivery(outbox)` and `ArchiveDelivery(archive)`, each with
  `deliver(title, content) -> None` doing what the legacy `publish` overrides do.
- `Report(title, rows, formatter, deliveries=())` with no base class (other than
  `object`); `publish()` formats, calls every delivery **in order**, and returns
  the content. Store the deliveries as a list in `self.deliveries`.

First, the legacy code (given, working).

In [109]:
type SalesRows = list[tuple[str, int]]

SAMPLE_SALES: SalesRows = [("Soap", 120), ("Rice 5 kg", 45), ("Tea", 300)]


class LegacyReport:
    """Legacy base: subclasses pick a format by overriding render()."""

    def __init__(self, title: str, rows: SalesRows) -> None:
        self.title = title
        self.rows = rows

    def render(self) -> str:
        raise NotImplementedError  # subclasses override this

    def publish(self) -> str:
        return self.render()


class CsvReport(LegacyReport):
    def render(self) -> str:
        body = "\n".join(f"{product},{units}" for product, units in self.rows)
        return f"# {self.title}\nproduct,units\n{body}"


class TextReport(LegacyReport):
    def render(self) -> str:
        lines = [self.title, "=" * len(self.title)]
        lines += [f"{product:<12}{units:>5}" for product, units in self.rows]
        return "\n".join(lines)

The legacy delivery subclasses (given):

In [110]:
class EmailedCsvReport(CsvReport):
    def __init__(self, title: str, rows: SalesRows, outbox: list[tuple[str, str]]) -> None:
        super().__init__(title, rows)
        self.outbox = outbox

    def publish(self) -> str:
        content = super().publish()
        self.outbox.append((self.title, content))
        return content


class EmailedTextReport(TextReport):  # the same code as EmailedCsvReport, duplicated
    def __init__(self, title: str, rows: SalesRows, outbox: list[tuple[str, str]]) -> None:
        super().__init__(title, rows)
        self.outbox = outbox

    def publish(self) -> str:
        content = super().publish()
        self.outbox.append((self.title, content))
        return content


class ArchivedEmailedCsvReport(EmailedCsvReport):
    def __init__(
        self, title: str, rows: SalesRows, outbox: list[tuple[str, str]], archive: dict[str, str]
    ) -> None:
        super().__init__(title, rows, outbox)
        self.archive = archive

    def publish(self) -> str:
        content = super().publish()
        self.archive[self.title] = content
        return content


print(TextReport("Weekly sales", SAMPLE_SALES).publish())

Weekly sales
Soap          120
Rice 5 kg      45
Tea           300


Your composition-based version:

In [111]:
class CsvFormatter:
    def format(self, title: str, rows: SalesRows) -> str:
        """Same output as CsvReport.render."""
        raise NotImplementedError("Lab 2.2: implement CsvFormatter.format")


class TextFormatter:
    def format(self, title: str, rows: SalesRows) -> str:
        """Same output as TextReport.render."""
        raise NotImplementedError("Lab 2.2: implement TextFormatter.format")


class EmailDelivery:
    def __init__(self, outbox: list[tuple[str, str]]) -> None:
        raise NotImplementedError("Lab 2.2: implement EmailDelivery")

    def deliver(self, title: str, content: str) -> None:
        raise NotImplementedError("Lab 2.2: implement EmailDelivery.deliver")


class ArchiveDelivery:
    def __init__(self, archive: dict[str, str]) -> None:
        raise NotImplementedError("Lab 2.2: implement ArchiveDelivery")

    def deliver(self, title: str, content: str) -> None:
        raise NotImplementedError("Lab 2.2: implement ArchiveDelivery.deliver")


class Report:
    def __init__(
        self, title: str, rows: SalesRows, formatter: Any, deliveries: Iterable[Any] = ()
    ) -> None:
        raise NotImplementedError("Lab 2.2: implement Report")

    def publish(self) -> str:
        raise NotImplementedError("Lab 2.2: implement Report.publish")

Run the checks. They compare every output and side effect with the legacy classes.

In [112]:
def legacy_outputs() -> tuple[str, str, list[tuple[str, str]], dict[str, str]]:
    """Publish through the legacy tree and capture every side effect."""
    outbox: list[tuple[str, str]] = []
    archive: dict[str, str] = {}
    text_content = EmailedTextReport("Weekly sales", SAMPLE_SALES, outbox).publish()
    csv_content = ArchivedEmailedCsvReport("Weekly sales", SAMPLE_SALES, outbox, archive).publish()
    return text_content, csv_content, outbox, archive


def composed_outputs() -> tuple[str, str, list[tuple[str, str]], dict[str, str]]:
    """Publish the same two reports through Report + formatters + deliveries."""
    outbox: list[tuple[str, str]] = []
    archive: dict[str, str] = {}
    text_content = Report(
        "Weekly sales", SAMPLE_SALES, TextFormatter(), [EmailDelivery(outbox)]
    ).publish()
    csv_deliveries = [EmailDelivery(outbox), ArchiveDelivery(archive)]
    csv_content = Report("Weekly sales", SAMPLE_SALES, CsvFormatter(), csv_deliveries).publish()
    return text_content, csv_content, outbox, archive


def new_combination() -> tuple[str, dict[str, str]]:
    """Archived text report: a combination the legacy tree has no class for."""
    archive: dict[str, str] = {}
    content = Report("Stock", [("Tea", 7)], TextFormatter(), [ArchiveDelivery(archive)]).publish()
    return content, archive


run_checks(
    [
        (
            "plain CSV",
            lambda: Report("Q3", SAMPLE_SALES, CsvFormatter()).publish(),
            CsvReport("Q3", SAMPLE_SALES).publish(),
        ),
        (
            "plain text",
            lambda: Report("Q3", SAMPLE_SALES, TextFormatter()).publish(),
            TextReport("Q3", SAMPLE_SALES).publish(),
        ),
        ("same side effects as legacy", composed_outputs, legacy_outputs()),
        (
            "new combination, no new class",
            new_combination,
            ("Stock\n=====\nTea             7", {"Stock": "Stock\n=====\nTea             7"}),
        ),
        ("Report has no base class", lambda: Report.__bases__, (object,)),
        ("deliveries default to none", lambda: Report("Q3", [], CsvFormatter()).deliveries, []),
    ]
)

FAIL  plain CSV: raised NotImplementedError: Lab 2.2: implement Report
FAIL  plain text: raised NotImplementedError: Lab 2.2: implement Report
FAIL  same side effects as legacy: raised NotImplementedError: Lab 2.2: implement EmailDelivery
FAIL  new combination, no new class: raised NotImplementedError: Lab 2.2: implement ArchiveDelivery
PASS  Report has no base class
FAIL  deliveries default to none: raised NotImplementedError: Lab 2.2: implement Report
1/6 checks passed


### Recap — Unit 2.2

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| Overriding | A subclass method replaces the base one; `self.method()` dispatches on the instance's class | method overriding |
| Extending | Call the original with `super().method(...)`; forgetting `super().__init__` fails silently | `super.method()` |
| `@override` | Marks intent; type checkers catch typos (3.12+) | TS `override` |
| `isinstance` / `issubclass` | Follow the MRO; accept tuples and `A | B` | `instanceof` |
| `object` | The root of every class; source of default dunders | `Object.prototype` |
| Zero-arg `super()` | Reads the hidden `__class__` cell and the first argument | `[[HomeObject]]` |
| Next in MRO | `super()` can reach a sibling under multiple inheritance | — (always the static parent) |
| Cooperative `**kwargs` | Keyword-only args, take yours, pass the rest, `object` gets nothing | — |
| C3 MRO | `L[C] = C + merge(...)`; take the first head not in any tail | prototype chain |
| Inconsistent MRO | Base listed before its subclass, or conflicting orders → `TypeError` | — |
| Mixins | One capability each, mixins left, main base right | `Object.assign`, class-factory mixins |
| Composition | Has-a + delegation; wrappers combine features additively | object composition |
| Liskov | Subclasses must keep the base class's promises (Square/Rectangle) | same |
| SOLID | S, O, L, I, D: small focused classes, plug-in extension, injected dependencies | same |

**Further reading**

- [Tutorial: inheritance](https://docs.python.org/3/tutorial/classes.html#inheritance),
  [multiple inheritance](https://docs.python.org/3/tutorial/classes.html#multiple-inheritance)
- [`super`](https://docs.python.org/3/library/functions.html#super),
  [`isinstance`](https://docs.python.org/3/library/functions.html#isinstance),
  [`issubclass`](https://docs.python.org/3/library/functions.html#issubclass),
  [`typing.override`](https://docs.python.org/3/library/typing.html#typing.override)
- [The Python 2.3 Method Resolution Order](https://docs.python.org/3/howto/mro.html)
  (the C3 algorithm, with worked examples)
- [`socketserver` mix-in classes](https://docs.python.org/3/library/socketserver.html#server-objects),
  [`__getattr__`](https://docs.python.org/3/reference/datamodel.html#object.__getattr__)
- [PEP 3135 — New `super`](https://peps.python.org/pep-3135/),
  [PEP 698 — `@override`](https://peps.python.org/pep-0698/),
  [PEP 253 — Subtyping built-in types](https://peps.python.org/pep-0253/)

## Unit 2.3 — Abstractions: ABCs, `collections.abc` and Protocols

**Objectives**

- Define abstract base classes with abstract methods and properties, and use the
  template-method pattern.
- Implement a few methods of a `collections.abc` class and get the rest for free;
  use `register` and `__subclasshook__`.
- Describe interfaces structurally with `typing.Protocol`, as in TypeScript, and
  know what `@runtime_checkable` can and cannot check.
- Choose between an ABC and a Protocol.
- Write polymorphic code with duck typing and EAFP instead of type checks.

**JS / TS bridge**

| JavaScript / TypeScript | Python | The difference that bites |
| --- | --- | --- |
| TS `abstract class` / `abstract method()` | `class X(ABC)` + `@abstractmethod` | Checked at **run time**, when you instantiate, not at compile time |
| TS `interface` (structural) | `typing.Protocol` | Checked by a type checker (mypy, pyright); ignored at run time by default |
| TS `implements Shape` | optional: `class Circle(Shape)` with a Protocol or ABC base | Not needed for Protocols: matching the shape is enough |
| `x instanceof SomeInterface` (impossible: interfaces are erased) | `isinstance(x, Proto)` with `@runtime_checkable` | Only checks that the **names** exist |
| `Symbol.hasInstance` | `__subclasshook__` / `ABC.register` | Customizes `isinstance` / `issubclass` |
| `Symbol.iterator` protocol | `collections.abc.Iterable` (`__iter__`) | Built-in ABCs recognize the protocol structurally |
| `ReadonlyMap<K, V>` / `Map<K, V>` | `Mapping[K, V]` / `MutableMapping[K, V]` | Implement 3 (or 5) methods, inherit the rest |
| `typeof x.close === "function"` | `try: x.close()` / `except AttributeError:` | EAFP over checks (2.3.4) |

The next cell is this unit's setup.

In [113]:
import functools
import importlib.util
import inspect
import io
import json
import os
import shutil
import sys
import timeit
import typing
from abc import ABC, ABCMeta, abstractmethod
from collections.abc import (
    Callable,
    Hashable,
    Iterable,
    Iterator,
    Mapping,
    MutableMapping,
    MutableSequence,
    Sequence,
    Set,
    Sized,
)
from decimal import Decimal
from math import pi  # noqa: F401 (Lab 2.3)
from typing import Any, NamedTuple, Protocol, runtime_checkable


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


def raised_message(thunk: Callable[[], object]) -> str:
    """Call thunk and return "ErrorName: message" for the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return f"{type(error).__name__}: {error}"
    return "no error"


print(f"{sys.version_info[:2]=}")

sys.version_info[:2]=(3, 15)


### 2.3.1 Abstract base classes

An **abstract base class** (ABC) declares methods that subclasses **must** provide.
Inherit from `abc.ABC` and decorate those methods with `@abstractmethod`. Python then
refuses to instantiate the ABC itself, or any subclass that leaves an abstract method
unimplemented. An ABC can also have ordinary (concrete) methods built on top of the
abstract ones; here `exists()` works for every storage backend.

In [114]:
class Storage(ABC):
    """Where uploaded files go. Subclasses decide how."""

    @abstractmethod
    def save(self, name: str, data: bytes) -> str:
        """Store data and return its location."""

    @abstractmethod
    def load(self, location: str) -> bytes:
        """Return the stored bytes, or raise KeyError."""

    def exists(self, location: str) -> bool:
        """Concrete method built on the abstract load()."""
        try:
            self.load(location)
        except KeyError:
            return False
        return True

The error comes at **instantiation time**: defining an incomplete subclass is fine,
creating an instance of it is not. The message names every missing method. A
complete subclass works and inherits `exists()`.

In [115]:
class HalfDoneStorage(Storage):
    """Implements save() but forgets load()."""

    def save(self, name: str, data: bytes) -> str:
        return name


class MemoryStorage(Storage):
    """Keeps files in a dict."""

    def __init__(self) -> None:
        self._files: dict[str, bytes] = {}

    def save(self, name: str, data: bytes) -> str:
        location = f"mem://{name}"
        self._files[location] = data
        return location

    def load(self, location: str) -> bytes:
        return self._files[location]


print(raised_message(Storage))
print(raised_message(HalfDoneStorage))
memory_storage = MemoryStorage()
invoice_location = memory_storage.save("invoice.pdf", b"%PDF-1.7")
print(
    f"{invoice_location=}, {memory_storage.exists(invoice_location)=}, {memory_storage.exists('mem://x')=}"
)

TypeError: Can't instantiate abstract class Storage without an implementation for abstract methods 'load', 'save'
TypeError: Can't instantiate abstract class HalfDoneStorage without an implementation for abstract method 'load'
invoice_location='mem://invoice.pdf', memory_storage.exists(invoice_location)=True, memory_storage.exists('mem://x')=False


#### Under the hood: `ABCMeta` and `__abstractmethods__`

`ABC` is just a convenience base whose metaclass is **`ABCMeta`**. When the class is
created, `ABCMeta` collects every name still marked abstract into the frozenset
`__abstractmethods__`; `object.__new__` refuses to instantiate a class whose set is
not empty. `inspect.isabstract` reads the same information.

In [116]:
print(f"{type(Storage)=}")
print(f"{sorted(Storage.__abstractmethods__)=}")
print(f"{sorted(HalfDoneStorage.__abstractmethods__)=}")
print(f"{MemoryStorage.__abstractmethods__=}")
print(f"{inspect.isabstract(Storage)=}, {inspect.isabstract(MemoryStorage)=}")


class ExplicitMetaclass(metaclass=ABCMeta):  # the same as inheriting from ABC
    @abstractmethod
    def run(self) -> None: ...


print(f"{inspect.isabstract(ExplicitMetaclass)=}")

type(Storage)=<class 'abc.ABCMeta'>
sorted(Storage.__abstractmethods__)=['load', 'save']
sorted(HalfDoneStorage.__abstractmethods__)=['load']
MemoryStorage.__abstractmethods__=frozenset()
inspect.isabstract(Storage)=True, inspect.isabstract(MemoryStorage)=False
inspect.isabstract(ExplicitMetaclass)=True


Unlike Java interfaces, an abstract method **can have a body**. Subclasses must still
override it, but they can call the shared part with `super()`.

In [117]:
class Uploader(ABC):
    @abstractmethod
    def validate(self, file_name: str) -> list[str]:
        """Checks every uploader shares; subclasses extend them."""
        return [] if file_name else ["empty file name"]


class ImageUploader(Uploader):
    def validate(self, file_name: str) -> list[str]:
        problems = super().validate(file_name)
        if not file_name.endswith((".png", ".jpg")):
            problems.append("not an image")
        return problems


print(f"{ImageUploader().validate('logo.png')=}")
print(f"{ImageUploader().validate('')=}")

ImageUploader().validate('logo.png')=[]
ImageUploader().validate('')=['empty file name', 'not an image']


**Abstract properties and class methods.** Stack `@abstractmethod` **under**
`@property` (or `@classmethod`), so `abstractmethod` is applied first. A subclass can
satisfy an abstract property with a real property or simply with a class attribute
of the same name.

In [118]:
class Backend(ABC):
    @property
    @abstractmethod
    def scheme(self) -> str:
        """URL scheme this backend handles, e.g. "s3"."""

    @classmethod
    @abstractmethod
    def from_url(cls, url: str) -> Backend:
        """Build a backend from a URL."""


class S3Backend(Backend):
    def __init__(self, bucket: str) -> None:
        self.bucket = bucket

    @property
    def scheme(self) -> str:
        return "s3"

    @classmethod
    def from_url(cls, url: str) -> S3Backend:
        return cls(url.removeprefix("s3://").split("/")[0])


class LocalBackend(Backend):
    scheme = "file"  # a plain class attribute also satisfies the abstract property

    @classmethod
    def from_url(cls, url: str) -> LocalBackend:
        return cls()


print(f"{S3Backend.from_url('s3://invoices/2026/11').bucket=}, {S3Backend('x').scheme=}")
print(f"{LocalBackend.from_url('file:///tmp').scheme=}, {sorted(Backend.__abstractmethods__)=}")

S3Backend.from_url('s3://invoices/2026/11').bucket='invoices', S3Backend('x').scheme='s3'
LocalBackend.from_url('file:///tmp').scheme='file', sorted(Backend.__abstractmethods__)=['from_url', 'scheme']


**The template-method pattern.** The base class owns the **algorithm** (a concrete
method that calls steps in a fixed order); subclasses fill in the **steps**. Here
`OrderImporter.run` always parses, cleans, validates and collects errors; the CSV
and JSON importers only say how to parse and what makes a record valid. `clean` is
an optional **hook** with a default.

In [119]:
class OrderImporter(ABC):
    """Template method: run() fixes the steps, subclasses implement them."""

    def run(self, raw_text: str) -> tuple[list[dict[str, Any]], list[str]]:
        accepted: list[dict[str, Any]] = []
        errors: list[str] = []
        for record in self.parse(raw_text):
            record = self.clean(record)
            problem = self.validate(record)
            if problem is None:
                accepted.append(record)
            else:
                errors.append(problem)
        return accepted, errors

    @abstractmethod
    def parse(self, raw_text: str) -> list[dict[str, Any]]:
        """Turn raw text into records."""

    def clean(self, record: dict[str, Any]) -> dict[str, Any]:
        """Optional hook: normalize a record (default: unchanged)."""
        return record

    @abstractmethod
    def validate(self, record: dict[str, Any]) -> str | None:
        """Return an error message, or None when the record is valid."""

Two importers that only supply the steps:

In [120]:
class CsvOrderImporter(OrderImporter):
    def parse(self, raw_text: str) -> list[dict[str, Any]]:
        rows = [line.split(",") for line in raw_text.strip().splitlines()[1:]]
        return [{"order_id": order_id, "quantity": quantity} for order_id, quantity in rows]

    def clean(self, record: dict[str, Any]) -> dict[str, Any]:
        return {**record, "quantity": int(record["quantity"])}

    def validate(self, record: dict[str, Any]) -> str | None:
        if record["quantity"] > 0:
            return None
        return f"{record['order_id']}: quantity must be positive"


class JsonOrderImporter(OrderImporter):
    def parse(self, raw_text: str) -> list[dict[str, Any]]:
        return json.loads(raw_text)

    def validate(self, record: dict[str, Any]) -> str | None:
        return None if "order_id" in record else "record without order_id"


print(CsvOrderImporter().run("order_id,quantity\nORD-1,2\nORD-2,0"))
print(JsonOrderImporter().run('[{"order_id": "ORD-3", "quantity": 1}, {"quantity": 4}]'))

([{'order_id': 'ORD-1', 'quantity': 2}], ['ORD-2: quantity must be positive'])
([{'order_id': 'ORD-3', 'quantity': 1}], ['record without order_id'])


### 2.3.2 `collections.abc`

The `collections.abc` module defines ABCs for Python's container protocols. Each
one has a few **abstract methods** you write, and **mixin methods** it implements for
you on top of them. Inheriting also makes `isinstance(obj, Mapping)` (and so on)
true, so code that accepts "any mapping" accepts your class.

| ABC | You write | You get for free |
| --- | --- | --- |
| `Iterable` | `__iter__` | (recognized structurally, see `__subclasshook__` below) |
| `Iterator` | `__next__` | `__iter__` (returns `self`) |
| `Sequence` | `__getitem__`, `__len__` | `__contains__`, `__iter__`, `__reversed__`, `index`, `count` |
| `Mapping` | `__getitem__`, `__iter__`, `__len__` | `__contains__`, `keys`, `items`, `values`, `get`, `__eq__`, `__ne__` |
| `MutableMapping` | the `Mapping` three + `__setitem__`, `__delitem__` | the `Mapping` mixins + `pop`, `popitem`, `clear`, `update`, `setdefault` |
| `Set` | `__contains__`, `__iter__`, `__len__` | `<=`, `<`, `==`, `!=`, `>`, `>=`, `&`, `|`, `-`, `^`, `isdisjoint` |

> **Note:** `Set` here is `collections.abc.Set`, the abstract read-only set, not the
> built-in `set` type.

#### `Iterable`

Any class with `__iter__` counts as `Iterable`, without inheriting or registering.
A class that only has the old `__getitem__` protocol can still be iterated by
`iter()`, but `isinstance(…, Iterable)` says `False`. To ask "can I iterate this?",
call `iter()` and catch `TypeError` (EAFP, 2.3.4).

In [121]:
class Playlist:
    """Iterable by defining __iter__; no base class."""

    def __init__(self, *songs: str) -> None:
        self._songs = list(songs)

    def __iter__(self) -> Iterator[str]:
        return iter(self._songs)


class LegacyPlaylist:
    """Only the old __getitem__ protocol."""

    def __getitem__(self, index: int) -> str:
        return ["Raga Yaman", "Raga Bhairav"][index]


print(f"{isinstance(Playlist('Raga Yaman'), Iterable)=}")
print(f"{list(LegacyPlaylist())=}, {isinstance(LegacyPlaylist(), Iterable)=}")
print(raised_message(lambda: iter(42)))

isinstance(Playlist('Raga Yaman'), Iterable)=True
list(LegacyPlaylist())=['Raga Yaman', 'Raga Bhairav'], isinstance(LegacyPlaylist(), Iterable)=False
TypeError: 'int' object is not iterable


#### `Iterator`

Inherit from `Iterator` and write only `__next__`; the ABC supplies the
`__iter__` that returns `self` (1.13).

In [122]:
class TicketDispenser(Iterator[int]):
    """Hands out ticket numbers first..last."""

    def __init__(self, first: int, last: int) -> None:
        self._next_number = first
        self._last = last

    def __next__(self) -> int:
        if self._next_number > self._last:
            raise StopIteration
        number = self._next_number
        self._next_number += 1
        return number


dispenser = TicketDispenser(101, 104)
print(f"{iter(dispenser) is dispenser=}, {'__iter__' in vars(TicketDispenser)=}")
print(f"{list(dispenser)=}, {list(dispenser)=}  (exhausted)")

iter(dispenser) is dispenser=True, '__iter__' in vars(TicketDispenser)=False
list(dispenser)=[101, 102, 103, 104], list(dispenser)=[]  (exhausted)


#### `Sequence`

`PageRange` describes the page boundaries for paginating `total_items` results.
It computes each page on demand instead of storing a list, writes only `__len__`
and `__getitem__` (with negative indexes and slices), and gets `in`, iteration,
`reversed`, `index` and `count` from `Sequence`.

In [123]:
class PageRange(Sequence[tuple[int, int]]):
    """(start, stop) item offsets for each page of total_items results."""

    def __init__(self, total_items: int, page_size: int) -> None:
        self.total_items = total_items
        self.page_size = page_size

    def __len__(self) -> int:
        return -(-self.total_items // self.page_size)  # ceiling division

    def __getitem__(self, index: int | slice) -> Any:
        if isinstance(index, slice):
            return [self[position] for position in range(*index.indices(len(self)))]
        if index < 0:
            index += len(self)
        if not 0 <= index < len(self):
            raise IndexError("page out of range")
        start = index * self.page_size
        return start, min(start + self.page_size, self.total_items)

Every line below except the first two uses a method `PageRange` never wrote.

In [124]:
pages = PageRange(total_items=45, page_size=10)
print(f"{len(pages)=}, {pages[-1]=}, {pages[1:3]=}")
print(f"{list(pages)=}")
print(f"{(40, 45) in pages=}, {pages.index((20, 30))=}, {pages.count((0, 10))=}")
print(f"{list(reversed(pages))[:2]=}")
print(f"{isinstance(pages, Sequence)=}")

len(pages)=5, pages[-1]=(40, 45), pages[1:3]=[(10, 20), (20, 30)]
list(pages)=[(0, 10), (10, 20), (20, 30), (30, 40), (40, 45)]
(40, 45) in pages=True, pages.index((20, 30))=2, pages.count((0, 10))=1
list(reversed(pages))[:2]=[(40, 45), (30, 40)]
isinstance(pages, Sequence)=True


#### `Mapping`

`EnvironmentConfig` exposes the environment variables that share a prefix, with the
prefix removed and the keys lower-cased. Three methods are enough for `get`, `in`,
`keys`, `items`, `values`, `==` and `dict(...)` to work. The real `os.environ` is
itself a `MutableMapping`; the demo uses a fake environment so the output is the
same on every machine.

In [125]:
class EnvironmentConfig(Mapping[str, str]):
    """Settings from prefixed environment variables, e.g. APP_PORT -> "port"."""

    def __init__(self, environ: Mapping[str, str], prefix: str) -> None:
        self._values = {
            name.removeprefix(prefix).lower(): value
            for name, value in environ.items()
            if name.startswith(prefix)
        }

    def __getitem__(self, key: str) -> str:
        return self._values[key]

    def __iter__(self) -> Iterator[str]:
        return iter(self._values)

    def __len__(self) -> int:
        return len(self._values)


fake_environ = {"APP_DEBUG": "1", "APP_PORT": "8000", "HOME": "/home/asha"}
app_config = EnvironmentConfig(fake_environ, prefix="APP_")
print(f"{dict(app_config)=}, {app_config.get('port')=}, {app_config.get('host', 'localhost')=}")
print(f"{'debug' in app_config=}, {app_config == {'debug': '1', 'port': '8000'}=}")
print(f"{isinstance(os.environ, MutableMapping)=}")

dict(app_config)={'debug': '1', 'port': '8000'}, app_config.get('port')='8000', app_config.get('host', 'localhost')='localhost'
'debug' in app_config=True, app_config == {'debug': '1', 'port': '8000'}=True
isinstance(os.environ, MutableMapping)=True


#### `MutableMapping`

HTTP header names are case-insensitive. `CaseInsensitiveDict` stores each entry
under the lower-cased key but remembers the original spelling. Five methods give it
`update`, `setdefault`, `pop`, `popitem` and `clear` too; the constructor itself
uses the free `update`.

In [126]:
class CaseInsensitiveDict(MutableMapping[str, str]):
    """A str -> str mapping whose keys ignore case (like HTTP headers)."""

    def __init__(self, initial: Mapping[str, str] | None = None) -> None:
        self._store: dict[str, tuple[str, str]] = {}
        if initial:
            self.update(initial)  # update() is a free mixin method

    def __setitem__(self, key: str, value: str) -> None:
        self._store[key.lower()] = (key, value)

    def __getitem__(self, key: str) -> str:
        return self._store[key.lower()][1]

    def __delitem__(self, key: str) -> None:
        del self._store[key.lower()]

    def __iter__(self) -> Iterator[str]:
        return (original_key for original_key, _ in self._store.values())

    def __len__(self) -> int:
        return len(self._store)


headers = CaseInsensitiveDict({"Content-Type": "application/json"})
headers.setdefault("X-Request-ID", "req-42")
print(f"{headers['CONTENT-TYPE']=}, {'x-request-id' in headers=}, {list(headers)=}")
print(f"{headers.pop('content-type')=}, {dict(headers)=}")

headers['CONTENT-TYPE']='application/json', 'x-request-id' in headers=True, list(headers)=['Content-Type', 'X-Request-ID']
headers.pop('content-type')='application/json', dict(headers)={'X-Request-ID': 'req-42'}


#### `Set`

A read-only set of permissions. Three methods give it every comparison and set
operator. Operators build their result with the class method `_from_iterable`,
which by default calls `cls(iterable)`; that works here because the constructor
accepts an iterable (otherwise you would override `_from_iterable`).

In [127]:
class PermissionSet(Set[str]):
    """An immutable set of permission names."""

    def __init__(self, permissions: Iterable[str] = ()) -> None:
        self._permissions = frozenset(permissions)

    def __contains__(self, permission: object) -> bool:
        return permission in self._permissions

    def __iter__(self) -> Iterator[str]:
        return iter(sorted(self._permissions))

    def __len__(self) -> int:
        return len(self._permissions)


editor = PermissionSet({"read", "write"})
admin = PermissionSet({"read", "write", "delete"})
print(f"{editor <= admin=}, {editor < admin=}, {editor == PermissionSet(['write', 'read'])=}")
print(f"{list(admin - editor)=}, {list(editor | {'comment'})=}, {type(admin & editor).__name__=}")
print(f"{editor.isdisjoint({'delete'})=}")

editor <= admin=True, editor < admin=True, editor == PermissionSet(['write', 'read'])=True
list(admin - editor)=['delete'], list(editor | {'comment'})=['comment', 'read', 'write'], type(admin & editor).__name__='PermissionSet'
editor.isdisjoint({'delete'})=True


#### Mixin methods for free, counted

`free_methods` lists what each ABC implements on top of its abstract methods, by
walking the ABC's MRO (an `inspect`-style look at the class dictionaries).

In [128]:
def free_methods(abc_class: type) -> list[str]:
    """Callable members an ABC defines on top of its abstract methods."""
    found: set[str] = set()
    for klass in abc_class.__mro__[:-1]:  # skip object
        for name, value in vars(klass).items():
            is_public_or_dunder = not name.startswith("_") or name.endswith("__")
            if (
                callable(value)
                and is_public_or_dunder
                and name not in abc_class.__abstractmethods__
            ):
                found.add(name)
    return sorted(found - {"__subclasshook__", "__class_getitem__", "__init_subclass__"})


for abc_class in [Iterator, Sequence, Mapping, MutableMapping, Set]:
    print(f"{abc_class.__name__:<15} write {sorted(abc_class.__abstractmethods__)}")
    print(f"{'':<15} free  {free_methods(abc_class)}")

Iterator        write ['__next__']
                free  ['__iter__']
Sequence        write ['__getitem__', '__len__']
                free  ['__contains__', '__iter__', '__reversed__', 'count', 'index']
Mapping         write ['__getitem__', '__iter__', '__len__']
                free  ['__contains__', '__eq__', 'get', 'items', 'keys', 'values']
MutableMapping  write ['__delitem__', '__getitem__', '__iter__', '__len__', '__setitem__']
                free  ['__contains__', '__eq__', 'clear', 'get', 'items', 'keys', 'pop', 'popitem', 'setdefault', 'update', 'values']
Set             write ['__contains__', '__iter__', '__len__']
                free  ['__and__', '__eq__', '__ge__', '__gt__', '__le__', '__lt__', '__or__', '__rand__', '__ror__', '__rsub__', '__rxor__', '__sub__', '__xor__', 'isdisjoint']


#### `register`: virtual subclasses

`SomeABC.register(cls)` declares that `cls` implements the ABC **without
inheriting**. `isinstance` and `issubclass` then return `True`, but the class gets
**no mixin methods** and the ABC is **not** in its MRO. Python also does **no
checking**: registering is a promise you make. The built-ins are wired up this way:
`list` is a registered `MutableSequence`.

In [129]:
class LegacyRecordSet:
    """An old class we cannot change: it has __len__ and __getitem__."""

    def __init__(self, records: list[str]) -> None:
        self._records = records

    def __len__(self) -> int:
        return len(self._records)

    def __getitem__(self, index: int) -> str:
        return self._records[index]


Sequence.register(LegacyRecordSet)
legacy_records = LegacyRecordSet(["r1", "r2"])
print(f"{isinstance(legacy_records, Sequence)=}, {Sequence in LegacyRecordSet.__mro__=}")
print(f"{hasattr(legacy_records, 'index')=}  (no mixin methods for virtual subclasses)")
print(f"{issubclass(list, MutableSequence)=}, {MutableSequence in list.__mro__=}")

isinstance(legacy_records, Sequence)=True, Sequence in LegacyRecordSet.__mro__=False
hasattr(legacy_records, 'index')=False  (no mixin methods for virtual subclasses)
issubclass(list, MutableSequence)=True, MutableSequence in list.__mro__=False


#### `__subclasshook__`: structural checks

An ABC can decide membership by **inspecting the candidate class**. `Iterable`,
`Sized`, `Hashable` and friends do exactly this: their `__subclasshook__` looks for
the method in the class's MRO. Returning `NotImplemented` means "no opinion", so
normal inheritance and registration still apply. A list is not `Hashable`, because
`list` sets `__hash__ = None`, and the hook treats a `None` method as missing.

In [130]:
print(inspect.getsource(Sized.__subclasshook__))
print(f"{isinstance(PageRange(10, 5), Sized)=}, {isinstance([], Hashable)=}")

    @classmethod
    def __subclasshook__(cls, C):
        if cls is Sized:
            return _check_methods(C, "__len__")
        return NotImplemented

isinstance(PageRange(10, 5), Sized)=True, isinstance([], Hashable)=False


Your own ABC can do the same. `SupportsToJson` accepts any class that defines a
callable `to_json`, anywhere in its MRO, without inheritance or registration
(`JsonSerializableMixin` from 2.2.4 would qualify).

In [131]:
class SupportsToJson(ABC):  # noqa: B024 (membership comes from __subclasshook__)
    """Any class with a callable to_json() is a virtual subclass."""

    @classmethod
    def __subclasshook__(cls, candidate: type) -> Any:
        if cls is SupportsToJson:
            for klass in candidate.__mro__:
                if "to_json" in vars(klass):
                    return callable(vars(klass)["to_json"])
        return NotImplemented


class InvoiceRecord:
    def to_json(self) -> str:
        return '{"invoice": "INV-9"}'


class PlainRecord:
    pass


print(f"{isinstance(InvoiceRecord(), SupportsToJson)=}")
print(f"{isinstance(PlainRecord(), SupportsToJson)=}, {issubclass(dict, SupportsToJson)=}")

isinstance(InvoiceRecord(), SupportsToJson)=True
isinstance(PlainRecord(), SupportsToJson)=False, issubclass(dict, SupportsToJson)=False


### 2.3.3 Protocols (PEP 544)

An ABC is **nominal**: a class belongs because it inherits (or is registered).
A **Protocol** is **structural**, exactly like a TypeScript interface: any class
with matching methods satisfies it, with no inheritance and no registration.

```typescript
interface MessageChannel {
  send(recipient: string, message: string): boolean;
}
function broadcast(channel: MessageChannel, recipients: string[], message: string) { … }
broadcast({ send: (recipient, message) => true }, ["asha"], "hi");  // any matching shape
```

The Python version: `MessageChannel` describes the shape, and two unrelated classes
match it.

In [132]:
class MessageChannel(Protocol):
    """Anything that can send a message and report success."""

    def send(self, recipient: str, message: str) -> bool: ...


def broadcast(channel: MessageChannel, recipients: list[str], message: str) -> int:
    """Send to every recipient; return how many succeeded."""
    return sum(channel.send(recipient, message) for recipient in recipients)

Two channels that never mention `MessageChannel`. They match because they have a
`send(recipient, message) -> bool` method, and `broadcast` accepts both.

In [133]:
class ConsoleChannel:
    """No base class, no registration: it just has send()."""

    def __init__(self) -> None:
        self.sent: list[str] = []

    def send(self, recipient: str, message: str) -> bool:
        self.sent.append(f"{recipient}: {message}")
        return True


class PushChannel:
    """Only reaches users with a registered device."""

    def __init__(self, device_owners: set[str]) -> None:
        self.device_owners = device_owners

    def send(self, recipient: str, message: str) -> bool:
        return recipient in self.device_owners


console = ConsoleChannel()
print(f"{broadcast(console, ['asha', 'ravi'], 'Sale starts now')=}, {console.sent=}")
print(f"{broadcast(PushChannel({'ravi'}), ['asha', 'ravi'], 'Sale starts now')=}")

broadcast(console, ['asha', 'ravi'], 'Sale starts now')=2, console.sent=['asha: Sale starts now', 'ravi: Sale starts now']
broadcast(PushChannel({'ravi'}), ['asha', 'ravi'], 'Sale starts now')=1


**At run time a Protocol does nothing.** Annotations are not enforced: passing a
string fails only when `send` is called. And a plain Protocol cannot be used with
`isinstance` at all; that needs `@runtime_checkable` (below).

The checking happens in a **type checker** (mypy or pyright), before the code runs.
mypy would report the bad call as:

```text
error: Argument 1 to "broadcast" has incompatible type "str"; expected "MessageChannel"  [arg-type]
```

In [134]:
print(raised_message(lambda: broadcast("not a channel", ["asha"], "hi")))  # type: ignore[arg-type]
print(raised_message(lambda: isinstance(console, MessageChannel)))

AttributeError: 'str' object has no attribute 'send'
TypeError: Instance and class checks can only be used with @runtime_checkable protocols


The type checkers are Phase 4 tools and are not installed yet, so this cell only
reports whether one is available.

In [135]:
for checker_module, checker_command in [("mypy", "mypy"), ("pyright", "pyright")]:
    installed = importlib.util.find_spec(checker_module) or shutil.which(checker_command)
    status = "available" if installed else "not installed yet (Phase 4): skipping"
    print(f"{checker_module}: {status}")

mypy: not installed yet (Phase 4): skipping
pyright: not installed yet (Phase 4): skipping


**Inspecting a Protocol.** `typing.get_protocol_members` and `typing.is_protocol`
(*New in 3.13*) list the members a class must have. You *may* inherit from a
Protocol explicitly, like TypeScript's `implements`; the class then also inherits
any default method bodies. A Protocol itself cannot be instantiated.

In [136]:
class LoggedChannel(MessageChannel):
    """Explicitly declares that it implements MessageChannel."""

    def send(self, recipient: str, message: str) -> bool:
        return bool(recipient)


if sys.version_info >= (3, 13):
    print(f"{typing.get_protocol_members(MessageChannel)=}")
    print(f"{typing.is_protocol(MessageChannel)=}, {typing.is_protocol(LoggedChannel)=}")
else:
    print("get_protocol_members / is_protocol need Python 3.13+")
print(f"{LoggedChannel().send('asha', 'hi')=}")
print(raised_message(MessageChannel))

typing.get_protocol_members(MessageChannel)=frozenset({'send'})
typing.is_protocol(MessageChannel)=True, typing.is_protocol(LoggedChannel)=False
LoggedChannel().send('asha', 'hi')=True
TypeError: Protocols cannot be instantiated


Protocols can describe **attributes** as well as methods, and they combine with
generics. `cheapest` accepts any items that support `<`, so it works for `Decimal`,
`str` and anything else comparable, while a type checker still rejects items that
cannot be compared. (PEP 695 syntax `[T: Bound]`, 3.12+.)

In [137]:
class SupportsLessThan(Protocol):
    def __lt__(self, other: Any, /) -> bool: ...


class Named(Protocol):
    name: str  # an attribute member


def cheapest[T: SupportsLessThan](items: Iterable[T]) -> T:
    """The smallest item; T can be any comparable type."""
    return min(items)


def greeting(thing: Named) -> str:
    return f"Hello, {thing.name}"


class Shop:
    def __init__(self, name: str) -> None:
        self.name = name


print(f"{cheapest([Decimal('49.50'), Decimal('12.00'), Decimal('120')])=}")
print(f"{cheapest(['tea', 'coffee', 'soap'])=}")
print(greeting(Shop("Kirana Corner")))

cheapest([Decimal('49.50'), Decimal('12.00'), Decimal('120')])=Decimal('12.00')
cheapest(['tea', 'coffee', 'soap'])='coffee'
Hello, Kirana Corner


#### `@runtime_checkable` and its limits

Decorating a Protocol with `@runtime_checkable` makes `isinstance` work. It checks
only that every member **name exists** on the object:

In [138]:
@runtime_checkable
class Flushable(Protocol):
    def flush(self) -> None: ...


print(f"{isinstance(io.StringIO(), Flushable)=}, {isinstance(sys.stdout, Flushable)=}")
print(f"{isinstance(42, Flushable)=}, {issubclass(io.BytesIO, Flushable)=}")

isinstance(io.StringIO(), Flushable)=True, isinstance(sys.stdout, Flushable)=True
isinstance(42, Flushable)=False, issubclass(io.BytesIO, Flushable)=True


**Limit 1: names, not signatures.** The check ignores parameters and return types,
and does not even require the member to be callable. Only a member explicitly set
to `None` counts as missing.

In [139]:
class WrongSignature:
    def flush(self, force: bool, level: int) -> str:
        return "nothing like the protocol"


class NotEvenAMethod:
    flush = "yes"


class ExplicitlyNone:
    flush = None


for candidate in [WrongSignature, NotEvenAMethod, ExplicitlyNone]:
    print(f"{candidate.__name__:<15} isinstance -> {isinstance(candidate(), Flushable)}")

WrongSignature  isinstance -> True
NotEvenAMethod  isinstance -> True
ExplicitlyNone  isinstance -> False


**Limit 2: no `issubclass` with attribute members.** Without an instance Python
cannot know whether an attribute will exist, so Protocols with data members only
support `isinstance`.

**Limit 3: properties are not run.** Since 3.12 the check uses
`inspect.getattr_static`, so it never executes a property or `__getattr__`; a
property that would raise still counts as present.

In [140]:
@runtime_checkable
class HasName(Protocol):
    name: str


class LazyRecord:
    @property
    def name(self) -> str:
        raise RuntimeError("database unavailable")


print(f"{isinstance(LazyRecord(), HasName)=}  (the property was never called)")
print(raised_message(lambda: issubclass(LazyRecord, HasName)))

isinstance(LazyRecord(), HasName)=True  (the property was never called)
TypeError: Protocols with non-method members don't support issubclass(). Non-method members: 'name'.


**Limit 4: speed.** A runtime Protocol check looks up every member on every call. It
is slower than an ABC check (whose result is cached per class) and several times
slower than a plain `hasattr`. Avoid it in hot loops.

> **Note:** timings vary by machine and Python version; compare the ratios.

In [141]:
buffer = io.StringIO()
for label, check in [
    ("isinstance(Protocol)", lambda: isinstance(buffer, Flushable)),
    ("isinstance(ABC)", lambda: isinstance(buffer, Iterable)),
    ("hasattr", lambda: hasattr(buffer, "flush")),
]:
    seconds = min(timeit.repeat(check, number=20_000, repeat=3))
    print(f"{label:<22} {seconds / 20_000 * 1e9:8.0f} ns per check")

isinstance(Protocol)        152 ns per check
isinstance(ABC)              74 ns per check
hasattr                      31 ns per check


**Protocol or ABC?**

| Question | Protocol | ABC |
| --- | --- | --- |
| How does a class qualify? | By having the right members (structural) | By inheriting or `register` (nominal) |
| Can third-party classes qualify without changes? | Yes | Only via `register` |
| Checked when? | By the type checker; `isinstance` only with `@runtime_checkable` (names only) | At instantiation (missing abstract methods) and by `isinstance` |
| Provides shared code? | Only to classes that inherit explicitly | Yes: concrete and mixin methods |
| Typical use | Describing what a function **needs** from its arguments | A base class **family** with shared behaviour (template method) |
| TypeScript analogue | `interface` | `abstract class` |

Rule of thumb: describe **parameters** with Protocols, build **class hierarchies**
with ABCs. The standard library has both: `typing.SupportsIndex` and friends are
runtime-checkable Protocols, while `collections.abc.Sized` is an ABC with a
structural `__subclasshook__`.

In [142]:
print(f"{isinstance(3, typing.SupportsIndex)=}, {isinstance(3.5, typing.SupportsIndex)=}")
print(f"{typing.is_protocol(typing.SupportsIndex)=}, {typing.is_protocol(Sized)=}")
print(f"{isinstance([], Sized)=}, {type(Sized).__name__=}")

isinstance(3, typing.SupportsIndex)=True, isinstance(3.5, typing.SupportsIndex)=False
typing.is_protocol(typing.SupportsIndex)=True, typing.is_protocol(Sized)=False
isinstance([], Sized)=True, type(Sized).__name__='ABCMeta'


### 2.3.4 Duck typing and polymorphism

"If it walks like a duck and quacks like a duck, it is a duck." Python code usually
does not ask what **type** an object is, only whether it **can** do what is needed.
That gives polymorphism without any shared base class: `len()` works on anything
with `__len__`, including `PageRange` from 2.3.2.

In [143]:
def total_length(collections_of_things: Iterable[Sized]) -> int:
    """Works with any objects that have __len__."""
    return sum(len(things) for things in collections_of_things)


mixed_collections: list[Sized] = ["chai", [1, 2, 3], {"a": 1}, PageRange(45, 10), range(5)]
print(f"{[len(things) for things in mixed_collections]=}")
print(f"{total_length(mixed_collections)=}")

[len(things) for things in mixed_collections]=[4, 3, 1, 5, 5]
total_length(mixed_collections)=18


Operators are polymorphic in the same way: `+` calls `__add__` on whatever the
left operand is (Unit 2.4 shows how to write these methods).

In [144]:
for left, right in [(2, 3), ("chai", " latte"), ([1], [2]), (Decimal("1.10"), Decimal("2.20"))]:
    print(f"{type(left).__name__:<8} {left!r} + {right!r} = {left + right!r}")

int      2 + 3 = 5
str      'chai' + ' latte' = 'chai latte'
list     [1] + [2] = [1, 2]
Decimal  Decimal('1.10') + Decimal('2.20') = Decimal('3.30')


**EAFP over type checks.** "Easier to ask forgiveness than permission": try the
operation and handle the failure, instead of checking types first (LBYL, 1.9.6 and
1.14.7). The type-checking version below rejects `FakeSocket` even though it would
work perfectly; the EAFP version accepts anything with a `read()` method.

In [145]:
class FakeSocket:
    """Not a file at all, but it has read()."""

    def read(self) -> str:
        return "data from the network"


def read_all_lbyl(source: object) -> str:
    """Type checks: only the types the author thought of."""
    if isinstance(source, io.StringIO):
        return source.read()
    if isinstance(source, str):
        return source
    raise TypeError(f"unsupported source: {type(source).__name__}")


def read_all(source: Any) -> str:
    """EAFP: anything with .read() works; otherwise treat it as text."""
    try:
        reader = source.read
    except AttributeError:
        return str(source)
    return reader()

Each source goes through both versions. `result_or_error` shows either the result or
the `TypeError` the type-checking version raises.

In [146]:
def result_or_error(thunk: Callable[[], str]) -> str:
    """The call's result, or "TypeError: message" if it raised TypeError."""
    try:
        return thunk()
    except TypeError as error:
        return f"{type(error).__name__}: {error}"


source_makers = [("StringIO", lambda: io.StringIO("from a buffer")), ("FakeSocket", FakeSocket)]
source_makers.append(("str", lambda: "plain text"))
for label, make_source in source_makers:
    lbyl_result = result_or_error(functools.partial(read_all_lbyl, make_source()))
    print(f"{label:<10} LBYL: {lbyl_result:<43} EAFP: {read_all(make_source())}")

StringIO   LBYL: from a buffer                               EAFP: from a buffer


FakeSocket LBYL: TypeError: unsupported source: FakeSocket   EAFP: data from the network
str        LBYL: plain text                                  EAFP: plain text


> **Warning:** keep the `try` block **narrow**. If the `try` also covers the call
> to `read()`, an `AttributeError` raised by a bug *inside* `read()` is swallowed and
> silently treated as "not readable". The version above only looks up the attribute
> inside the `try`, so the bug surfaces.

In [147]:
class BrokenReader:
    """read() has a bug that raises AttributeError internally."""

    def read(self) -> str:
        return self.buffer.decode()  # bug: there is no self.buffer

    def __str__(self) -> str:
        return "BrokenReader (fallback used: the bug is hidden)"


def read_all_too_wide(source: Any) -> str:
    """BUG: the try block also covers the call."""
    try:
        return source.read()
    except AttributeError:
        return str(source)


print(read_all_too_wide(BrokenReader()))
print(raised_message(lambda: read_all(BrokenReader())))

BrokenReader (fallback used: the bug is hidden)
AttributeError: 'BrokenReader' object has no attribute 'buffer'


**When type checks are right.** Duck typing is the default, but `isinstance` has
its place:

| Situation | Tool |
| --- | --- |
| Validating untrusted input at a boundary (JSON, CLI args) | `isinstance` against built-in types |
| "Is this any kind of mapping / sequence?" | `isinstance(x, Mapping)` with `collections.abc`, never `type(x) is dict` |
| Different behaviour per type, kept open for extension | `functools.singledispatch` (1.12.7) |
| Describing what a function needs, for the type checker | a Protocol (2.3.3) |

`singledispatch` dispatches on the type of the first argument and accepts ABCs, so
any `Mapping` (including `EnvironmentConfig`) takes the mapping branch.

In [148]:
@functools.singledispatch
def to_display(value: object) -> str:
    """Default: str()."""
    return str(value)


@to_display.register
def _(value: Decimal) -> str:
    return f"INR {value:,.2f}"


@to_display.register
def _(value: Mapping) -> str:
    return ", ".join(f"{key}={item}" for key, item in value.items())


for value in [42, Decimal("1234.5"), {"port": 8000}, app_config]:
    print(f"{type(value).__name__:<18} -> {to_display(value)}")

int                -> 42
Decimal            -> INR 1,234.50
dict               -> port=8000
EnvironmentConfig  -> debug=1, port=8000


**Predict the output.**

```python
from collections.abc import Iterable

class Countdown:
    def __getitem__(self, index):
        if index >= 3:
            raise IndexError
        return 3 - index

print(list(Countdown()))
print(isinstance(Countdown(), Iterable))
```

<details><summary>Answer</summary>

```text
[3, 2, 1]
False
```

`iter()` falls back to the old sequence protocol: it calls `__getitem__(0)`,
`__getitem__(1)`, … until `IndexError`. But `Iterable.__subclasshook__` only looks
for `__iter__`, so the ABC check says `False`. Asking "can I iterate it?" with
`isinstance(x, Iterable)` is therefore not fully reliable; trying `iter(x)` is.

</details>

Run the next cell to check your prediction.

In [149]:
class Countdown:
    """Iterable only through the old __getitem__ protocol."""

    def __getitem__(self, index: int) -> int:
        if index >= 3:
            raise IndexError
        return 3 - index


print(list(Countdown()))
print(isinstance(Countdown(), Iterable))

[3, 2, 1]
False


### Lab 2.3

Four exercises. For each, complete the stub, then run the check cell. Until then
the checks print FAIL; that is expected. Some structural checks already pass on the
stub because the class shape is given: "Square is not a Rectangle" (exercise 1),
"no assignment", "is a Mapping", "not a MutableMapping" and "only three methods
written" (exercise 2), and "issubclass refused" (exercise 4). Keep them passing.

#### Exercise 1 — a `Shape` ABC with sortable subclasses

- `Shape(ABC)` declares an **abstract property** `area` and an **abstract method**
  `perimeter()`.
- `Shape.__lt__(other)` compares areas, so `sorted()` orders shapes from smallest to
  largest; return `NotImplemented` when `other` is not a `Shape` (Python then raises
  `TypeError`). `sorted` needs only `<`; Unit 2.4 covers comparison methods fully.
- `Shape.describe()` is a template method returning
  `f"{class name}: area={area:.2f}, perimeter={perimeter:.2f}"`.
- `Circle(radius)`, `Rectangle(width, height)` and `Square(side)` are concrete
  shapes. Every dimension must be greater than zero, otherwise
  `ValueError("dimensions must be positive")`. `Square` must **not** subclass
  `Rectangle` (2.2.5).

In [150]:
class Shape(ABC):  # noqa: B024 (add the abstract members)
    """Abstract shape: subclasses provide area and perimeter; shapes sort by area."""

    # Declare the abstract property `area` and the abstract method `perimeter()` here.

    def __lt__(self, other: object) -> bool:
        raise NotImplementedError("Lab 2.3: implement Shape.__lt__")

    def describe(self) -> str:
        raise NotImplementedError("Lab 2.3: implement Shape.describe")


class Circle(Shape):
    def __init__(self, radius: float) -> None:
        raise NotImplementedError("Lab 2.3: implement Circle")


class Rectangle(Shape):
    def __init__(self, width: float, height: float) -> None:
        raise NotImplementedError("Lab 2.3: implement Rectangle")


class Square(Shape):
    def __init__(self, side: float) -> None:
        raise NotImplementedError("Lab 2.3: implement Square")

Run the checks. `Triangle` deliberately forgets `perimeter()`, so creating one must
fail once `Shape` is abstract.

In [151]:
class Triangle(Shape):
    """Forgets perimeter()."""

    @property
    def area(self) -> float:
        return 1.0


run_checks(
    [
        ("Shape is abstract", lambda: raised_error_name(Shape), "TypeError"),
        ("incomplete subclass", lambda: raised_error_name(Triangle), "TypeError"),
        ("abstract members", lambda: sorted(Shape.__abstractmethods__), ["area", "perimeter"]),
        (
            "sorted by area",
            lambda: [
                type(shape).__name__
                for shape in sorted([Rectangle(3, 4), Circle(1), Square(3), Circle(2)])
            ],
            ["Circle", "Square", "Rectangle", "Circle"],
        ),
        ("less than", lambda: (Circle(1) < Square(2), Square(2) < Circle(1)), (True, False)),
        ("describe circle", lambda: Circle(1).describe(), "Circle: area=3.14, perimeter=6.28"),
        (
            "describe rectangle",
            lambda: Rectangle(3, 4).describe(),
            "Rectangle: area=12.00, perimeter=14.00",
        ),
        ("describe square", lambda: Square(2.5).describe(), "Square: area=6.25, perimeter=10.00"),
        ("area is a property", lambda: Square(3).area, 9),
        ("Square is not a Rectangle", lambda: issubclass(Square, Rectangle), False),
        (
            "positive dimensions",
            lambda: raised_message(lambda: Rectangle(0, 2)),
            "ValueError: dimensions must be positive",
        ),
        ("compare with non-shape", lambda: raised_error_name(lambda: Circle(1) < 5), "TypeError"),
    ]
)

FAIL  Shape is abstract: expected 'TypeError', got 'no error'
FAIL  incomplete subclass: expected 'TypeError', got 'no error'
FAIL  abstract members: expected ['area', 'perimeter'], got []
FAIL  sorted by area: raised NotImplementedError: Lab 2.3: implement Rectangle
FAIL  less than: raised NotImplementedError: Lab 2.3: implement Circle
FAIL  describe circle: raised NotImplementedError: Lab 2.3: implement Circle
FAIL  describe rectangle: raised NotImplementedError: Lab 2.3: implement Rectangle
FAIL  describe square: raised NotImplementedError: Lab 2.3: implement Square
FAIL  area is a property: raised NotImplementedError: Lab 2.3: implement Square
PASS  Square is not a Rectangle
FAIL  positive dimensions: expected 'ValueError: dimensions must be positive', got 'NotImplementedError: Lab 2.3: implement Rectangle'
FAIL  compare with non-shape: expected 'TypeError', got 'NotImplementedError'
1/12 checks passed


#### Exercise 2 — `ReadOnlyView(Mapping)` with three methods

`ReadOnlyView(data)` wraps another mapping **without copying it**: changes to the
wrapped dict show through the view, but the view itself cannot be changed (item
assignment raises `TypeError`, because `Mapping` has no `__setitem__`).

Implement only `__getitem__`, `__iter__` and `__len__`; `get`, `keys`, `items`,
`values`, `in` and `==` must come from `Mapping`.

In [152]:
class ReadOnlyView[K, V](Mapping[K, V]):
    """A live, read-only view of another mapping."""

    def __init__(self, data: Mapping[K, V]) -> None:
        self._data = data

    def __getitem__(self, key: K) -> V:
        raise NotImplementedError("Lab 2.3: implement ReadOnlyView.__getitem__")

    def __iter__(self) -> Iterator[K]:
        raise NotImplementedError("Lab 2.3: implement ReadOnlyView.__iter__")

    def __len__(self) -> int:
        raise NotImplementedError("Lab 2.3: implement ReadOnlyView.__len__")

Run the checks.

In [153]:
prices = {"soap": 49, "tea": 120}
MAPPING_METHOD_NAMES = {
    "__getitem__",
    "__iter__",
    "__len__",
    "get",
    "keys",
    "items",
    "values",
    "__contains__",
}


def live_update() -> int:
    """Change the wrapped dict and read through the view."""
    backing = {"soap": 49}
    view = ReadOnlyView(backing)
    backing["soap"] = 55
    return view["soap"]


def try_assign() -> None:
    """Item assignment on the view."""
    view = ReadOnlyView({"soap": 49})
    view["soap"] = 1  # type: ignore[index]


run_checks(
    [
        ("getitem", lambda: ReadOnlyView(prices)["tea"], 120),
        ("len", lambda: len(ReadOnlyView(prices)), 2),
        ("iteration order", lambda: list(ReadOnlyView(prices)), ["soap", "tea"]),
        (
            "get (free)",
            lambda: (ReadOnlyView(prices).get("rice"), ReadOnlyView(prices).get("rice", 0)),
            (None, 0),
        ),
        ("items (free)", lambda: list(ReadOnlyView(prices).items()), [("soap", 49), ("tea", 120)]),
        (
            "in (free)",
            lambda: ("soap" in ReadOnlyView(prices), "rice" in ReadOnlyView(prices)),
            (True, False),
        ),
        ("== dict (free)", lambda: ReadOnlyView(prices) == {"tea": 120, "soap": 49}, True),
        (
            "missing key",
            lambda: raised_error_name(lambda: ReadOnlyView(prices)["rice"]),
            "KeyError",
        ),
        ("live view", live_update, 55),
        ("no assignment", lambda: raised_error_name(try_assign), "TypeError"),
        ("is a Mapping", lambda: isinstance(ReadOnlyView(prices), Mapping), True),
        ("not a MutableMapping", lambda: isinstance(ReadOnlyView(prices), MutableMapping), False),
        (
            "only three methods written",
            lambda: sorted(name for name in vars(ReadOnlyView) if name in MAPPING_METHOD_NAMES),
            ["__getitem__", "__iter__", "__len__"],
        ),
    ]
)

FAIL  getitem: raised NotImplementedError: Lab 2.3: implement ReadOnlyView.__getitem__
FAIL  len: raised NotImplementedError: Lab 2.3: implement ReadOnlyView.__len__
FAIL  iteration order: raised NotImplementedError: Lab 2.3: implement ReadOnlyView.__iter__
FAIL  get (free): raised NotImplementedError: Lab 2.3: implement ReadOnlyView.__getitem__
FAIL  items (free): raised NotImplementedError: Lab 2.3: implement ReadOnlyView.__len__
FAIL  in (free): raised NotImplementedError: Lab 2.3: implement ReadOnlyView.__getitem__
FAIL  == dict (free): raised NotImplementedError: Lab 2.3: implement ReadOnlyView.__iter__
FAIL  missing key: expected 'KeyError', got 'NotImplementedError'
FAIL  live view: raised NotImplementedError: Lab 2.3: implement ReadOnlyView.__getitem__
PASS  no assignment
PASS  is a Mapping
PASS  not a MutableMapping
PASS  only three methods written
4/13 checks passed


#### Exercise 3 — a `SupportsClose` Protocol

- Make `SupportsClose` a **runtime-checkable** Protocol with one member,
  `close(self) -> None`.
- Write `close_all(resources)`: for every item that satisfies `SupportsClose` (in
  order), call `close()`; skip items that do not. Keep going when a `close()` raises
  an `Exception`. Return `(closed_count, failures)` where `closed_count` counts the
  successful closes and `failures` is a list of
  `f"{type(resource).__name__}: {error}"` strings.
- `resources` can be any iterable, including a generator.

In [154]:
class SupportsClose(Protocol):
    """Anything with a close() method. Add the member and make it runtime-checkable."""


def close_all(resources: Iterable[object]) -> tuple[int, list[str]]:
    """Close every resource that supports close(), even if some fail."""
    raise NotImplementedError("Lab 2.3: implement close_all")

Run the checks. `Recorder` and `FlakyResource` inherit from nothing.

In [155]:
class Recorder:
    """Closes cleanly and remembers it."""

    def __init__(self) -> None:
        self.closed = False

    def close(self) -> None:
        self.closed = True


class FlakyResource:
    """close() fails."""

    def close(self) -> None:
        raise OSError("disk unplugged")


def close_mixed() -> tuple[tuple[int, list[str]], bool, bool]:
    """close_all over a mix; also report whether the good resources really closed."""
    buffer = io.StringIO("data")
    recorder = Recorder()
    report = close_all([buffer, "not closable", FlakyResource(), 42, recorder])
    return report, buffer.closed, recorder.closed


run_checks(
    [
        ("StringIO", lambda: isinstance(io.StringIO(), SupportsClose), True),
        ("plain class, no inheritance", lambda: isinstance(Recorder(), SupportsClose), True),
        ("int", lambda: isinstance(42, SupportsClose), False),
        ("issubclass", lambda: issubclass(FlakyResource, SupportsClose), True),
        ("members", lambda: typing.get_protocol_members(SupportsClose), frozenset({"close"})),
        ("close_all", close_mixed, ((2, ["FlakyResource: disk unplugged"]), True, True)),
        ("generator input", lambda: close_all(Recorder() for _ in range(3)), (3, [])),
        ("empty", lambda: close_all([]), (0, [])),
    ]
)

FAIL  StringIO: raised TypeError: Instance and class checks can only be used with @runtime_checkable protocols
FAIL  plain class, no inheritance: raised TypeError: Instance and class checks can only be used with @runtime_checkable protocols
FAIL  int: raised TypeError: Instance and class checks can only be used with @runtime_checkable protocols
FAIL  issubclass: raised TypeError: Instance and class checks can only be used with @runtime_checkable protocols
FAIL  members: expected frozenset({'close'}), got frozenset()
FAIL  close_all: raised NotImplementedError: Lab 2.3: implement close_all
FAIL  generator input: raised NotImplementedError: Lab 2.3: implement close_all
FAIL  empty: raised NotImplementedError: Lab 2.3: implement close_all
0/8 checks passed


#### Exercise 4 — a payment gateway behind a `PaymentProvider` Protocol

Write a runtime-checkable Protocol `PaymentProvider` with three members: the
attribute `name: str`, `charge(amount_paise, token) -> str` (returns a transaction
id, or raises `PaymentDeclinedError`) and `refund(transaction_id) -> None`.

Two fake providers implement it **without inheriting from it**. Both keep
`charges: dict[str, int]` (transaction id → amount) and `refunded: list[str]`, number
their transactions `f"{name}_{n:04d}"` from 1, and raise
`ValueError("amount must be positive")` (a programming error, not a decline) when
`amount_paise <= 0`. `refund` moves a charge from `charges` to `refunded` and raises
`KeyError(transaction_id)` for an unknown id.

| Provider | `name` | Declines with `PaymentDeclinedError(…)` |
| --- | --- | --- |
| `CardGateway()` | `"card"` | `"invalid card token"` unless the token starts with `"tok_"` |
| `UpiGateway()` | `"upi"` | `"invalid UPI ID"` unless the token contains `"@"`; `"amount exceeds UPI limit"` above `UPI_LIMIT_PAISE` |

`CheckoutService(provider)` depends only on the Protocol:

- `pay(order_id, amount_paise, token)` returns a `PaymentResult`: status `"paid"`
  with the transaction id and message `""`, or `"declined"` with
  `transaction_id=None` and the decline message. Every result is appended to
  `self.history`. A `ValueError` from the provider is **not** caught.
- `refund(result)` refunds a paid result through the provider; anything else raises
  `ValueError("only paid orders can be refunded")`.

The given types:

In [156]:
class PaymentDeclinedError(Exception):
    """The provider refused the charge; the message says why."""


class PaymentResult(NamedTuple):
    """Outcome of CheckoutService.pay."""

    order_id: str
    provider: str
    status: str  # "paid" or "declined"
    transaction_id: str | None
    message: str


UPI_LIMIT_PAISE = 1_00_000_00  # ₹1,00,000

Your Protocol, providers and service:

In [157]:
class PaymentProvider(Protocol):
    """What CheckoutService needs from a payment provider. Add the three members."""


class CardGateway:
    def __init__(self) -> None:
        raise NotImplementedError("Lab 2.3: implement CardGateway")


class UpiGateway:
    def __init__(self) -> None:
        raise NotImplementedError("Lab 2.3: implement UpiGateway")


class CheckoutService:
    def __init__(self, provider: PaymentProvider) -> None:
        raise NotImplementedError("Lab 2.3: implement CheckoutService")

    def pay(self, order_id: str, amount_paise: int, token: str) -> PaymentResult:
        raise NotImplementedError("Lab 2.3: implement CheckoutService.pay")

    def refund(self, result: PaymentResult) -> None:
        raise NotImplementedError("Lab 2.3: implement CheckoutService.refund")

Run the checks. `WalletGateway` is a third provider written later, inheriting from
nothing; the service must accept it unchanged.

In [158]:
class WalletGateway:
    name = "wallet"

    def charge(self, amount_paise: int, token: str) -> str:
        return f"wallet-{amount_paise}"

    def refund(self, transaction_id: str) -> None:
        pass


def card_flow() -> list[PaymentResult]:
    """Two card payments: one paid, one declined."""
    checkout = CheckoutService(CardGateway())
    checkout.pay("ORD-1", 49_900, "tok_visa")
    checkout.pay("ORD-2", 10_000, "4111-1111")
    return checkout.history


def upi_flow() -> tuple[PaymentResult, PaymentResult, PaymentResult]:
    """UPI: paid, bad ID, over the limit."""
    checkout = CheckoutService(UpiGateway())
    return (
        checkout.pay("ORD-3", 25_000, "asha@okbank"),
        checkout.pay("ORD-4", 25_000, "asha"),
        checkout.pay("ORD-5", UPI_LIMIT_PAISE + 1, "asha@okbank"),
    )


def refund_flow() -> tuple[dict[str, int], list[str]]:
    """Pay twice, refund the first."""
    gateway = UpiGateway()
    checkout = CheckoutService(gateway)
    first = checkout.pay("ORD-6", 1_000, "ravi@okbank")
    checkout.pay("ORD-7", 2_000, "ravi@okbank")
    checkout.refund(first)
    return gateway.charges, gateway.refunded


def refund_declined() -> None:
    """Refunding a declined payment."""
    checkout = CheckoutService(CardGateway())
    checkout.refund(checkout.pay("ORD-8", 100, "bad-token"))

The checks themselves:

In [159]:
run_checks(
    [
        ("card satisfies", lambda: isinstance(CardGateway(), PaymentProvider), True),
        ("upi satisfies", lambda: isinstance(UpiGateway(), PaymentProvider), True),
        ("wallet satisfies", lambda: isinstance(WalletGateway(), PaymentProvider), True),
        ("str does not", lambda: isinstance("card", PaymentProvider), False),
        (
            "members",
            lambda: sorted(typing.get_protocol_members(PaymentProvider)),
            ["charge", "name", "refund"],
        ),
        (
            "issubclass refused (data member)",
            lambda: raised_error_name(lambda: issubclass(CardGateway, PaymentProvider)),
            "TypeError",
        ),
        (
            "card flow",
            card_flow,
            [
                PaymentResult("ORD-1", "card", "paid", "card_0001", ""),
                PaymentResult("ORD-2", "card", "declined", None, "invalid card token"),
            ],
        ),
        (
            "upi flow",
            upi_flow,
            (
                PaymentResult("ORD-3", "upi", "paid", "upi_0001", ""),
                PaymentResult("ORD-4", "upi", "declined", None, "invalid UPI ID"),
                PaymentResult("ORD-5", "upi", "declined", None, "amount exceeds UPI limit"),
            ),
        ),
        ("refund", refund_flow, ({"upi_0002": 2_000}, ["upi_0001"])),
        (
            "refund declined",
            lambda: raised_message(refund_declined),
            "ValueError: only paid orders can be refunded",
        ),
        (
            "unknown refund id",
            lambda: raised_error_name(lambda: CardGateway().refund("card_9999")),
            "KeyError",
        ),
        (
            "bad amount is a bug, not a decline",
            lambda: raised_error_name(
                lambda: CheckoutService(CardGateway()).pay("ORD-9", 0, "tok_x")
            ),
            "ValueError",
        ),
        (
            "third provider works",
            lambda: CheckoutService(WalletGateway()).pay("ORD-10", 500, "w"),
            PaymentResult("ORD-10", "wallet", "paid", "wallet-500", ""),
        ),
    ]
)

FAIL  card satisfies: raised NotImplementedError: Lab 2.3: implement CardGateway
FAIL  upi satisfies: raised NotImplementedError: Lab 2.3: implement UpiGateway
FAIL  wallet satisfies: raised TypeError: Instance and class checks can only be used with @runtime_checkable protocols
FAIL  str does not: raised TypeError: Instance and class checks can only be used with @runtime_checkable protocols
FAIL  members: expected ['charge', 'name', 'refund'], got []
PASS  issubclass refused (data member)
FAIL  card flow: raised NotImplementedError: Lab 2.3: implement CardGateway
FAIL  upi flow: raised NotImplementedError: Lab 2.3: implement UpiGateway
FAIL  refund: raised NotImplementedError: Lab 2.3: implement UpiGateway
FAIL  refund declined: expected 'ValueError: only paid orders can be refunded', got 'NotImplementedError: Lab 2.3: implement CardGateway'
FAIL  unknown refund id: expected 'KeyError', got 'NotImplementedError'
FAIL  bad amount is a bug, not a decline: expected 'ValueError', got 'NotI

### Recap — Unit 2.3

| Concept | One-line takeaway | JS / TS equivalent |
| --- | --- | --- |
| `ABC` + `@abstractmethod` | Instantiating a class with unimplemented abstract methods raises `TypeError` | TS `abstract class` |
| `ABCMeta` / `__abstractmethods__` | The metaclass records what is still abstract | — |
| Abstract property | `@property` over `@abstractmethod`; a class attribute can satisfy it | TS `abstract get x()` |
| Template method | The base owns the algorithm, subclasses fill in the steps (and optional hooks) | same pattern |
| `collections.abc` | Write the abstract methods, inherit the mixin methods | — |
| `register` | Virtual subclass: `isinstance` is true, no mixins, no checks | — |
| `__subclasshook__` | Membership decided by inspecting the class (how `Iterable` works) | `Symbol.hasInstance` |
| `Protocol` | Structural typing, checked by type checkers | TS `interface` |
| `@runtime_checkable` | `isinstance` checks member **names** only; no `issubclass` with attributes; slow | — (interfaces are erased) |
| Protocol vs ABC | Protocols for what a function needs; ABCs for families sharing code | `interface` vs `abstract class` |
| Duck typing | Ask "can it?", not "is it?"; polymorphism without a base class | same |
| EAFP | `try` the narrow operation, catch the specific error | `try/catch` |
| `singledispatch` | Type-based dispatch that understands ABCs | — |

**Further reading**

- [`abc` — Abstract base classes](https://docs.python.org/3/library/abc.html),
  [`collections.abc`](https://docs.python.org/3/library/collections.abc.html)
  (the full table of ABCs and mixin methods)
- [`typing.Protocol`](https://docs.python.org/3/library/typing.html#typing.Protocol),
  [`runtime_checkable`](https://docs.python.org/3/library/typing.html#typing.runtime_checkable),
  [`get_protocol_members`](https://docs.python.org/3/library/typing.html#typing.get_protocol_members),
  [typing spec: protocols](https://typing.python.org/en/latest/spec/protocol.html)
- [Glossary: duck typing](https://docs.python.org/3/glossary.html#term-duck-typing),
  [EAFP](https://docs.python.org/3/glossary.html#term-EAFP),
  [`functools.singledispatch`](https://docs.python.org/3/library/functools.html#functools.singledispatch)
- [PEP 3119 — Introducing abstract base classes](https://peps.python.org/pep-3119/),
  [PEP 544 — Protocols: structural subtyping](https://peps.python.org/pep-0544/),
  [PEP 695 — Type parameter syntax](https://peps.python.org/pep-0695/)